# Deney 01 — Video Analiz & Hot-Moment Tespiti

**Amaç:** Uzun bir kaynak videodan (a) transcript çıkarımı, (b) sahne tespiti, (c) ses enerjisi analizi ve (d) LLM tabanlı "vurucu an" (hot-moment) skorlaması için aday model/sağlayıcıları **kalite + maliyet + hız** üzerinden karşılaştırmak.

**Tek kaynak politikası:** Bu notebook algoritmaları kendi içinde tekrar tanımlamaz — transkripsiyon, sahne tespiti, ses tepe analizi, hot-moment skorlama, önizleme klibi üretimi ve `hot_moments.json` şeması hepsi `local_pipeline/video_analysis.py` içindeki **kanonik** fonksiyonlardan gelir. `00_youtube_automation_master.ipynb` ve `verify_local.py` de aynı fonksiyonları çağırır. Bu notebook'un kendine özgü katkısı **çok-aday karşılaştırma (benchmark) döngüsüdür**; master notebook ise tek varsayılan adayla uçtan uca çalışır.

**Nasıl çalışır:**
- Colab'da Runtime → Change runtime type → **GPU (T4)** seçip baştan sona çalıştırabilirsin.
- Bazı adaylar API anahtarı gerektirir (OpenAI / Anthropic / Google). Anahtarı olmayanlar otomatik atlanır, notebook hata vermez.
- API anahtarlarını Colab'da soldaki 🔑 **Secrets** panelinden ekle: `OPENAI_API_KEY`, `ANTHROPIC_API_KEY`, `GOOGLE_API_KEY`. Hiçbiri yoksa sadece ücretsiz/lokal adaylar (faster-whisper, PySceneDetect, librosa) çalışır.


## Google model router (bu notebook'taki tum Gemini cagrilari icin tek merkez)

Bu notebook artik `genai.Client(...)` kurmuyor ve sabit `model='gemini-...'` yazmiyor.
Her Google cagrisi `local_pipeline/google_ai.py` uzerinden gider:

- **Colab Secret adlari:** `GeminiKey` (eski ad `GOOGLE_API_KEY` hala calisir),
  `YoutubeKEY` (eski ad `YOUTUBE_DATA_API_KEY`). Deger hicbir zaman yazdirilmaz;
  yalnizca `configured (GeminiKey)` / `missing` durumu gosterilir.
- **Tek anahtar, cok model:** ayni `GeminiKey` projenin yetkili oldugu tum modellere erisir.
- **Kota PROJE bazlidir, anahtar bazli DEGILDIR** (ai.google.dev/gemini-api/docs/rate-limits);
  ikinci bir anahtar limit artirmaz. Hesap/proje dondurerek kota asmak yasaktir
  (`ALLOW_ACCOUNT_QUOTA_ROTATION = False`).
- **Rota:** ilk basarili cagri rotayi bitirir; hicbir zaman tum katalog denenmez.
- **Ucretsiz katman veri politikasi:** gonderilen icerik Google tarafindan urun
  gelistirmede kullanilabilir ve insan gozden gecirenler okuyabilir. Ham telifli
  video/ses varsayilan olarak YUKLENMEZ; yalnizca yerel olarak turetilmis
  transcript/skor ozetleri gonderilir.
- **Kota defteri** (`results/google_quota_usage.json`) YEREL TAHMINDIR; sunucunun
  yetkili kotasi degildir. Gunluk sayaclar Pasifik gece yarisinda sifirlanir.

Bu notebook'ta kullanilan gorev: **hot-moment incelemesi** -> Rota A (`gemini-3.6-flash` -> `gemini-3.5-flash` -> `gemini-3.5-flash-lite` -> `gemma-4-31b-it`).


## 0. Kurulum

In [ ]:
import os, subprocess, sys
if os.path.isdir('/content'):
    packages = ['yt-dlp==2026.7.4', 'faster-whisper==1.2.1', 'scenedetect-headless==0.7.1', 'librosa==0.11.0', 'openai==2.50.0', 'anthropic==0.120.2', 'google-genai==2.15.0', 'pandas==3.0.5', 'tqdm==4.70.0', 'moviepy==2.2.1', 'imageio-ffmpeg==0.6.0']
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *packages])
    print('Colab bagimliliklari kuruldu.')
else:
    print('Yerel calisma: proje .venv paketleri bir sonraki hucrede otomatik etkinlestirilecek.')

In [ ]:
import os, time, json, subprocess, shutil, sys, site
from pathlib import Path
if 'google.colab' not in sys.modules:
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        venv_site_packages = candidate / '.venv' / 'Lib' / 'site-packages'
        if venv_site_packages.is_dir():
            site.addsitedir(str(venv_site_packages))
            print('Yerel .venv paketleri etkinlestirildi:', venv_site_packages)
            break
import pandas as pd
import imageio_ffmpeg

IS_COLAB = 'google.colab' in sys.modules or os.path.isdir('/content')
PROJECT_ROOT = Path.cwd()
RUNTIME_DIR = Path('/content') if IS_COLAB else PROJECT_ROOT / '.local_runtime'
RESULTS_DIR = PROJECT_ROOT / 'results'
RUNTIME_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

FFMPEG_BIN = shutil.which('ffmpeg') or imageio_ffmpeg.get_ffmpeg_exe()
FFPROBE_BIN = shutil.which('ffprobe')
print('Calisma ortami:', 'Colab' if IS_COLAB else 'Yerel Jupyter')
print('FFmpeg:', FFMPEG_BIN)
print('FFprobe:', FFPROBE_BIN or 'bulunamadi; on kontrol FFmpeg donusumunde yapilacak')

def get_secret(name):
    try:
        from google.colab import userdata
        val = userdata.get(name)
        if val:
            return val
    except Exception:
        pass
    return os.environ.get(name)

OPENAI_API_KEY = get_secret('OPENAI_API_KEY')
ANTHROPIC_API_KEY = get_secret('ANTHROPIC_API_KEY')
GOOGLE_API_KEY = get_secret('GeminiKey') or get_secret('GOOGLE_API_KEY')  # Colab Secret adi: GeminiKey (eski ad: GOOGLE_API_KEY)

print('OpenAI key:', 'var' if OPENAI_API_KEY else 'yok (bu adaylar atlanacak)')
print('Anthropic key:', 'var' if ANTHROPIC_API_KEY else 'yok (bu adaylar atlanacak)')
print('Google key:', 'var' if GOOGLE_API_KEY else 'yok (bu adaylar atlanacak)')

# Paylasilan durum sozlugu (Faz 6) - tum notebooklarda (01-06) ayni degerler kullanilir.
STATUS_VALUES = {
    'experimental', 'development_placeholder', 'review_required',
    'approved', 'insufficient_data', 'failed', 'production_ready',
}

results = []  # her satır: task, candidate, quality_score, cost_usd, latency_sec, notes, status

def log_result(task, candidate, quality_score, cost_usd, latency_sec, notes='', status='experimental'):
    assert status in STATUS_VALUES, f'Bilinmeyen status: {status}'
    results.append({
        'task': task,
        'candidate': candidate,
        'quality_score_1_5': quality_score,
        'cost_usd_estimate': round(cost_usd, 5),
        'latency_sec': round(latency_sec, 2),
        'notes': notes,
        'status': status,
    })
    print(f"[{task}] {candidate} -> latency={latency_sec:.2f}s cost=${cost_usd:.5f} quality={quality_score} status={status}")

## `local_pipeline.video_analysis` modülünü yükle (kanonik tek kaynak)

In [ ]:
# @title Yerel pipeline yukleyici (otomatik - acmaniz gerekmez)
import sys, os, tempfile, subprocess, base64, hashlib, zlib
from pathlib import Path

PROJECT_REPO_URL = ''  # opsiyonel: gercek bir git repo URL'si verirsen modul(ler) oradan klonlanir

_LOCAL_PIPELINE_MODULE_NAMES = ['video_analysis', 'render_utils', 'google_ai']

# Her modul icin: (base64 kaynak, SHA-256 hash). Hash, gomulu payload'in
# bozulmadigini/kesilmedigini dogrulamak icin kullanilir -- eslesmezse acik
# bir hata verilir, asla sessizce bozuk kod calistirilmaz.
_LOCAL_PIPELINE_PAYLOADS = {
    'video_analysis': {
        'sha256': '719335b9ce67dfea3d9d5c4055d03f42981a35651571f9c11eee8b4de554f553',
        'encoding': 'zlib+b85',
        'data': (
            'c-qZ<YjfK;w%~XD3XHmHEv6+!c6a*Db&q;0lQ=V}$)hVyy6>#lB~ub*aZHiAeArQPUA4b#zijRQ+VcVkfTXNsy0@mQW@3@V0dR0|'
            'aNg+m`|qNv2<J(pZsTQ?jlwicc12vM$7-FGqfNGn(o!v>GFp^zmZ~JXju)y}hk3MA^IdOpJfns)-EaoYxYan`?$Wsm)1{hBX1lB^'
            't9djFt1{b!2y3<pi!#djvvIHYCM%<PmffhyvAT((ZK2|_P}yCoHdRu_qXj^Umth&Hd6X{Jn=rqL(`)tNr5Y@<%{I*OR}tMtd6@Km'
            'UB|^X%GD-YMu{roPf;;c#Ue^0nM{_$0~}kn#leJ$diClxe39i*UO<B-bhD#>gJEx8m8x7vDqNHmd>JO!Sss_`4Gys$O`%33rw@P&'
            'eN@Xh8*QWTrnik3IArb@;U(Du!145o_!>uHVI2p9b6RCNd=et`B)*N*I?5weWW9M-u2m{nD-?iA2x{~Z#;J;M6R8DEU&SflIfR+6'
            ')FzD668?u^m(@0j7l6-UFHV=yHiG|4%ValN=JBe;FBaK0jtW?s_kc}AWd(GSB8C;T&B7%tPp-DrJc)~SRKT=$<r*6Ph-k^uWT)~7'
            'KMG}{L@B*h3{VgQcYs4^#qYLxbQ?!^YLUcSq&}QT9+g#|0-?p}XcKL+e5YXjLsgV{1uKz9<K8ovG%nb{;2}^zo6xk_W@!;4Wdr`G'
            'FPsvfcgE-~#{VdQpvz1x<KkxA>-YP;Ri16sY__Tp_-v-)%{I$R7%V_1nk{-gd6q}|ucDf7^K20ng?>`Tn}}g;!*ZR(a}DTc_{lGj'
            'j`1`7M*kjC<4;*?z^}ssLFk`2Xfuek+H7|!EL6Jf_0HcvegENn_RG^(AI{FzRDJF#_^%&*gnba>jKicqRQ)BaUy^O{!EBp^i)fuC'
            '%P7ZJyacm6`p+tc{oNljgm4S<yT#wC542RRR`DXn=C~O6a23Lm(rfr|SrOsSaOrmd;vPo!;pd;<zI%W6e0Ki!!@Fl^vtM35KYKen'
            '`;Yf$Z_Z!7ePfMwyxD$_fX18bmjAusfA6CChW-d|V}9}dEkl3t_8Cn3-PvCR(x6|IVP4`#n|QfQBK!-88~<5U1+brf(ChV<(F&;~'
            'OKzi@dF5ex9St4<tH`Q+5zUrW&b+FK7DFH;Xb%&+h9}cEuttoaj{lI7YXF;AnkXK_j0Raheh?CW5963lh8}Euc=G)uP@|tz2|F{o'
            'Si$ZrVZiv$<td{Uc|A=R_T#$Rgz0FN!!j;mn<iO$J)+&Icy_ogfZt|mjC&K{XXz3d0RlSv7#Fmc^p+#egPRb!Hg4QFmnNfmSitrT'
            '^XrPYoO%!2`WcW0@LvM<4E9t8dlWh|e5!y!sjDj++gDd=n^eGq;<N&u!Ek7?6mX_6xeIqik5LtFw$LSv-(uLjn(&Y6CvfPf<B5jU'
            '$DUy#&}no7+ZA4(y?rs(7^8pVm9z3NPBq}A?DvCHvv9OPK!1evfxBkNCntgVRIi--wY{F$0GHG>fkAKwe^L{CX+ZpGswSrn_2gk('
            'MCz9?siLzy&+<Xv8_ZA8L;)Q%d;JN4Cz~*d`*qjMhPnHp>9`d&h)F<?vnA`nH3qio&*LOcH_<Li9YCk*v;FfA`Mr#%%y$;X-A!t+'
            '*|K@<+5Be}0ii8t53B-ENT436A131oHI6<mqHU=L?{`}s{?M9AfLs{9*fZHaZ?FQ6mLf*<BiD&P=?6u)iwmGtjebSv1c20t_GjxZ'
            'e6zdiQ<d_OtFmQy9V_74sk#YwX?UZYvG5|qyDS$h$E!9pJ=CE)pJjLx--IxoFo*soM@!h@^J-V!xCCY`0wCO9x?#&Wf>z3#PL=Te'
            '=8iNgn6$N$L!#NtAB-BA(FP!)oa}{)2Tl`YH$(wdM0F#F)>}A?caAv3`V|D*zNpzJke`8#C%|k$^5<23kX(I^^|igE(KTRag;cZ&'
            'KUq9Uqk?FL8Pg{GIDqL7_aHf70K(AntV)-Ix*38AhUz=`AHS-p@H;Ir%xLVhcr}A_V-dl*w;X`ba}6Y4(D{o81J8D`jutl{Do@pQ'
            '1kz)fn-5XcYJdUp1lt<wmgY6)<iVgGN|nw;@|^L@<zTT{4wV^Deewe<2mwNzER{4-cWV^H*ICM11|F9SK8=q+naDB_8)5dqr-011'
            'Ky%@>2mn`Cpl@xel#~ZJnKwX)Y6&xf4UQ_wHr|3*|Lb30zB(hQx<eQ*oZLTWNkUppoEIg*LtOx+MSL2rKqb5j^JPII!4e*WesBYW'
            'g%je%3pm@aRkT6HFiNxPdJV#Rc%5b-kSq!~^a&m)7@(ukFvcLQpq2yr$cOhYMt_8UNaiK|Doj8iK)nM+6V}3TWGFDa{@X7EY6B}r'
            'DhHmp5o%MQQo>xqYnUNX{7uZ$3DvKlMP+vd^-J>s-PVyK5D&-$jJ`1b8IYU~>J`StE6OFP%2WHRmp`A;^C-_7&ueZGeJrQ%^9sZg'
            'P)IYx=CogxtI;2UvLf7O#k4QeZ#ztx91wuJSsPr4w;v8t4+<uQV;WmzF+ep2=P?<d1dZnN9Mxg_iSR0GcJCPgtRLzk%^`rSRyYs$'
            'Zq0@;Q+M<%+knUnC(O_M!0e&Lm_Np4A9_;`Y?0v4AmyIXpD4Q643MB~f#L@N3aT}8Ftfl9faHOXA=ZL!=HH3o?}FafVUZvr{SJct'
            'Jq7Ne?<r5hZb_mT*gs~=I5*swpP)WG1*sW1B`U)W)dcxPoNlX9B$T_jT*LSk>!iHV(14Jlbpv=i8Y6bdS%fz|g-3eFUtKl&25xl+'
            'Y6i2DqT0fj(Q>TbLbv%H9!l7M8r?<0nm_{@MVoE8qmM!52GxQG2F%Wb53>dEu*yZ0M2iyPzIUPJXg`4u&^m-Gj(XHW4K||q<Cyf^'
            '0yYw9%?f$p7N$>TDC-P^;o!8?JX&QrOO&X`HuRSUB2F!u5p+#yMFx#e9UTM(eBELU3Nwy(5VRfdFW}9atbBndsh0NtW(%M5Dyd+q'
            '!1qDg*D&w<PIsm&IGoEt0RPO;mUoEK<M);o9o+~=(65+|v<wmFt$2aw=Ab6+(R_|bfq!hA9Egt_P))PEa`93F#v!v@q3xgr4DF&<'
            '^FyEmG<DG*K%VxPfZk|O4sU|*>LoO9knHFc8G1gBlk5(K4utri9j66|yKbOg(l?hj^6Cj$0<+m}AWNgU=N}nq2l#04;5dTdLF7Bc'
            'v+CQ3CPir(xvEmVDr5B=6_Oj+sIXzHPhkpjLzv@Y=|SK34R~2JAu<#;>*)^FO|+YWXggnqgsxLH@UTX_VV1(D^a5FfUJ;mN?;CZF'
            'j4;|{c=U|!vV5u5QI#`i0SO=Z8OV3nG1`_>%fhaW%Xm<ZduOlTz6Zq3etrAyxn+$j)*yQ!`sok&9RHq2WghP8zu2sZ=ZOB7ZGj-W'
            'm23IK*b+gsBa2~ty(SACsOEqg{Hx3>{+tZV_(wR;=vaiA>E=mxOXj>apif&KLER+NBVvWRL$eV5HBYjPhmwca2n2dr=WqsVaPc}!'
            'x%VQYd1gtq;eSE)r|02qSl}Pm@u!3gj{tiS(Z{H+Xh>zWPU0IH$W2H+MgIUfVNLeYGQ`QEt&$E-5Y+N;QO?NR0K_mO7r|h)Lg{uk'
            'kJD4MaDx&O-dSvT$b>k9(@=g>v-<Zw3!*Z{pHn5aLEAcF9%^}sY?7Wn1INj;3Iyc~18Z1={njzIC_J*Wl{W)mA!VjQIaqVl8$ZDY'
            '|LgeVhwmRhefo^-QF0Cnj?i~-{KL&(KdBAqDeBj!zp&I@0s+wHEbs%@gU{1qk;f#1J!YqiIS$o3<N}2PH3;_;aR3r6pjnlojI->C'
            'Sc~E;$Q8U8O6Y5>k5YFD^wHoxW!~d1j{eep8nnP1Z^Im&XPX;j*!-i=ay?pJXW0$?Z0qI=8uyH08N^-ZAgSpPNOwdVeYA!#&gNm7'
            '^2Qn^*>!?`1R`|AX1^xCZ`4Z^;6~&_BeN=qJU~-0mKc~2E*9FG<$Quu8Afluzq%R~#nqJs>@KT7X=UY$7k@H2olI1<MI-M3!9^cK'
            '6yXCg>Z+6PwLm0p9}65ZL{dG~d{=^g_$%saBhr}Au?V~sMOLk90viQgQY77?_O4j~je&|hH?Sc#Th?M}?xa~~q6SBch?HF5`l}o{'
            'xdGFMS@&@}<pZ;&0z~u*^=@K7-(E-QwxmBtv3%?9uV84wE9DbsU)mpR!Y#s@oZg;J2&j1wo=*OI-~RM=MXV5n(P&B=W)R;!J_#P3'
            'oXW2?{c_qYPRVbR#Pg4^y{Q90K*8J=h4xigl>9jUI9qM#?_G8K{mEqO5YzMQF5Sc*5sT<9j<T_Os-BFGjZweKZ)08@q%&T~5M8(*'
            ')nbh#n-n%Vtuv4ZUOoN$+Yj$+^*%;7WKq_h*%TO%LY^c5LHZ#B$r`O3qmc#BI4-NQB<F69j?^5zwZzHtXjLKa0STKp&e4ao7H(vK'
            '7PurtPr2&<U<Thm!Wjd~N^-`WuhH{|UJtch19Jv0w;av$?5==Lfpp@POP^u7y9;+j)QmrDOLiKz_~byApjL$WbyS+^lk--FLMoO8'
            'a0v?nlof>=K~9bkud5I^a2c`hC0w8)YSVYPzy<bcgpS9M-;KiCEMCso<Og~s)A}a7iDsXoJYyQoZtD$!6eO-b%PyGv?09HNoOwik'
            ')2TV=L`j#w3zA3i$Rks%n!Tx#<Y#$3FcN=TgHsDbsP+kU-jPv=-CHFnhAZGPAO)s7?SLX)I9BIX4(k>P-#M}=b`ynYQeO(6BsQbi'
            'as?coO`OXt;-Mrc-T{~V1e6Cy<mB-W^&$}S94q6%MlzjMz~PGhRtG?r{qY#>j(kWA8Zw7{c<9sR;>qcyC0g(rG{u!NE5pQfYO<l*'
            'VPOqphjXTP+qCV^K=aZnS3Hju2z>=e=T%ZGql<grqV1aI1ug9+Sef=vsLduD+oYua(Be<|l3@}&7FjMv-%a3HyfjK8%LTvn(O#cC'
            '{vN|qB5fQaPE(frkAHxH@^+_Pe@uT1C)h4Fv*P#%a@k`qp15T1<b5k(CjXl)0Xnaci7(=tFpnQx0TT1IL`p$VDB$z3SazqlB+`jV'
            '*hek50D%~>zJ&8+Q9oX6tA3Ykk8w}HL}uk~TQ>kH8%@4E%UPVRvQyyqSpx4~gh0hzz+Bni$o%ikGzwMEfX}iOMR1MoIM@orZ0fTB'
            'sSNn$3@y^Ij?p#w{&@}VD0nFf7i;|jnds*uBlR2&)e$fo{U^e}f5Y4B8BivSael-TI7+h_^cpSJGk&U$Q2i~APSu42uLegSz(4qX'
            'h)&OPcXWSw?^?@f*Ym_?GBM3EC8+E*t{rbP4lkQJZwB?+>cyP$KSO81ruGlUj>gpt4K_oCYll8Agg36qY2ci8^_svgME_Y8UPlsY'
            'GKWK&1~PpD>SMS>TMs~Hjle*`7kH9v&@5R@KlkBif>sHf?v@jMQ}j>OlgZ?Ms1S`aBkdL*I1K!VsY!-g6K0-O1rfuP>CbVsSdD|<'
            '>WJ28MBaC4G(<yhsf*`&|F?M%YzAV(9*0|um>bA%aj2lw2wTwDtD@^MJ_8;Bn&}|W<FLm-=2`DMXBsn<?HW_DL0C>4Tp9O!-G*#K'
            '7`lR;(N009*2Se~yoJ%=%P~7x#b=?U{TL&2TKyQZ@E`$Kk@LcgNF(fXqg~%@R-iEUyz;?c4}{&a&6cvY+uFgau2%RSn$Ty`2cf;?'
            'Q;Su!bz;4y5TEe``uyGlbl;$!JqYPU1}<9{E;{glP1@1WM{0wweZup;vbb&wXzOjutots1ZfWm+&<29YR91>`R=>ZuB30^x&Nu2i'
            'M-W3$9=<ip?9%}{(7t*EH?-bTuotc|{A0E={A>pAnKrXF;%Q3EgpZDH9S}pJuXZ}S@9mwnh$I}AJfj%pVqn`Abx?FIcEDza*6gZF'
            ';-b80i7dRd0}KlF^X?oWKPOl!ybdv<Pu1Owt!|E_gCm^J5oqCCRQitiEY@;FNIjz^abOu8h1dfg1V=m;W9R&)8FFX(@TQ@?4pYD?'
            '9IfbhXVXb*qp=AR*Ug{G2%pP~A%-sE1oQ(I2z0yIB0(Pf+2HZi{~i@zfe9~t@b_b&h=GP?Xx#zdV8r|5>xytqk?G6m1|U=*zG`TH'
            '$nVu)E*_0jB|{!6J-$F|6|9a-E$j_yWq?EwIk^y%vkfB~zPE8dX!&FuE|)Vsnt?sZK^w$z3mw~^5z$P&Hh|Ar{dy4S^;)T!#J7@1'
            '9o2$hPYjl9G2XSh0RtB(O%=5zHlP7YGXM>eeIp1^_Py&!x)_-L^r2uLAj=m~gCnEp)J`qrTnazX5~#?KQFv20T1uI1XFw3>cb=-_'
            '$q<bevn2pP@yK!<;M?(&JsyZP00Qc|Mi&=Ddvbh2tYsN5O2b87QBHxSWq|lqkZqT&O5_O$BwmO(1U3A8M6xId{^@p{F2g)Wd)HCE'
            'X*{ACvVuT<HP#C7Ly&66<MHLaIDg4FH{U7t%)_41xe`mXyrh{b?=nTuDBA#)AwaN?9mA7^hkzQNXGO>hi6%MR8}e|B`4oeOKwsoj'
            '(wIdy-V|1gRTNSZ1Uwz=rU0+T^0fXlxR|gYNJz4qqr?w_c}AlA01y>&_&c;40L*~9FY+$HJP*VI?%gm>JSjC^enhe(z}JXVf_ia!'
            'Iy%0zOexr&Y+6%mR}~k~`qJ@NgLXU!=f$9IFj64DZ!Y?JE&G=N#y0!Qz=MG=aWwXDJ<->GWrVXDxIV#6{~RjV&q*_~g&TOn!yl?2'
            'f_rBYh+Ag}7%a-$tk)hP<&283XgQ5>R;NuY*qnlAfJ3Bz>j7`@Z=zxNW%r`}lo(Ws4(nSyP&Bgm2dot?Ye5@JK7f5fYAxY_hd|zx'
            'b$q=xYs;UGF>Dj*%Y6ua1?H%uW+Mcl{-lnh(eX(`x`QF0p@9fi28q1#i8FtB!=Ny3cThtPj7N5heppTnn$2-3!#s|PLDOw2!iWC7'
            '*XgRV+bB<fl0em+8D)*7Ddc;X4>V_nT@Oc^vg|c9(LP2;KWtEdW;qB!OpMaP^#Q|5{V<+9G%yj{;n?P>S*<7I@18u+d<9#19xiUW'
            '0Q}pNj#QQ*9T&CKc}D97Vj|`3Ew!~>2aQpt8i?wf4QI2Zq@R<l7nc$+c1}}ahq5de`}qVG2w8EuNGdo3*l);DU928&!W}z#Hqey7'
            'k>dpChB2(<2~lYj=9o%IiGi}1z^pmml01qq1%!k^2!Q6G2xF{%#STico00WH1FS*8{Lyx3@Tvf>_-$20OBSdpfzO1xl7}Z*5}>Fk'
            'M;elvN9z#HH8~?mTw44+utV)h1$MefvV!t0*IB8({b8yy6G(TRWlPMqQ<4no*x2l#e;t0!te`>D!Vq)}%gh9R>vqUWM5nYor(Mkv'
            '*WB2JXv5L^g#)E$(LJ?rqp!Sa$Y#H=w0St2M*3}I`&crgXR2yrAx?-rKO=R)>MgJ>WtQ1ox`ruw<A>ST;qKlfv~>Zd@a^wC1>INI'
            '1Y{S_>;!-de=Q$53nk~FZdsGSg-sT9D@4wvta#0QDSLy$&py*a*VuYAXp%uqO?q=V<U}Yk+s?@5b#H7C71_ZUQJv9TnD#y}>oY*F'
            'dGn{Ki#;u=C2$hr+Wd3Sf127DzHA-%P7vQRp@7Z2&54qgSEsc!2D=jyXq!dANkT2tBbX-h#$;SLq%~?8{hzcL({YAc0%e%Q*J-pg'
            'blL#c^Fp&q7)^AI^XZ#!9je&%TO5&oH--TH$?&+_PTnBcGZeo;zgK5pHI`!J?6I#v7H`ca<XWLd?qCfsr70eaCE5B8ud|2z?O|y0'
            'N6~?#Z`AW>h4a0|_>o&6y^!QT8t-FfHas-qlqdp__`uRAow_X72sD;R#_DCsDL0&nPf|l^L=q7j@Ql{7g%(Hn3~o>iV8eMf^bXsg'
            '*MdSIGwaFFIbZ`fb!KW=T>IM$^k}q(_IdszLH}qO=Ug!%Oh$yD(e1JAK6c0CC}ckOw~gI>dtmLYX#7qaIR~6A<g|6(>~HLMYd!ZH'
            'I-8Ho(e?xDOPl`SJNqYpb>B>L^Om?ya&wjuAM1q^-H1qVX4Op?)S;H`N{v=DNyFT)!*uBhovMwO3vV7fma1<3gSU{YZ+s`&e>nz1'
            'xHhfH9!=<<Qlq|FfqGbV+B;wo#b$(5yWPp2c5yl*#bE}sMXz|f0r0yg<4Nr-aP_oj7>jvWu@h5UMGa!CM?x`g96`x$e*!ZiFbrjM'
            'dvb(Cy9R~Bj&Qf?vS?R?UF${c#Z>Pl7^`R5W*(;?8wg)s;*yp-A>a|HLiHoE(80Ega=qc40DSXZy;1O#S>b#z2mqOT5`CnZE#Yv;'
            '@oPIc`Yyuocu+VeqvOeh2dOX)p-5s1AZ0R^Q0ePDik2~)8=QAkaAYi-{BT!DAJ8#oGg-p8o||w0fvU?qG<`63L7*5r8o>N+hVF@i'
            'MDbXd-q22^X};x&VWdh};7MGgivuM<@K5n{hmT-|2-b6qb9?jly}Cy1V=W-JZVVRSi6ZlMCT}U!z8D0q7D|*flUdiMOU)y+l2;o>'
            'S+d8Y&u32{`qZpN{B+;qKa*pdwHV_TTCALY$PQ?QBDD?zZEY6-R4XUO#7|aE5Y3%dr>?vbEwrfYD$6j=dgwumRjHPH+7vlQixd~P'
            '9iz~4SorY<@Z8cd)lb%(aI4U3OOIB6(i1X(E}bk`F9PF&o!yuAs9I;Dhr*v5FRv3^QBW}Ccm)_kdvxBvcslwY;po%<`Cp^|`@fGa'
            'zwHlc2qs6zjm$uAZWCkp>cWXMxOB6V_K&;qt&a3SA3N#P>_#7Aut#gG2YqDs-&DR`EVGMZ6BajECV?Fbp#Rkd81>)uW3yO+PuuLw'
            'KwEhY?y^xLMH4RUI!(qSWjTY7$5xRPXGec&e1&B_IxxPXHWW4ED-7AA;T`?)?)$cdgfH73x5Pw%bXk#4nfM5<uHU~17;+RjU#iCr'
            'a{Y!R($wvWYGd(=KcP*qUQ;4Be9~gUc%UC%b3Q}+B1NDNYK!Srm{AuEYC0LJf1d<_$2ifMWUK@JdhvPe%|mYtH`~qLG6q9PZx(XZ'
            'arXkx>a*RR$R-N8Iq!NNzQqMPMlOQE%m@YmlSWcRrHrP1d|3Vi*}JpQCv;@t9&sXPdOCQNkJy`N=W-=?b%s7w3>)-iD<>Z6hh4ZD'
            'woTR!@wIZqK!=1uo*I|b#oYz`W|n@33+L?oOSZ4U7x>DZ=e;Fi6|QdTktE3BlVNj1kgVLY!6xJ5CwtH!A9NEAyq*l(*B=(~iLKa?'
            'KQmoMABV<Ek;0a&HYD^699&pd7GI_^Shaatjda5B1@&`@#08@0P@O!8+M2^(^!*;;5S})M(P%}F=~EsUZCze4UiC)}7&SG>$mII~'
            'M_fmGIO-4Vlod#tIeutiDMlTqkaCxvp8;D~qKY(j4CEu(nk(1%Z$7mh0+!(olX{{;+he?b&{^3D(Eh5W58r3%Opp19?tF-D)_=Fq'
            'C5)EUz5#lfgN6@wz|Ln|KKoW2(iT>@T2hZgL5wBA4Hu~Eds6Z`nc5vvcG9J7&$kc^Y7(UBaEha~q=jrJj4?p5t=HnD6{0Ys038HV'
            '#nJ$dyV{QFeC$e7<cXXY0~c!sUQp`@`W3@(mPh%?SWfq;-a$j~V}OW03&~a{x+VA3C~@NsJTsAF9MTq6ZQBB5b~?_R!ND329u-^9'
            '>wVSTdc4iHgLdhO<lCmfsR^v1;S9TsIn$2kO_)uC&b0Zc*{sc+Gi`h5Hg%WYLLn;T#5qw4u!sP)(c`X04gKBXz2cR%zaVk<Ay_q6'
            'YKNr5lDpveMgt3wSIqER)i7;70JJ%_PLN_0?E>iU2m<LpbQ*u7p4vHqb2N)pOD=~CLR*444x<s;_hV3Q=3!YTXfaY;!C<VOqKx7}'
            'WQhgf<>cH#xdqKN#(%)MpA|YAbl*rPg~Eyjm>=rFBPYJaj;TsASM$A%!lcldjGW~h>V#J~lc9GcTn`?l3Z*!Nc^Q`&tE<jS;`3M~'
            'nA1>euFDxh_>hW}da$^lCyWJn9Wc9M+_g08u#o6-S(GfPd2y{4izXFx?<g`(5sa4DZI&6QpX&xkzG!lEd2wV*mPeQOAm+7zB0lpV'
            'C`TOahD9@f7?>l(c0v<b_Cau-%nR_E(7siJexJOp9cx8;QD6O+`rF$#AI@JtReyi`HwAp7+=ZWFRe{(6y5T1J6!e2VJkhT6EOvnU'
            'B=!UH97g$2Jf6rs_6f_5k`AL|qpXJ=c5gv(T__p`meAT4Oywl)vFByBilby{0{MlN^*gG0gp}*1{_SM4b!@3*Z?5Ba*gS2I3V0OO'
            '#FmgC%ILF<!xU<XrRF5*UAM$(AE^Px{Od=QWO@{s;)3-YuC9zt`)(a$loq^7;uOn7I5oZELLqa+z${pauu4j)TrG2#dh%Q>EYkIh'
            'xww|9Qp~i!jsRuZY6WU`Ar-{KCFT=PxQH>wGh_v2wqZ;dm8(EDMFD15<ye=_NA!)=iz-P*k_B#Ft+7@&^KT5FN+P!vacba-Sldp@'
            'P2}O7RSHeX8P?aC`s|yX7g>UZFHA+UM|h3_V(bZFG?=cj+$a!MBVE_<9T@BlO|QYBxdt{Jey)Fbi3ZYvt`VnqG#|tu>vOavYIIX9'
            '(Nx2^w*-Ts|4JY4zg0G&MfAFii}k4S&7qZBDBZn9QnQ9Vro(A48S=+NAl0~>_HTev`a@lZa=PkI9)G3}?qL_ipO8BWCQ&I^5@RsC'
            '*s>E?y-FpY>2YA^xYt`&2e1el9s|zn*0az2@5K>{6-Sq+lau8=6>_iRqWFWp4U^7P$G~j5u)+hmq0j;9xbnmh$)#OBvte8RXEWZt'
            ';w+8MgJB~n-Ydo8mSgd!=zaK&dL3>}DY|4gR=<kygn9KtU6uwfl3@10u1G1zfgHjEhu^$@d;Z>36CoB8>hcsQGwzTVQbCYVR|w&t'
            '+9%^<A6kylrCgEcI+nN-eQGS@)hf!Vuqq9zgdS`51|U6PA~phB8mpuKek44a@{V;uy#j?|tf4dZ(0Oq2nq~o;(a9+sc2ad<PmLNl'
            'MDmb})OZ6wXC$p8#D*@~U@=iwTyvQEDF((?=}n4Pnfx&FOTIW|U$EK<gQ#oI<Wz-fC`TY8pRt5ua8~#gTHLZUa!!D%L3MPsx1Ekv'
            'pg$k6%`z>mrCPe`usAgCZG_9is%Q9B8#*dCOdAy-J9$iFOtoV)@$DZz5atJ$y27ol^>9qJ9(uM`2jX9aq6zVA)W23(;B$>deza?v'
            'E9*oe7{%7^V5KK?%1Ys-DC`UYF3%^3rS3Q<1oQa)hs0<6=)Ucw==Huib;OkdRnxM<cyBB{q^~_-!bI>Spr{k8#6_V+7m#oe90-e~'
            '+X8EmLd|%iL`Ij-pc=ZyOCN+7>1GEk>C7SWqBe7!1lcALpoW5R`leg12MZf&xvCP;c{g~J?F!g5Xb_7+lw*+<(WvZ(93EH3ITi#<'
            'Hsjul53gR$&YwMfBR3b!o_~1vlpICpXU{Mk=#LnN3SiC=Kbwq%#A^}<iPy0rOETqMNF}AUWY2f~R7<q*3O04c9)!pDR~Yo**K_<-'
            '@ROyuCK4`)I3YA~@C%D}#9LY>@cP56_b-2b_44c;4FAM>|N7~FSU08|JFnipK6~^2eD?F%JNX2IdvKV)l1*Kk7Os<Q&YCz~zC?FO'
            'i6ug!d#K1R$QsuyY@4d3xP}Ptt}X?_=^M3B4KfCDOUBh$okt+s&Ja{2HTJdOa%Io@FR$MI`RS|K*_*R>fBF0D*Rz*@`RjX%Zu;J<'
            'yh!Cq4y?N9RatzfMB*8DK`8q5uWw(S$x(JBd5w5PSQQO_e)gLD`p6YvPl*!rGj6YA%BfH*e`BUmv@EC`S`_(2%$tTg;{94@rO2w#'
            'o^UoL<$#hI92CAPFR)+2Kx^GZiIK5uC+fP1cao{1r3=<}lw2@oiz=@Ead@r7uP%4kCLjKG)sXx;eE;98A^j$5&KDfYUsp;=3hEYk'
            'J;;|-QbMwy*)}vBAeAk)la(o-)lW*EG?R><Wg&Wk-n~B8{0fhVE7SZ+g^N5ZP-N9x>({5h4A~$Miy)<U>hlPsM^dL?hb?t<8pyOd'
            '&GuAbVlN+5<y|(y9VQHbrRflD7rae**c9hQf?c83Rl3Y^yW`z$(RGM5E9Kt0t?&rk+RPF{+3V4WbEZcyuMsDO!$j!TuLx^2@o-ok'
            '#5^+kgQah|WesFIsaOtuid8z(Rm2si|9U0_&Tzg9Z=*UxfMe-3UV{fOkTIWjUl&6`MNSNH?L?&2j5}Ww&%yzY>DGlts&zk~_vuy?'
            'W>NOV8>~99D2d$bIx18B)D94fm&Cs6uW%vgsZ&G0jGT{X+PPb3A7e!b8f>eR2P>UxlL`>lXbwUcoGL@9Se>fR4Cs;i<2}A~3#`*%'
            '4!J_=I$B^n!Y7sGDfR|B&pQ5OUlT=iG~=vtahxAx4XbiMojhvLgJpEbi$|&r1$s;#ypaM;Vg`S>E4j~sJ%C1_8!|v>czVEH6uf&R'
            '>Jk$n=M`$U$mMm8GrWL_9sdxFLHB_N81O^W`|bpt-QS)3+Yf)VZsIXlx(Fftq>h^dkz*3?9pV9usbu=&$EF=dVKhpbHb*UgU7Rw+'
            'OTbaF203^zIv&=rI6x_cV{&@jz?L(DrmEGBH`<%p+HSHPcwHaM>LV$T<In9<Z(ZHmL{Sz0nb*-&^t*y%USlW-0^f6^JN$O)Ov1^C'
            'ua9A!8gC`DOY(x9%Wj?9oI7uQu1DV4!qB@34y$*7^_d+i=Y<(^J%rIQMpjsD0g2jj>qnE-8T^1YIWq@xW}eb%I8+a%w$~K^Ujq<8'
            'ES;KiVv?|k<{fQ$mZO2<q=&jL5(|W4`Y^>X>g*7fS8!15kmJ<|hVD~Dd0^YA3%e{Cg0X(0HI=^c1FB{1jlHoEyy{1RIx(znyr*;Z'
            '19=t^+Bi&igCOYc(y8~NX&U2SDe^Ti(hk{XW=3HWji>Mzu>F?G!)gGC64lZWt3>Y8gAW5tW(l2+^L<xZ$j>nd9(>U&tI=XsxWl5`'
            'wQXT&bZ1JyZ{#tEDE0!hQT9KhTBAW0Taw!_iUtHFk?gn#c(O4Xi^;8LmRXAvJ57tgmDw_hRc5KWJV{03xnSRu$-f6!@6Py&3Z<`h'
            '<BMuo`3E86%9!A0oD+97l!QKG+uLBU&kfy-h;DY)-(0cl3vNKvb;G{)I>9pB=o>LgEEb8^v(b%$`>ztT3Br>KtT%!4v3Dn+oE%M('
            '4H`SZDhlU9dT4g@EV+*KBwD4<f0B2hk6S3pa2;DOqoPKe<}-_-{iG(nwp6*L6+K;UxpWVT7<Dk8u&R0t)<r8xMuB*UXnRG5_EPz@'
            '0y2q*U@A*@Y)3S1!?fkX!^hlV&*c{_<ti!e6xA(Gy`fe?`*nyobpHGib-VW}K!4mpugSjl)vBb9`lm9!Z`JYekb0JFmkc)BwaNh2'
            '!qo9<&3WwwOkx0wF71~M_kC`J8M;*(<JIFc&DQ#zWi554DPvD+Wl+|WY<Yh>Qaq@YywOO)@~`e>T|nO_=}q6sbh0mng!Op{b?$Xj'
            '&|FX{g`^%tJ<8;h@src0IEx#-O#mBOW!V~nx3p|6EbqoiTP3P&dK1`I#tOx-*(AVy+&5<a)3h70O^II+Je7rZ7>$h3PiLd`b%)!T'
            'VQWp`pk?TQVoke!EofKzOyT}gQY0v#){e}1IAr;uU}ZC+?s#QWhiHn?d4t}=akQ`}&?g)%?Rapq63J8d+qR(FMl>-ur7(X&Lwg{x'
            '9_wjuZz*5~@o*@Xj=7#rT~V@y5Umy3TNkYn4X;Zm8kk;&*aIxSGlQ(zT}xuv9zD;Cnm6U`9)M_(coSMpZB&fTpGgbXcT<170<&KB'
            'i7u*F$CSwWiD}wsp-sol!2Rh*zd4#to!sB0NxTcw8{fs$^e?@9bN=%AS<}BHk!Ouc2Uo~?MsR-BsYvXoG{KK`YB~9KQIl}aGaZJD'
            'Ap$1AD^&k}!UeWM^`BK78J{4BUFetEd4r0|QPK1t5~s!Wt-HV~l-HEYz76%Iz8l4*HKeSV7eBBHZ_yx4u?3`k2>3PaR4%)TH<n<S'
            '0i}+l=<}0GfO)Jhxug|lMT+iS0v!q~OcC<UvcMz+I%l-A6Un%!v`dp0@8x-IT(Gn@b{v!LHX6xV*oFzjTu)yQtB&y_E*H9><XsAL'
            'q5DM(9eS)IA~FfLOX<!g@*x|SQ;AAp9g8EDI!{5Y6<L{|RA5tKBEmeA89Pq3Cxbq{5}!3m{jiVL;<E<S<6z<xb3h_5YEN69+#{(D'
            '^zt@XuglifelF@jh-M&u<6WKS!~32iU0_sHd67oNe-WpUK?A>+hqb7qOO%nu4q6gQjs%R-sz3_hA7u98PMqRUSSoUAeb@4tho+&C'
            'w^H>f-g-P~Xhs{H_F82$ZFQ0%gOceLs<cB&TGqW80@u>OPluD#AiW0smJ{8&<67)q5w$W0QxsEXKv{mC;rgNRyBX2-OLs^IOspZn'
            '4M0DDA?|nZ2gbADqogJF{DD}3N9v(z=stu}zF=*n4&gG&$;~0sqEB%hGK`BcPjD1{OKu%LP+FCde7VrsOw6fePoE()^5eV@JEhXS'
            'fL!5J&-<SePtBFh4+^OIOA=AhHbTNl&?%teV7W|6TL`X())s$jNKNN-;UBgDYi2I@apSdI7}z4n)4^pAhq&bIrsuH!M-SD0{aTx|'
            'T&6Q?x%w!<+vi5N-;H2i^Q?a$ptg&W%*#DjBOOw7xAQ)PgG){Q&W3i(OcZ+ykLADkG@5o;c)Y9;6U!}$Bh9ELCuZUW$g@^9NMVIY'
            'oV|Jf^4*zzrRR|9zF>iLN@LLCHPR<jou<qyy1A38E@cw<7hhLdRp>Oe!O8cN0%H^;tAh-_+M;g7Q;<9u(@oM+LJvh*n9~+7d(>mr'
            'Qngw|cjo#%Eq_Yjo-OxoGsG)^4YBC2A3A7BqLNf0vh|z+l0q=2I*#zwM`C{>>x3)d3QLqBPxZ}FAkWiX$9Tz>2_8nFTZ@epyo@P('
            'ShN>rNW!xwD_zSt(_$}1%oK9LBRWA#?eB>7y^j<HCIa_BWa!W&7fNj_S{4@5sQeVgk_^T5jyzEF^ulrD+`-v~(d1YpSQH|>mvGE+'
            'Z70ZYxWU<fygYyZ^37im`bgG%8*e$s2Sj&DM3SIAwE7SzjWLerz``B%RaBDxEB9n@YOh^zOjj8L*g{1m-3O{ij>H4{Q>{pboQueD'
            'jCcnlD-E9C)A$B(;6tNJ5Sxy$xN{n>P<I0bPHs|jXcDjJWFJ)OXnl5#=#N{eB#9C6qApOTDPL4@W$}Y6uY#9R5ACG5e_*&e?l_88'
            'q?9f6O0{fK#ySP*!uGl{+0%eaWUYPc%;{fAG?7e{YL$^Hj$t(mj6Cp^nBO39%{skw=%Dqe@y(&pg8!nfwaA1*nTv;Kmf^$ku(4?3'
            'xYp*?67P9s(Z!+}oC0c2W|Xb5!t7Z&8MgQRNbLzg4s;FeZ(#F%?SI1F>=da2qvHocgbTiU{e5NdNwJ2A)YS4xK6KAp9UiO$Qf(T0'
            '?2!EUF1+pQ07|k0Osy*ytUwVePSA`W!N>k#?Zk{6qk-Bt8v?c6>tlh!HQ_AT*t(>1Z_VDY=cYL5O}}w<YhWjG$dtQ7G(BkCST4pH'
            '*a4zLYflW-+jIIOXk6)vxfbSfUAD)d>gt3TWBdTHpeVnk^VA%vKjLZrkE*cmM@FuMVH}aVu43V)Xlnzh1TT)4?SVcp6I+161i(8%'
            '#=y)BJF|%st_;5~kV_X?oweaD7iKPE^cYMq7|&)YuKMg=-$3i<dSG$O!KzQ0q2WAClUUe=+J##QrNt^8$#sOH3{XBW_V79ebc74o'
            '3xm&g_xC#8aut(J&%eN!;?6nL6XRcz7XwcZ%f<TC2}^zbXExZu=+OFTT@USMvn&lZB_W1#&n#msE#A2M<|U7i#uXqSy%}uc0#j*S'
            '=55iK5R!M*3-FO%&vig|#@+bj-Wqo^^_~rTdsKjF%d?^5@R&OMx;-hB?&;w%20HDBR=-Wfu@ZsocK3M6*G@_JL$rvKr3rTU=Q;I>'
            '6V*ZIv&Ttg9DVP@gG2hWWS`Z;s<jo~hOiAdz>=;Z^*_*8?zvpF@!GzT&;C@#&TCXOhvVhr>YOUu7$CT4vjrS-GpTzZ%6K&A>(Y8x'
            '{Xf&v>+CR>He&26qNB5=FoXaYe6FK;F06}pFELEyhe>ch1}=0H--LF>2CBHC57(MLFtJ#Ee%27l*VLL->>kly0L7fytm~9?jQ}o|'
            'Oe5&#Li#MF31ID`htE?p*P%7y2Q+k>b@+9^<e?)HT<$rgbzRi@E@r#d?o4hOZd_O0teez&fAzF!*D==~yOx@kvQGAO$ZeJz=NfBa'
            '1AFI*A)?lOrp@4q4witR8{fvORtC{|0d6-(@zBk|Vdiep;7~{&^Vtj4xjNMC;Sg?OOksF5y7w(w^sLQfQ`c(N0>-;t1mjJR8_wL@'
            '35yH79PW~oDDIrpz6;NiOoS+OyZ;Wupc$dA!}n?^7b34;4xNYOHt+4N9M%${=x@y6pq;L~P;XQ0RAT?M2dmAYLc&X<N`Es!px*p@'
            '>hrf0fNoBpl`oODIo+2?+N@Yt+r%4gO5Y4_hQ*zeMf`Jn(2ow_(8J{49k&g*;T$a!Yj5SpakMk#M%#PO4<X{qn99p72Tl<X=YSel'
            'TNFw$+KX1qt^mwi1=Y)1c(vbasQW0<sY|*J^FrD>*OCwULTu0;B$DvHA#Q(-O2@+H+Z2<_jV>n=65n)^VSq)R-Kp@7uUxY;uS>Ft'
            'Y0J_<SevxsYBFp^!`|EQEQIXW=I;GSQ(LEF`P&y(i~tsGBFd7aIb9q{mq(N$X||KE#+lOLthG^1pH@+%^_1MTA8!KeLFMZZCCpLA'
            'JZ(wjXzh=qi3j_NAAFHA<S9h0vM(=*EI_RkNrYpqc<f7g^-4xJ&3wRg)jD3x<J`Vo0adbC^7l}w_hfGrx!cA{X0Bx%{3iMv*}Gc|'
            'ZDK)3Guu-bW)h0`1*@MWNVlqp^Dy}$dz)Jlwmn?TAbeP>;cZq(b4{{6Xz_OyTrA43xKtOzP&Y|D*G1;OU;+=8erqZPcJZa&VXiu~'
            '(GEQQ*{Z^1CK()7F^(r-*NmH*I$u4>krF%B6o@fSmbQE`(=Tfxcj`&md{i$E>@}<vgT5Y?^+%tZ%EP_2Ve5-2rv5EGQ@Wf-*Lj{k'
            'KYQ`?!>jia$Z>92UmUzo8`T}ORAD&d)b*dbr9r2Bw;Dck?Kxbi+A{$;v6{ABiO6>9g`FB|CUK%k*<S1>tmCp;(APde`(K*aPOFy0'
            '2%V?MdEI-qxNM#;0Xo5|duPd46qsd$M&C7S+}Ep!_U-0-?Xzr|8V3gNL(V<_88{S-fv|W%5J`=lXz-f2p02v7c6NE6+p!nqiEt&<'
            'ev8rYFr4<;Jv+l;LJR*s3-WB%k-H$(&yIb2k0qtpHel}~b~|Wk2(3DFT{W{?qa+1#Rbw;xsp+qqG}c8(y<z_Juh4GI_HwY|SV)Zs'
            'F25V!6*KWa3Q-wZq4^=Y6)e({%Oj4RYs6hKg(_;p8^(q&SOzVf)Z-Np_Hvu9XLAat<r7f`ZVo~RDu=Dk+P6A;WJVN}>s_m-8=@SF'
            'gBtj^fp}`=o=9j!o0PKQsgWaG#Q1&-T=uTBG-XN%NnnXi#8NL%tmSAmu3ABX#a67XFB`eB#jRzE+>l|46sO8YKB&+8!v*hubmEDr'
            '5EBZ!FetkmDrh@>pVQaDgA_v15?aa8##j?B+!Jry3b-r$xUC!7vOp^18j?*{CbFW7Ky;z(M|@oq0MsQ%sbH+NP87>Q+L}b;Tfx`f'
            'GzM7N$kX?OAGtAsseMs8tOlfq2rCguO=@X7ZqS`o9+gIG0y&&vyHWzj<s(ty0HkbT-#;>!rsN)0G;E;r-a9SY?2vR~w7uwVYix*K'
            'Q~mtmmX4jU1B(5Ptp&A>)e2xV-2w@f6|SVz(*71s(jV2YPv5<H`Q|UD_C0wRC=zY9<xUN-Vz|DGuVjwbx5FPb_+GEKNYLBl1c#W='
            'wKv%)+p26%-r(E>=a`!UG4M;_G8Yt81FB=0Rf~1Bq=E(JE)3Qru<khp>DHC?`c`FSH6D*m3)W7#g_dZRrb^!<t)%u307X^;h9uuo'
            'Tw=%9Ey@bE^}d$eoMIiGJo#Zpu6i;C;8y^f@wbexS^i&##Ts;@F@HV?>WVs4OoxT(&I&qwIYpJ;;L#c5-4`6=d}>#585~bezNZNE'
            'lkefIpZEKoVS)#vO*#b6N$8UI>*yo@HP96mY_&z=IAz|pY&pYAPdVktMFP;lIc0zl78Iy+QqPGB)SIf;(IUqCAuvFI8Lou!0X0@h'
            'Ij++-uv`{~fMW>Ocv)?>MXL@)<3aN{#{bO1ViCtv@>~unc$}`>VV8&%#*9Z?ndQZF&>y0Xeu^pS5<NM^;9nMw$2^KbzbaRwKlU9o'
            'dP^<i4O;NZN};HqCYNl`YnzDVayg;ux+w}FBySF79LPrD^?b(x%kuOejYfr~tcjjX=ah-s?GH_;RGb_2{FI7zh}XyKGc38W7f#z!'
            'Mhs|H0!Wz<MjD2?@=-$3c2{tQl<_KDloFqZQ5O0Uo?&E!r-Z9agclUzEW=PyYvlTZ=;$Lfc93e0A*vOI^*gsoNY}P!2_}nC{mmrA'
            'V-lHzm9)i?FdSRE1&)6kaphV#jZ?~UL1P%qQd>KgLi%1w3gDfakPD}xIp-Ec1QLdAir|kLyRVL*vw`d_I^KwBEFy}Z)Ag!AUvhxm'
            '8)B!$%B4Q6MRPb+;62XukGg=AG};%;$8L|=yR%<jp8b0EeD>$R1L_NvVSLV~AL|0Vi+B@b<S?N41~x5d#IZIvIo4ORS&-l*aIb9x'
            'xq(&hciW~3zG-=H4*&e@)!DQ6Fm64Mr*HnIw%Ml`=v1Md60Mj8+9!2o431$1uszEq1$fj0rtknanISmUcyOISK0RF6%iF)aH{{{q'
            'L<-h>J@1*MC+4?JdHUJu<KD8y{@BnHHqk9!f`}g`VU85X$vxhP0xP5|aClms55%z)__4_xA;Wzu_d`m(xD3RzAa#~z7SIc7Zv!$f'
            'srFkZ@-8)|eV~V4000915^p8m-~njx;zPTy3p=?>YK?Rsu~NWNWRBo)m&V8$L#KP*d3`TU%G8!ObG~qfaM^3Afp5};<i=F1(6oCG'
            '0H2+9e2{qqs`HFkv;CF~Dpps>-kqBUNHiORHbjD2OO<{HZc9@X!3X%y%#=g(I<_DjP!MqCBj1#P&x#!~U|@Pw*Rvjk?X7Ac$`wQ#'
            ')w-jEiu$<6_`y5e8!E+Cqs2dskYw}v$TXYW8~Pvedw<*aUxc63F^PHcr#Out8}$#p2OD^zU?br5j;Lbbl_R;m{W}WM$yr5rKZ2@s'
            'Y8M~qYhi&xW6;Rs%Ne$1hjlKp3-V#UY`)bR*{RaREGH7WlIV`(wDy3h+sHn>Fi_A2*m5MH<;>nw-Jdtjn)}<^%Dhwk#Mz*Y4}x|G'
            'b*Yio_WHHox_NJ!51{!)7mThU`0pDtpp4MQ!0D;~tyygMV|l#y24lB{^{U}(uzBKXv-Ez0(~+6DnX=Y*83cfGA9Q_fc$nS7JnXVY'
            'x16nHpylVzr$$Vq4_F**`yedq5AYYB)hGs|MQsd+AO-e(f_aata)F#Z!&*zoqZ`nS?X4D6`H?57>avR#fNsqfz<q&Q9CW}GHWC8T'
            'Xj~ptY$_DV+{lI_UKCag2OZ7CABj`@eD90`Rj#_%&+YV}7XZF+HD#~-3Ejk-Z@NAVt1^SNi5C{DJ>@Xs21)7Kn?~<x*N@FlTQT0J'
            'H1F48>%u5N3|3tfNde<Mj<<bn@nk8eXuBQ(>YftFL9xdPSMUVIFI+{n-9jsL7nin56c?ENh=!(t0XSqMN~03yMdQFyX;h?`qElp_'
            'k_z+2uH&SCX-|0R+`Q)>kVF4e9Ga7F*l{TBIfaJq4j;Pv`LKO8^k~y|N1`Oomu!)M%ky(;OE_0>)imNpHqlC$Qn^i}qI|QA)X`_Y'
            '5$i}rH${9?%i0aKnwN(ky8<~HFXTvZ@E%}cYO9E`JErE4+~$~^Q&P=pyHsSeQ+H{p&!)`NIS8k<Y7B{Kt*dLzu&7TXK|L^GYfo`|'
            'ZAe-Az2OQYJNJ6EsW0%@duQLa?E<83U2xJOLZiBvjvNvd_NvEbzjHMm!%mH>*R9U&4a{#wU;CJ7_SVLh59_d{<$-;+ad^`wG5T&Y'
            'Yc}N;rq|!yE+!+AAX4X}V9zecq+(`&Vpg1yfqVKzl0nD**~1}ePqp{miR6Zv4{-}Rm2XhfDwk+2R=Vu(Z<spJY*(|Gw$mv%#E#e9'
            'jZ;jhb$7V^89Qm0CFUXH+97?2eax;OAxG3MyXgmqoI{pjsL3<i#P>jqV|zx*HUR>w?L|?FnUg3-MHrkJ`BtM@QEfIMr3>v}OrNLv'
            '0A%R2;<z20SsserF^#<y<rp9k6-iXs2W*Em%N<jTRqN}mQ_%83TU3j+sbi^HOiX>d=v614J!O(G-feLDs<AM`fdePPN}y>TGKT`W'
            'x_^iT>(Ch+1C{HulPzPr<yvDkJMSBA3)hrtL3K3WVRe2??O*3|0SVO9B^@7*!+f|GH#F1q)QHYJc-V~)O&+c53v7A<|BUz+N8sKH'
            '8+`A1MRhIs&Nl*aWW*#^2#)x!z?i?QM>P`18;h6GRtm;TmBKO`X<_p*$A*?1UKTEQMv3b@)m&e_(NWq=Si`2(YS?0o8#c^q-p1m('
            'GTIDJ8<qNPy}HFxGpXHB4bWdKS5>rRFtoc0_2iL9PCr{c^VPMO@`|?MT602AR}m&G%AKmu)cGIsd-oO{9HEgUeB@lJpOpX3F6sK9'
            'q5bf23wT{IM7!;5>9=jfezi4gz|BH!zvX!KwRY{pwJ(;+3iMshGTt#Y_uy+WF+d%!;qdNZqk{_CjHzA#f9ikJbMpF#cepT{oT(jd'
            '%Xf_^ddtu+@yMsDpJa<L8Bv)cj^!QQp0peCJwCD5o=;HKI;wIMlNVO;KZ}cCxuYLBbI4L%1%9Dky?TvO42QXB7YWA-Y^ymOA8UIE'
            'ZAVD|wDn}MS~;=e^$%xiR<wBi!-m0U3j*N3+N_KX|AI{h+PLb`4>?JlP{1y8wNH^~{PiAn1=$;VkwLT(91NuSxN|UV<lH}PrE)*='
            '-Z@gCjr*YOYcGhq@yvHCn26E}<JdalN?ZLqk6k`>aQx<vEwu}pocOWFw+Z}YG!HR`)`8x#5-lAJ_`YU3>gf=Moe*y}Tw3nJ!>fnq'
            'ZyQ*=Z0ka*OXGcLG~3`uS58OKmgAE35Ks-1uw3mk2UdpH2VxzaB8qOtZa+c0%f!}xPn)?+M(t@zR4vh=jgKe{mAnFtS3ySvsF(j^'
            'c&%H`91ACXH=(P*IbA(1FzOW_pFF|G6mRN0Kb8uQ{b-ZXVt6lIzK2#P_<$G=yvD8)*l56tXUmw3pVD#yVDGKMoo>ywFD%*FDhTuC'
            '8Qnr}mzaz<1%J=ERow+Ui%pm$dn#%EZY4Uuv^r-~d%@d`_adgDyx>KmgpdcyW_E)Nm*KXA^|y)(=_@j{8;2~0P2J+JaGs6m2HGLU'
            '-(4rsNTNAM^@+}9;!MD)sh~JWxtb%FK%`}a^E+LvK{LjzkKk07Gtspofqbv4T<$C%+V^{_SQIG6y<FB^40ZhleZ4UiU>GV%@UToi'
            'Lz5+0*v1Gh1~rr_?RCMz2Q(=iLCH!I$4aIwzDy>wT~?J96*6JdEas6?;^8>n?$SBMd>3wAd{leheRwnb<>m9Ux3i~jp1%6~`OEVk'
            ')#TXQo2XceKx+fzE57fde8lxGammb>06B9EryMl$X@+iXjX;u&3ap4_tzEs=_WEH}%`FDBQ^2T>Xosa)Qz8ZiAt%>a9+zwH!sTz&'
            'm8RS)I-x)m+~Pnn)kY%aIl5aiXoJU+Y%MpPnZ5)ZReRDEz;G$&(K@`9@Kn5atq{TSM@?u2R!`*uu~^9!ufoL&qgk&mqxRt?6?6p^'
            'jPEDl7<?QraKDrb<8$FvI0&iiYoANMVL1TIQ6B#mB_@B*8k3e_*{F;q!VYeay6VxsqJ(w>Dsa@E2u%^<;yV2sXQSU>PK{j1`~_Lh'
            'Ur`FFd9nI~rk++w$t|4LS`o;dICX|oK1X`H0ZmJ-!%q}(+w!%XP?RoyK*ll$cv+bHXR0k7US-KNX|J|)Ky@Y0^1i3alI_L*Mm<d>'
            'Nyp0EiTUd5Rak}z=o(nKBVLSXTRw&@Hj-Q(DQQakbRBB`VP6qH(s_BnMu;npP-<n>SPlx4yfZmI!YG*d8dX?<rEB~WbA%=!?gFpj'
            'fOQ_VR2rtzCh8^KHNqE!m;%27%Bb#xwoIp~oY{1xOI`L*@pqQBPBTfEZ^SGuia47MIXz%{gPBi^ky~W0a5RNUc3nj%tW&{XjIkc?'
            't7%8J3a41h;fY$?9KBOVAO8OI-Ahr}Z)i<1aDADj%SsJyva|xJ50F-D<5H2c1gnz3f$-P5az$~I<p~$CJ@Pj7Q4ua9H1=RoC&0`<'
            'a1VY9+<l#5-ej@C$`R>~%5cX!m=hPZT!c42N<ju_eIv+iLT4=g&m*j31nAo#{?sNaQw~($A)xV5>u%?xmzaYdOkCn9*xX(n&JjBb'
            'X4zbCZ^f*G;Qtntw44GeP6;9B2WP0!5*<)eQkvb%niBGsRlfu0ri{qBO%PU$DUnhhABg%dt_k+FwZQh*0Bf91!SL|XUcI{ds^M@('
            '-UG6XwjvvYAOYxGfFe^oKAseY<>{LN5rhDvG$kO9XCiWGktzC!webrp+nf^?`zWhn$}#2#tus1Ej8qnvg}K^_3Nw*nqb&*`3sjV9'
            'UGpUpM$}u(F}g!Lai~rvn&%sRg>$v3de{?Uj~l!Ji9#wkbDJ`)Si(aw4Q4H)7=J|}Y@>&R+{7LQvi5*h_(U$gL1SB|!p6zK4I22#'
            'c45{!r_cr$FKM=V+)={DGSs(Du5o{St+QgZT(hH*H7307XV40Eq>lI9A>?#|*tN{H&B+B;`QI0~1CT)ZN`++}ukcm{R+>UA@U<O>'
            'bkVC4=C%VRhHg`!(+iC~p;Jy+Mz2-?Y3<x>1H`xjNtpIPA-5M4B5<&c+XCd>ByETrNdttlic-q!+sDjh3Pjf(>&;iaKGFN$LcDc5'
            'u8&H)Mg7uh(Mh(_`q(axLwAz?Q5cF_U1?9Dz^^Y5*l8x_T}}UQn_{Lm<ybX++N}Gqe`Z*48}tmk1TA;wKW#^gEv@RikK25mEhW49'
            '2bAOD10ARVm~vc%aA@s9lxsjL$800YDGB=js+Fkz6{}GF3l^Mum_&b%;#1b^t`bxaD#X)0KA3-jDpYviMSdNXBa$~p`l!XiH2aU<'
            'E_xyXs=taqMoYOwW@Oj`LJhvCycP4{=5h8=oh!Sw#q(R5ISx&`nZ@{9n*APCv3}<&B%QU6zq%sHL3NJ*#j2Ay*&r>^zFysl?ktXw'
            '-l4Tq7aWjUsSHS`XbgCWvGf|SpmL5zVY=NSOn0`qg3S^GnPP3V2-rjcl({T?L4r;4C<>DrkVh*}80gNdJX=I;MhO822_`j>n3i;B'
            'lA!i_6YV(ONa7-#G!EO_=mr;JGl@_oEbh&dz>yYkI_=Od4r&HvMVzXm&j98A$PE)l9o}RiIsJDeznF6fmxzKt?aI5jK_{Kjen(aZ'
            'M`Uy%iGspVo!}i;Cf9Kw`sk&YCBiEQmJZe7(OZ--I5atej$6|zf=%9u<;V8R4E1E2C1@KtFYb^}+bXX+nK1l$d-+&@nJr{qe1^f^'
            'Up5nBs6eTc7UKp|{y*-AZ+!'
        ),
    },
    'render_utils': {
        'sha256': 'b35f4b1004b927d6b69d715ed563a41f6c14381fd82ed2fdd02e10630b3e04c9',
        'encoding': 'zlib+b85',
        'data': (
            'c-rl~+jiSXmLU4BuZU8fX^@#9P05#Z(Z#H#wB<@qIV~?snVxE;h5|{DL<t1gxJafoTx<SdUgmS=Yvu>eFPwde2t)(`O1@OrsqV_|'
            '%A|mZjkxUCvG03-fB#)r=D{Qm#mkqgb+`~&n9Rbgw~p3f93`PxhVeSga*;2CES!nSMhu>exzSiP>PPEMG7+7dWi(yxMF~F3MUaT_'
            '-^%DZh{L4t#q%@{CcOwcuCaGgh-tdsh;%NBWt59JT}0C^bef6lFpK7!ahy(rxWC@)F<cR?)@fGc+<cYJN*MGk%ED<8Z^S&!;6*5?'
            'lbnReSLszKiZCy_{k^?6DfCPhFrzFK<vLD-SuUX4x}3yOzN8Uvs7YVEDnyz9*et|G40ImF)Mh4bvIv0gnU72QU?AX049l07>vbH#'
            'dZlHtE(@^^ie-Oqe}8{(o~0`>9?#37%);?l&I-ULX%Q4rn&f+X^4T)TmvJ;vfBrR36ZJa_)vtV67E!GKDktkKorZa?o)k-*15Ap)'
            'JPV4jh*qKcuKL>-`0tNt5^_5nFnn7zdkcT=?VX+-o&9n;{`KhfFUO~1ByRTv{Ief^01ScNNfE^RzSy6I*I}Hl@xgc<2h(tw#;`j0'
            'Dhscp@CNV)5ChO@-)9KH8b*DMe~bMn$;<gXnnu_ih8N(&c>t?Suc6_roZ>=_abg<)v4xTS^5fa7v)9Mt_dmTleS7?F{PW3+<JUAs'
            'uZWV3=L;`i!QSAHRXB^v75<8YY!UXZANF7~CaDL2o*%tEdv)?=eD>4v&&T7p?@nI6dTq?rn}zeBj0<llAJ9V>6Jo&_5I2W`!OH``'
            'dOJ^HB1y0c;koxB{A=)QDNchV_uMvl^by;9e`vf2V5-;Qc$&s(HV*O}Vf@Qa1Ni49{9}Tc1bIWdmoMFRtKh@<Ce3Ep<dO4y3Wpq@'
            'KO9&uvPG1Pui^ROq4@%EJj|x>S$Oj@i1W~R7l%aw6e3>-Q@}lFAUkgPKi&e?W1P;%fS!PJ`CaRI*tVWO2iY`?<1!bg&s!IC@ONL('
            '!>6CRpl4aRdp-Lm$RY!Tnr6|O4hF2>YK=JB(JiReiy&O3i8zDr2iYb}eDQ7>gvllg%xAwwd6;c-^U=?nJjkK|PK*ukr?kvNv*CN7'
            'wPBX#zIc%Z3m|3%?5;2V3}gPy%f)M;p765<=T(NoB2<9y2k{0zh9e2#fNlT*bnC9W#fT?0TsVE+dlxQB=-e-gc^xg5!Qta4*kkZ;'
            '@aR!6dGh%1@abgm`0=CZ?}NwFgYe12+5GVLvq$0Mr@{9JPal6D9v&P%9UKI~{QD<QXY<*?;K^h*AJjmXS&VR(MX}C@4<2N}O@9#;'
            '%W?wilue-p5Vig^T|HQ&=^_pvV5j`SDu|K?={$xXat8VI!98XZN7FC?BsCY|<mKyr@u6sH_7w-$@RbQ%fk2~yD#rmGF}y<XdvJI{'
            'fTN&EAieudQ9)I6_yp+o7$xEPM*KNk!*$E-$p^Tkrx9`@t2H1vT(A$aG>#|1^h%_3mipqTEYcNxb1MFHawamk(2<Eq(i;g67B45('
            'W0eA50T*x)6>-?Z%Zd0ZlR^ne*+ejb>#pBO3deB3KZfyV-!RX)dkZam^7QXd3%~!`wD6#RxQj0Eiu8U(5(YbR2Thninmv6ud-!N}'
            'Fn==r{?Wm~?;nPbg6aIx1c<`?_m3YQgpY$K-%p+d0nv@wU=Vyin|yy5JPHT%hmXECO<;6*z?1kdkAJ*bEQ|m0_{HCs7RW(;HW{!c'
            ')w(mEcK92|v?ccNPe;JaoxB9D?%m0o<5S7q{rUJW?@!*nK<=(n9YvN%m@9#cYDrfdJp*xrE;W2ys5=k;xRJM>{H7P-b;dC8^{l!I'
            '=})?a@mD5qUxE?AnrHJ6S=7Dm-rgQ?n!x7gX^bZ{;ENr`unIfKJ>wT={P$4gMFw;50-#g?igxcqF~)!CcV1WYegJW4S`3-xM#+zW'
            'gU~Vy;nS-qiE<=!@?)A-kvyPDC4CabCWRy@)F4DXD)=0H)Ey;u;SqNG@H$#|y8Sr43A0X@x<_*Z!tc~<h8h{H_aG~YC=t!f`q1Jn'
            'vH;i_<SEMVILp$^W{L2h`Tma)NUW<6c2iGYuI#P;=I^rYGqK;$c@<tyQNp9u0@)8b<sdSJvrhByx?3AcSN|n<d*-9q33m}Jr2af$'
            'h}r^PPv(h2;JT~%CTRvT4iNg;SR#Sk3lk-1BRqTY@gKP3rc2tr1X|h4)VT^bNY+WrR=;5b?l@PlXGY`t=nQTS|9OM#-(G!P0E`@*'
            'v$gg0s>v0;n0s;z!&?buTk(_j`n3U7LLY2GlJfG8CD1X!^yCQuF)}f)ZRZRC$*MNgY{j268p$y@ZAvBf=EMT<55LAGz$-lHE%<k?'
            'pkK)4-%4U5??Adi#`7|cJ4j}UZU7GPD$Sw=j2_lR>L&W!N21XU+#tX=g$eaU0yhD=1H9HUKtf(EG@{XTSteJA7ZDJjP8_TzvtTG&'
            '_K|z)JbC>1;ghZ}CV;N?Ch;)(<r>u&9fH&~w?MvMh9CG>M-f!GMo{fy(bF+rA07S)YXck2>j#uH0CXA8qO49le-5sYa%vh0AVhE-'
            'L^1Jvq#24ghBgaknTvIlByhq`e>&>HqV@L(*pGb43BJo<4T0x;m53Zz8(<>|I;qZ3<rUH`6n+544Xz|qk_55%6W!1R>m^=<q>&O)'
            'E@tUXf?6(ALnTvqloTlec#**++CwfEUP)aU%@w#^rhGHGn)C{lFQ^17XQ&Z_oefdf7T_3BwFeJ#*!d|b*We3qpG5O{n4yXf*QKx4'
            'jQ*=uY$VJTKi9Z>L9G-Q1I^@%{vs?o4sWM#4!jT4)1g66`8;wj$ukg<Va_bat*svAX$hP<vK9FTPUwp80(C+NOQQme2$f--3ODTc'
            '{nZux8x8=@KOYh63Ofc@>o~olKe@vU&@xI$;|Cxs@DKDvn81Hnhf^dVe9)@(IJfxL3->NH!1tq^maN0t)%r0v+UjdTj7ILa?^sy8'
            'XG}=WBGWab*UVaOIw28W3;15GILz~}@cD6;riEJcF^_uO?MosCOa!mA{15e1Y$Fbg7pwyGpjy9XPW3{qpb6#9B{Un~Zw*;*+eaPv'
            'q1D|t_pZ7G9bUL~UVKh=bIv|80Os}TG_R&>likKK%Rat^!`5JGqKRLLCf#Zrj1YL=D$p7+h&DEHJW@Rk(hOck@^anNui+HKd*IXE'
            'Bj)vQ)o8YTMJFq^pvX4%%;{#cztV?4`L44LBnnm+tW8hLV)W=SU4VeDYY+h(Uc(w<pR5Yw$)*VNj%v>rp|MDE!|E8Q;Z*4NyB+9f'
            'v&Zks1P9CvvKJ+=O1L_Ft;y>Ou9HPp!aW`2k(dOi5X_^Go;|c`Hd#oj|4t(^H?Hx$*V&X|jio*u+_BWe$$lo5-hX$(4^z}I9n*i1'
            'y2F?hY<O26);fcWxij~E`ODF}S3_R&ILd=07guo7aJkivpydMO3j^HN*?{U0DgI!r&|4oTgZ*t7Yi*2xZq>HZO_sZa!QT234S-jj'
            '?%o}U)*wusg{y#8paf3vW0b`KY(+4OMCaB%X0$^W+b*mIu0p>-G~0g}WLI%S-~y3{IiRapgj3**Ho-cI5pLg+#gwo+-P>zDX^q$A'
            'uCvxK^|@!v6r~Jx$gbc?2An=xNO7=g%Z-n!E&sZ=F#p&(iii0ejl$yzfXkv#?>lJ^@awtgR82tEn!_$lLj0rKH~#_wZX+fkv09p8'
            'OJSDc|NhVa%}c!s7GadiFS7MDUq)+E)XIl6jE&E|8MqGv7l^p<1>`_OA|ZP$4GU)ERQD*!3t-EwaRD++)76SThlA^j=VZan#i|5B'
            'r_4jej6et6ivTuT8l>mYG8d#Q2DTR%5aij#rTkhiy6jrqGF)Qi3JtM@>h>7e2CsD~@>;XkWKVWf%Z;)<U&u5ZV2<e(NL(OY*<F#s'
            'sl5z8gz{*bW2nQTcQ&fEY_*cDauBPS!vNMf!xMv&U0sNTSsqhc*fp}A^DYs)hLT~pnuIg76XA|UAoNriz%UnCuwEiOwVEBE;eD-p'
            '&qseiSU&&?K2H;%dIj%N9tULRhs}b|+yvQ7s7J`v@%VEB7>M^kbJH6TqDWR^m9sz%cp^H_hd7_-zx_7s>Mb+0EzAU}3())RH@r$7'
            'RMHCEFC9{Uh#$lOeH#BdKe#}K(Hj~{FA0=h4=%)a0)P!Y_%A&<9A22agU{qvOv)%GdS65!h>nd_>!1`#bijv`GAout6bpT^45P)e'
            'kbg|#G8^a7VinK>7=fvpZ~5nAny$vMgdpdOIovnc?VvvZOi8C@KK@WW9rPdH35`2#>`vlaAI38VVK|JHW^;cqyxzYP9PkHDw|NFP'
            '$n5T()N4#G2_JOl{Tf2Q3kje=`gRRCo8sFrSh3Iz==9+y=?l^qyagWI;b_K~X6f34Bg?8|T>1>|5X9F-h7{71dICjo1M&Ri=eMtq'
            '&yL{}0UR9iu0fIrmmF#c!yf4%U_1+a2U}POye@OnK>#zrc$lDH3L>9$QK5<EeL#FThNSFftyf<4zdVP<zPO}bm*>2K7ni(*(nlwU'
            'AFy8`;NFJCUOdO_6(es*=|h9f$dzYaCgjh7Iwd}Q@lCMF1$%mgGtuh_<hzJA;19VVmkJdweQ~KbSdPxzwo4ue+&|L*mHE>&nZm;1'
            '4GCj=!}5EKR}BzC+(4!;Nf^#@@dn+L;Of(kO4DV4^p7Y8&5}OFiYz?^*mJn;$|M37wl6O0y9#4k;dZ=y_3N=*dfr6RMdYb=C1P;6'
            'SkfU0w_-*cRNKOvw2XnK0U3{xRK-X;VO=jLi*cS!<wjl3<zM(*uGV~U*>~3rrW&ltRcl6C9lSV!N3gXyc<A9kz_r#n#IwE%K6Lod'
            '<FV~jGy(4<c+r*2a*#}ya7BEOeGgRMLDg514X4NE&K?`pgFeorPg{7H%k*hTqap*$>qIx--eS{X|IvK=;IRAc;ZVL<x-a%knVF{k'
            '15Ro%I3Tf9$VHJFAvo9Y-T6NL7~K-F-41V=&~AtD-Clar{q|NZ1$_n{FMH{-dc2{>3^$+SdYQYVZWK}t;^kq|l$yTAX1M43=eOzv'
            'Z7+C5YNLcBHx0*W7A=7Fia`!vlg0sg%Tbg}Q3D<N`wrY&W99oX8+$w-_!79kd{3>N+8SCp&)eSOptt9@`Xp~J_8nt@InxG0SxxvG'
            '@nZ^9Y=%y0XfFp;&O$N4R)FU`*Z0HT;cyo694`5EP0>sT?0P~LeFGe=i(e7BWst3a(a4D=1p>J>=mrE09m)!U+GOs7jcrhqh<so~'
            'I9niRvjJMyg$o<bH&7Em<QpXMAot9eMFRNwzm+(ApbSG17?AW=rO-$7qQlCcKKT9N0Eu3TwEN+sgNK6HU&U03gTd2*0gmW-UvXNq'
            '6gnY63Q=GYWD)}+1hg1Gs1(0A<`{`}kQk6sa#Q#ilD#P8-4Ov^P)WETJrruM^RO^hlvXUwfJ|RSNYUoz$btxVJJbXm+bS1NB6j<E'
            'Ih`W8-tT`B(!+IGj1JG4KHkFl+FqQ`jX$2bsBJa<nrEhwNcaoH@)2=jquYur+rlu~1~~t+0jdMB>r8LqBkk*>myv%+3>kIa_Zq0('
            'PrVjeXKvUIicque&i5ykF<0UU^INz0pWkI$(V2u-fWBce6b89uuGgk0-Tkh`PzU7<MJj`vhWp-FNPIm6GLy!>0M-Q%8N{swxv`mi'
            'yqnv^KD-Z-@AU;a-dv|~xl(NP9WX#bCJ2es3wh@Ws%stQIoQ89K)dU~-CgTkcL0y-%=6FD2cQiwI?+*!Ner2w6-5#V!Z><|lX_4O'
            'h<R&d*d%AoeBs$LBv=vZ0?7_FB?}mktD7ZY#btfdfctq(JeXp7frv7*1RQU0fxOcB0sfGzFHQrW0-xsaxVb?3k6damLML~WzxL!>'
            'z!0CoJW&fouJt#|6p+`LPA*xxHy{h=dBw2XJeO3TG@zbK`|$8M+qb=7v>4sGh|+ZRq9Mt_<Gbmr#<J>L1{Ndjf?o}!<5PwTPnv<q'
            '7l9(}32J3#;RXI!MIRa|7Ix4L@n6K^qfcN4SNsS)=yP;o9|5oVc*u0}wmPfZVNeu6Ge+<(jFEVb9u5ZQ*Z{<%`Um9;HKG9iHKKjb'
            'N5GJjJkwE}CTO`H&VV-IzL1A+M6V73`c9(g?#yTH$`|j5wYIb=UumY=JE}~iQp-?6O5FzVVfhf%=*2nweL>_81j+^NsO@N?bS-It'
            '%5piXaUM>U^5K85+HyqyVdAC3Ec(v@P2#W!XK(pIr4cBBi<Hb1(XHM_^p=2KaKR)ATuZ$e?e^CoJ2H#)9SyoK-r=2=Tl(V4y8>As'
            '^j;`27$+~_ddq;@M0o`y+F2L*v?NV>AH$4DlcI_5k?{R^a`uzBRI6|)Isu5glRV7eie~8jDO`I+r^{x}%leXBA|p)@7r@1l8jy63'
            'M5e@LuL7>rA34R)u(%9o3Z%&%z)69<?d5529%Rr&V(psTrbMtn2An+2#8rZDrFg3b(gzX`+kyfrP}`(LTDpRB2Wvvg5UKqw$i)|k'
            '^9q(`sJ4Q1Ccwa_H*l!{>nUNg>`swAMJUC48~`nWLBLUYw_vuO$2JL($CdjM$7;P{HK7As`Y1NwOMS6P#}1$<92UgtUzj)S;WB`u'
            'Fyng#bqi`HAutbD1RuH=c)cxy>o8|e5cD(2<v40Lh1wAO3SluI8ZJ$JYw=CP(?5*Fpd$Ly6*`EWn~GqsN6|GVYSZ5eyYUY~aQN{p'
            'V)(^gW%){{s57lj6CkpEp}+@yh45qE0_jtM>dukv8B8WEL^4?7kyjS;-c$G}eLNp|QXy+wzVX9-CYXqvXvY~0?=oD?J!OzkW>C?='
            '!zOMWaAcpNXM9x8dvG%hh8L#QPEu=ZC0$LvXpPOWd-^<L1B})i9#Fp|#bc1$J3^a}8VgdU)KP(lyhdBBy=iaBJiBLY8O#@LvSzDS'
            'Z;yRRvTDyQjkj`|xpOChqb=tsXpRoL(i>lrMD%__Ii46l#YV|kxS=G+4*89UqOg-<ypiC-)w<aDB%TK(h9aj2ND3&KkOmJ%nU;$s'
            '*%VL`#iNc3O$1~HUyD;T4@nKcH*#HwCIZogOD(<WO9vt*>SGv<ad9CgT<l4Yp}82-tW5K3&5UjW>azx~Lz^aXP#UeJ8)&O3c!;A-'
            '(titv;>Xv=gTaCI5#&$hsA!op^5>F~hW#`8)nR4Gh{p^P5&1ovE>PSR#ua3Rv`JahLFEc@%26Wp6kz$B2`@unsBi@EHDJZEj>3BM'
            'K*ZBKj^<<7Fbu;8XMi<ykGnQ*38rq?mM<K8_EiXYG4{nZ%ATl}!#G&w*t~`b5eL)pMqcep&{cUbcRb>FfU=BBI;8r~Rqizy+0k(}'
            'O3LtVPl{CpQf;n}GYzxRQK1r^+do%5`9|HNtiJZ%JUH^qJ8&MM3Wrol7PN0zMK7mD+GQRg^8&aQO7RrtYrNwS8tS^A(Ix5tgC)!X'
            ')3a<t?Q%4|lu1-bb1>=nWvmSR#9!oY*tfk%nX5x}GEqBUE!qQdFa|n}|LYIzMN_CHIVDmPbY`TqKj`{`|CZ430S;>9<%zdbR9*qZ'
            'U=fbyo{Yl2MUcb(V7`5DOVH@I$P;mu&SE%awF`u9uaP_^H$kKhl-%VXf!D9jc69?&bN2p6SK4knT`?dlkDG@z@?p^RGj4SJVU3Z#'
            'O|Y(jU^=I&Hhg^pOSjMjiL*O?rV9OO4CrL^Q}pGpDekZw^JBH6@}?MB1QV^SNDGWdN<m#kDYP&PpTsCQJVG;j(1>gXa(yDU)v>HX'
            't=(uQd6XSuPls;(YF6lCsBU{DuPK)S77g-_;X)v#Htj3m1gfy*6Fn?Kv{rFHfjsn$f+qi(<NaM+ivN0g@`mmYj=U!WYF)0v{d7Dr'
            '=0Tiqgv=&D95=f^|KI=lzqN`7HQ_9h1?cvz0|e0_M7e|T#`jRho@w6@7$REeP={@Klt_zgUZALy&gnzsO^pUemE)lAD1gl6+l7S6'
            'mjk$%xMX*#euZGW81QrB6Ua7Ro4mYdEr0cZcYyv*=lCDYTR^xVJtHnE<h+BizYhzzXPJj~v-2${RIZLXqAtsRXq`!U6mU{~N+Hov'
            'jbRcB(|Ogdqd+;`D9orUDw<LLN03{4;e3u>K!hVg4sz4rXC&C&<x(CupYUF~L*<qI7)3^P$+E)?4A5~DVoMZyPKEHD0FCZydC2!l'
            'ncnN)X4Pz}#AIp1M`D*(8VNi^g_PZuyBq<eoE*q9qnLVfCJ?De7mzFCPhXs%23Je`@ViWOkS&E2bP`tDdM~8ZGW3{zM8Jvm$dK7d'
            'aEx)6aU!qzSm178&XY9^B918519-a?n%>ol5Q(w9O>aGQ9ERfj)<YDWj*HEj-Q3YFghTyi<f|7Y#s8482W*-#o^5^6@AuVF@#LWb'
            ')bx}{#@>9~cq<j(5y|;vm|SP*=@_!>8H`E0y~(|&K|J8^D$0uZkcEqo?aJf|xhz*fQUx=!+Y;gke5cCOq0_G{gS<!ce1N}5EB#<k'
            'hgDJh(gQUCW~`<10}vZn)Cl8wWOf-~qnp&;#sRv6=<&s?Y2LGnJo7rt0`>z9R?#NS?oSd;vD_8NqZUH<jY_XW&b~P@)+{%zKxV-#'
            'CwON_0Zsh;=U-)7nK?N@lc|-sT?3%<b`$LLCdkP$ffEw1m%$_~@@FXeDmS*vOHEcUFGY%8i8qR?a^l!IDqI}BIejIFh^~P`hz@z?'
            'eF*T{7t{5@;p0cFCBy^H;;aFTWkHgXXHm!wnymPuQ;UjqOc#SFg|-qOa5lK68<M#)lDiFf_@PyRh6n((1Z*yr8|h~iqiT$g9U^{T'
            't+ZZPz|>#PUiO|oGvLYV1J#9u$-&&VF4KX?*l5IlKu4#~U%i4cWFQLW{lC!-rg>810%Ad33foE@Ls5qmAKK5uAe$~bSw;1J%fIdX'
            'Hv6tStp45k<M6jW{`JSM!ImLNTEs?pr%W!tfo?<i-}!(4ExGvikH2F3A^+ikV)N+zK^>9Nss^gEJXw=f3QUTrGX`xwH%B0!-;Y0c'
            'Nkf`%f=wjSt8zoc!(h=59xHZ0ZW*(mq^FT1H-=~0%ksvx>HP5zzx!?0g<ZNm^dE1#AM~%zAMpjOi~+BYmg#k8U<4hov&RUpj*k9_'
            'Y<JM@vdp5sO~MSWi|#J3fad;(clVbIpj1tF-g?rNrn>T^TY~1{j7=Qg_bb%L+jgB}!d7QJ+!|6<@PRSUa#R2$Wf8&;rI#2vJ$PMm'
            'Q_8eIhSVQdk^pwax8mWG!9aXRt1F@7(<k_hRWtHaT{(E|)?cv_@|WMn2be?HZY>^^1&kWkxkL0SN}%xP8~}?NNX?VJ#U5~zV9QIm'
            'E|3^2o`MYPI{wwKE*RTMo^RN?qkLa;Ua)rabj)9;0M%sHMd;}L!+$D!k#<oh0_pJCUZWX}={aBlS;CZ3R?Q(OFodN*?|ySKU2O}}'
            'L;_h-;$;Ism^^oU`X#Fj7cr703L{9+nj;Ai&c#P<Gra+(a<K?A3%)7nH41q(-Hi6nZxyiR2K&{rwQGKBnqKz7BJFmjwB0mdQ^wV+'
            'LrU#E+q7|~G-Qs?-bsnhE_ZlOl=iEz9Mj!hKQ#T`n>BQBXdA7<(j^@;+cdw%gWn%Cs545EF!B&%xPkRZqGF?XVP*Rar0){0D@Lff'
            'k_ky$><j3TugJ`ken)}u7?#H^;S=a^2xPcKuNr)XBa)2)v%MKGN>Q9MLcsZ(#w9r@vTL3aj`{4}1kA4w?>K&4?Tdpp6~cb)GsuQs'
            '6d$I@Uo-7sAr}un@SLb4L0%tkVwsvm%87XBFkThCJN3C$$J4pW>68Md5+};KX(UaRVU>@`gI3W*N~x4|1IA4$?s&=6l0yS}YD+4-'
            '{+vVrGY4*^`G%IcdhAJj<)xKz=vzr5=|-oRXF3;q2H<m#V$nDf&qE859im)s3i3BG)D4YkKJ*AJf*s7wLaCy}4`Q%q%3Y1$HVnh`'
            'T4iLowbQ$ys}l1tyyQAxAiv@Z-+5(2Fw$R4(+u@t1`Ol)C3!T7L*pH<v65fw(6`-sRJ+{W3Wz*VLpe*oIx|DHtukocqRfcR`W9oR'
            'zTrl;D3>II&a$4n)4+X1`J!S%cogRjPGLLwd+%QHo;$uri&1M>+uGett*vfv+OXa2jqBTAZ|kfN>o6}Qy9O)pb+Mq52@nN1ytvYn'
            'o?n=fmmxo}H#tf2>M0z)ockO6E+rO_jP->fd3`o0=AJM1?%Y_6T(gGVB|35EE@MAyj9rVqSRgB8yzvE6LF?`L;DRT?;zZYPc`c=j'
            'ZXE+E+@hh)z1y4Zhufu*6zC2t?A%z19a#KAGP<;$^9Z^PNek|U<(4303NiClVIh{*x8GJG6tv-&T1eRCF<^W}2GXpG>R5RfH7j4|'
            '3~+}yYTqy)cUz(AvFlARIVPI4da!r*$<}})4p7k2zssqi?~-l|zg<>M<{6=L4sZj^;{j|@=ekR$=NeCsyw}Klo~uy*I*s5T_a+j1'
            '--zDV`v;(%DrMjzOhP&1&OK6gE8b}pU<SLrZ?v<S2a?k-Z^rMA-@G_}C)RMPL*VJY?y%V?Na#DxGj<LDE@ZV<BC+J9ZGZ7fGIzqi'
            'DmmH^=(7-W7S6~$1i0W;I#ca@V$iv<DfZ`8OWoNAd0n-P<0x0!60*N`s8f%g;Lk3~_XmT)z{VCJG?iXpI5?bbhqtCuz<5MZ=x_g)'
            'KR6sb#I$8*Fv1iNa*R59KxGQ_%sc9b^dCY-^MWN5%Dt(Ac^@6pidJp3Xi@<<NBu*P0bTKsk?=j#X?oHsAv$xymmY`+`I@3|HXtbX'
            '#VLAS%OK8VBP2LM9=<XPUXO;o_g7_HL~GJwuR?TJL)ieZ7iATgpcaMkk4nczmm5X&3q52vXc}Ie^QbSBUykYcM^2EKA#rb~e~lHc'
            'EolfZrri(6s&!Mmazh>h|8aqqQ{wXJAu(r|MiR$2q=9U^yGKH4Rb?=EpWNGx2?5ZtL2&V8RNn#Iv_}GhtI1-P$(7Il8OP+?caClB'
            '24m{1mQ_O9vwf#|TlwpH(a<qr&zMwRt}xS+Y&x)fy;P<^nyTyM1%ps4jPJYk*EIU?nA@Nx;yZERE~@$coo&e_OhNsjb?4bq7NE_@'
            'f&Rc;mIkjfgw2A1xu!4<Ycszk==JW62xn}S#y9<;3f(;6u~Q6Z&T}xYzULZ5bGu#0qgL@g$E9iXO{b-K_>#;bXh|nIWe?me9eijL'
            'MXm0vYy@l<wrET=``{x)M9MIMz0D}!U`ba6j&pQia6JS-O^fuvvk!)z-g-UXziqtv0JcVj?iQ}wAH<G^v@qT6e#=Lw4U@o@7Rm#('
            '(lp3KN7$?5{g%j;rcT|uIZHV_sQmMjkWzo>OdOpmP=xt^*2yUyjgzC+xU0r%jLR#->dv|=Q-i#V#u(-olMX`zR+S&(AX${bLcXxV'
            'I_`oVB~gLNlsL7%Ipk$pFukf$?dtpVOfO3=npT4Kc+O=?^r)wITBbyZ77508U}6Eh{wwn@oh5OS(2c7iWjj(YEmXCY>4s;Bf<}o6'
            '&~Xf~pT)G_brRXo-9jO4);7_7txV+lmhKrUGdPU72+)Hvm)@Il#$L|4k6Fcz1N%p@4V7XXj+As7$&@WX9yB@AF^4%DDOW6CVf0qu'
            'l^m;;lxcL{^PfK>VpO@|2v1<Y%(tC75@9;N>u4H|yy?26?8>Wki3bEkppge_^3=1gVPZzcnA6l3W721i_*4DrjMyO#NGY0-1CASI'
            'nAMn|vPoVCGZ}>15dVO@Ex8_#sSpH=JuP$Ez>!pBs8cXfKQJ&fSkY`pPG%Mx?MEH^Z&dw~FSYNu=f36Cgkli`E}+{G5EzoOf?Qmx'
            '19NF*N19`em|2Bn6)@+U%1j@ScQCmaq|<4cQ7k<filT&__b7tYxwO8~=ao_dWMMB#{z?M7SW^@fx=TEBzsZuO?LDmEj5-=(eL3o='
            'fftOyA@-f?u5Vq`d7n5#Lz!bD*oDi_)_LJp?--lpOwyc5_!X8ALC0t~sI#&jrE;g}(Lln<sAGMvN8}9Kd-qdIx+(JP(J!Nqft#05'
            'bSN)e_nA$tv~l37i~;?lIG6@kePf<2dRemBx1>aCSFBX0npMH$Pt*lBQiYZ;)RZ{($LqQxS3yJ>S}cF}-G;%&Pa}Yu#L}_v5t)Bn'
            'iZJ-+x!2+1HR!aHL}HU(bq(o4@`Y%X7>Oqq__4fo;84-m&ZA)j9CX#=g84W+I~fMobq2oX`H`WiT6T@NG(=oVPd#dy<^wI^7(;Lb'
            '9VxC_D^?x*2-tf45q<bOIP`<A^R9*IwWEr0j((L=yW&;N5$Y}%?y=^J)@wW}sCD>eD2`Y<d#|omEO~|zmsrY#-2s{SlBX)RNYXM9'
            'WnNMu>;+~MxyjOGfv`IJs_xJ}vf%sg!X$#<s6Z&Qc`)U~A%csAHD{cjz_97M=mS86c^U%U3S+okao*5k87-E5JzlQc5*p@55Q2My'
            '{!@yWmD#aim<7V3fyyjQs5+Py^nDoyF4HWb$6BZ(FexuLd3LeWHjf>Fv8@alt0{TVl*v2}7K?D!r%VO-2m2+=vfw6{xlOQhm;v7u'
            'ZbF%JWr1<42r!X#?BFp5y8r{#BZ!oQAjNM5aPeZe6Y|95aW`vdgP@R0TvJpOU#5ChrGcMx>;1jgC-2A4Pu{$I_2T%=^W!ll_s5j~'
            '$EQD?ynaE3@b4cJlCb^<ePvUia|{O#3Ppv9HRky$=?Ebkg<cStz9x3wb!puZnpBDtb4gCD<S?f)#mtC#M%)WIkyli3F2|t1$3uL3'
            '^zQ7Gy{(XILwjfD{rApq@4xMK`rrPsOTYdf50!-dU*5cWe)8fNgaY{E#mU*x>(@%Q8Oslk`7FV}J2XA%Wv~KK3O>(v-O7VT%K%5~'
            'HTf+YvyoXV2nvR!hZuvcreG|E<CWZz9prdsa9(~6WZrd5OcvzG&5*8~bL=G5TH&ytNnh~Lb$=ueT)?9Ve0Aj;9F}pEJ^a8rDgsPR'
            'E7kAHou^RsQOr4_H!wO63*?k%wuM~z!5RD1r)MNK(=@wQa`&1q9cMh`^{7mJh-wyYkv6L4wq;I#q&AT5L+skeg+z9=(=|$ySPN!G'
            'Qe&69M^-6;ag5tr1K{?ShPnkrB%3Z{U4UpauIjpttF_HLQs6{|2I(iN-&LaqRD2nIz$3p5G4&<i&hKBH{dDrn84S3T3H+B~l%X~j'
            '{>V))4wa(`NMNXHV;G+!cA6wt4yv_y`z#E5m9n*RGVT#>BBhSW2{DrSHWLbTKo$?*)2;ac`6d#DL^>hg=~?BaP8p46LbH|VFN1mv'
            'IU!LkBeIrj3_Fxm96g$D;NPj-rvM~Sj@`xdZ7SOWBDBu)YU-#LWk_;1eh@d5{AOUxO4eq-Y^GaRVYtp^&Oj~GnIZhNE+pt@Rq{if'
            '!a-93x#(zm#4XgD9x`L@6Y4~v_M5wue_yJrPjAAKN}qvXK(dc<c;RZX2^Pj|r~)zS{&Dj*kS>G44Vo|QV2YfiOAOXQlsS4gXby{n'
            '@v$a`A*Xm@=-YIzYa|*S?gMQ!Z;Pnmbv3`~*Z^dzw$|QQsi>_l+S2*sWI!WGuG<_;N@>|T+}M{bZPG?9YXk(3b4N>+Fm0b75o4*I'
            'sK{_h!O*FQzhscGp_(Ykkk&W|HSz0PiLNEfr%gZxR{yclRjZW@oV0gSsr9Y-7;Z9{{CC79Yi!NFbKl{GQNN3^yWBD4l&Q9XzLmRn'
            '{V0axntZab^(w;8xWKzz?Hp|^kpmN#@*F{5kYND-)EA5|1`Xy0n1)i;qLl}y-q1iVblnHY*#m`q*stsEQr(;;wcI9xHE+?>j3cLE'
            '=F084VJEsZQ<XDOE!Q-&S=CNrXfA~rr6l6;m%9zKH;oI%<Rukk{R@$Xp~h!C?CPMP5g+#p4HDJ9)s6&i;&!i=wYc?+S*e=uOTrZg'
            'YOn#apb;h*07<DZbj^5sP#t9u-cVRj#hQRO3yd4kS86JGue@4G`W*)!BU!UZs*!sI-1X407;1-9>CX&w3*&k;rzrKY`j%bN`g1Z$'
            'iaShFH8_m<QMKcm0=%M}^0b5aq9^GyZ(Spj+OMRp3GVB{+Hy(of*q?rTtNCoUyf{y^{vp9(J@NvdYD#9F8kiSa%PAgI@(_0%}-@-'
            'S>26cqrC|*VajpboswzhU!_z>+rR)dhITISTVG-!izs)cxYn7jl{#2{6FzyR|H#9b?_pWq;J#2>%rd^CVjZ(r@ja3x3g^TlYrq-3'
            'V2d;P3nl2hdUH>R;Vh#14Xn!?N3&1suC(2%yxmCW3$y&c&$NWkA(hpsfs=aj=2)&c=G0O-qe%TH&+7|Q$$_!56qP<hRRuvo1%Whf'
            '++-;w(m>W7y#}zXy~@z7q8}6=6H_&Hkp~+x#E`j3`}tF9bIkjS=9vgnQCAtVNOwV!-kw@RY{I-Ai4F7U0|)LABf#?KG+iWEXIjf6'
            'jtliY2_(90?Glu1U8FA+U8a;I^YTdbR$$7k6ml6+$V-+^0!%DgY}ifv5?SPetqo`!FlHnH0uIbE10*UefcC+;$0$m=zr-vpCj}NK'
            '2s4rx-mo@64Eh6hZO4kC3(oQ_!6^V(%tb~+W>+!~xLzBcL!939s;`q83#;n&cpDTvR`{#}SyRryLJFt^Zs_#5FY9%XHa;%VK10M6'
            '$Jp0(F?eAjW++(qe8X&jv24HzFfUkR1e*^RTn5HP&;-xNScO75e|1~HaO8J!H!LJPQUeU@H#T<9)Z*SMZS0M~=TI3H6j8~UQNLqd'
            'uW1^yzNy<;1KHaCFzt)=`Rns(OJa4~wYh%~JGF#+mEKwUXKBWd`0N_rSS^4<3v6ku(UYs`3)AK=OzkfqeXu|9jX1LimP~cX@xFeQ'
            'i}3{J%xlXg?NFu?)CFZEG>WI95MybZ)jS12zJwd>Qb$>V;LX~CER;6?l<M3hrQ~3~VQP!Y(Mc$am>UTcQHoEnz#|p{VQLPKi27q3'
            'Pk?GhRL&dpHw$i%q~@7}VDLmTVH^Q!)Hqe;zRy{JYdhvfFuaGWtb0t2jWDGTvx92`xeH~iFW1F)ou4?nKaQ9~(-6(V+LC*MVQNOo'
            'Sfm;kgan}OY*6)Z3Nx9jr;-y<ASlM$pf;(Wq&Jl0=p(ZBWijpcqdaAUpEXcKWz2G4k`!qL#4z=NQ57kb%p;A1W~EFLR4;X#PnY2;'
            'FbdS0CD|2<>v-K7SL>O%65h}e!n`^?X3UW_G-VjWs4@vTz)R*J<1Bt`$JjG{2&f8+v}aU<E`p+@a`w!SjWOK<rmHfmkaMG|>~6;c'
            'zU;4@%M(laCZ_2eHl~218lek89-s+i8e-ixv`tgmam+R0?O-V3+-c+`lnc!`mij;5q?8OJuzFW%QsygIDaxUS)FDO_^D|)RiVcW&'
            'a(FPFkLJzf0SO(KYpxXwGqk-*cA(;Jq&LI}XkZEzwKlGFQMGTEJ3ix3HOL$u3mu(?SLO;yr3C8*(DduxN;Y|GX-e0N(Gr7YV6@?M'
            'sfOVy((uep+ioiIt~5E33lq3x8CjoVvmVlOJug=si-YTm2Q*^)p)VeF?<WFr1}1|e)ae2&QfV<*V|9VpkI#pX2N&&~%=NWjSvyfN'
            '`p{O&41WFYa>b3>Rpp2uwVMh5V=jG;#rEJbwdk&-qnPr5No>Pg+#LD#ymxSMUcI2ibc#09x(>{#)@4^ZuvCaX2MnpdSC^SddEeKW'
            'gFsaPvIRG39ye$*vmpjBjGg5itT9|t*-%ClV?ZyE=!l}mF=Zo{gThxgC{Us0Wz8-C1~8d-SxTK5Q~OyoLlM}Z=&lS7i{L&YEx#RJ'
            'ibK*UHSqaCuIR)0g?=K(#t*9?Qu9co{HXOwl&L}8VMZ*WdNiJFsshydAzkcuY95ek6LPxUl@3djQcfsk6?=Cac4r$92cxMfjGlMy'
            '<E?uyn(zf4z4!I&-TM^n)@S(seu)P#TkWLgLpyScbzI{4n{lQHqUA{A<u+)WBkNqzzF3vEU>P6lnUSe9Z#JcpP4dM00Q)-@6>Ahw'
            'IsqAAk5w#;42Bex&ngjRRArwZofpGa*!&0b#r>xT?tgp_+>4r%qVY&Zmod&eKDSOmbor_t%c4RO={-C7cD19Q{Ss5$+(Cl0Kd|i5'
            '5HDsKT2;GdgR2`Tr|g{6y~X0)5IHC07kPH3TMRu7Wj04E2iKjstoaa<OEf#mN}ZGbPS-ad;o3Eg5&QmO#E_c5E+emK=-;8{^=Z8r'
            'xoJkdZG@utTQbx<@`u${^K)|Qe%P|h77ytv3Aw*9W8*b0H9G7fe_jW@?fyHuhf90%XSs*F!)%EDlj;U!#H6dL<4`W)8&0cES4;oz'
            '_9nNbFZ4;rn|2jliK{S%b-cqTeO7)Hr|-_ho$lx|0!BHyw1!z(tJTW8HLSjkH}xxbv`d~g`6}ZsbFM7=&x&0dsUJ<my0+uI2Ff4~'
            'ciBC|#bpd8mf~_E$9*G)+m0aT`0ehUk>qvQ98_KlD{ts)#g?lM82IfAXR!4S2daJLRY-%}4sYcrWiUCyqo4g-6DU3M%(dj&*B#B^'
            '+iR%Of1__E#`L<it^eQu^S?af;VnMvm`~8R(d(K|e#4Z|p3nh=<WZx<50=7B`?eX&E+0!NqW$Kq#l1fq>`K>l*nip%Ym$Y*)p!xC'
            '-3(w)9#^sS)j+F=`aj{kv^|yYV^T)u%QT}5dz>(Dy@Y$R%CoBkfN6kNkYcjgi4{*oma*?Do{t7kD~E)lnuxo&r9yyHrKD=E#pK-b'
            '(xLDIbOAyQ6a&IIWeT}ajT}Q>EgXg0a=@zS$%hiy`QV}odbPm7iaQWgvl<N2mN0y$VuR$s;a!fIL<(uvvpUK|0hy}cfsN6jvD5MT'
            'h^fEBfBg~vHN<~C^BjvKXLcVGo><j}HJwC7Ab+td!^t}bOoB90cN<L<)YIB2%$|e&pH}?|(VP40t@uG^%y_I=DEBZl_s2c>L1o)G'
            'P<^cv(cH0l<*k;r+9>8a))_ZG09AN1FK0t=e04tdoQL%s0!^o5$A=^_<Q7(>ISb;fZv~>-M8Eh_##(f7uo+`o)enj~VKv@eVu0?k'
            '(LH{Dh{;cYjUJ;QrqvHG!oLQ;mf{q6g`LU~>d6Wf2KfCU63A>3CF5)L?C@~ss2T`zBjMcvLN41hje(_sb<^|sm!I$|eF^{UoCcLK'
            'Z0`N?<(<7t*y0S``d9(4@-OY`5{Ct9Hu5$3dz-gGL$=5Fh#KPP^b~EmD1EYoOkOi2BQtKnU-_DXW|)TOViU$x9Et!P2bg;o0iYLa'
            '7+^swkhSO%u`UZzka8WlTo#F^6!^G4Afd~$%F5{+VUSe8F8;xFih5z}wy(|8d}jgK-BjA8aJDbP`5dHnE{!*@bXiwH7Qj425iD{V'
            '%z~9v;(4%sUKTw{roT!l+(l*@rx94%BtBeN(W#9sFj#?+{$-%vs3MA{p8x}Pb3z;SgS{7$kH;O0V1E@u|1m(npjoh5AWy$ZuT%o)'
            'C71Ft-9)_eQ&x$;O6KW>X-nkim_BPLt{?RWgYS&DbMGyj@pobV--d$n098byyKQRs7q{KUY6oA#g5W>G(uCPjGCN%cv-D;Nm}Sxh'
            '?|l}mPcebdP#l`i&EMz0KKf2jquiR$OR_Hy#T%eMzIX`)CDFfFWA`?TfTV1m6aM+)lsHi1!AV&lf~r3&!Oi;-#1{4V$20~B`V~eW'
            'O?~kTvccF3;K6-;6`r6g2F>=r$^ZTZ{>$gn7e|<P-UdxeLi5ohfDW`i&|XP`zWAA_;cNBdo%->sFOFp@3~OWLIvx>XzlX2*x0*h0'
            'ecCKdb-VDurd{yBC5BtyGr%)N#DDyM`RV(o@@MbBAJG4STtNT3<ty5zQ{CZWi*5YJ@V`R~EL=SB|D26Vl7AT7K`SyG6D4azgxA3a'
            'C6rTEgdC&ZLoPF;$gTw?g5rxaa=$Pfh9u#%+|`*dpgi=!=RfSYS3c-nyOx}6Zrsxiog77(%7Md6YIJSkSfe?A1|9sxABcHt0?WxS'
            'J``h`+}QpKbvV~x+L&V?OmlM~+)w78{LTE_0e8{Pn%g-Icfhx@#VcCuHZyYh)qyIH5I|^A+@!X21Xmk*gH*A;a$^Daui+|hfVP6W'
            'jO)^`MNSu5ym=;&-{%;(zoL)3QPd7c%UIX|4CM0kT3)vE02v;2b@sUU%ye3=%9vU3JfyH#0dy^x^!J*Tl30+g0o|nR#L7y)zJxt|'
            'aqyYy&_-o!^A?+W+4ccC`a4c$J#Nspo3g7q(Dyr<IthTI$ksWJ`o!FIsDy#$^VoJ?Yg^Ys2XjdyMwX65f3b0;t65+&Q2Nol;P#Rz'
            'Nu!nPoO`#ozx_6?-w*%t+i#PD?e<m=ckA5=#JP1Oh~|FvSvF%u`<(R19Qr}1>na{>5Dh)rmukNMA`0SkfhK0S&0HNJ84tF#=V(FL'
            '`uh#gj7wbJul{W{=<R-kAbVFDEy*i>C#QmEg*9RIyjsm(c9I`9oL?lRM}q^6&3q@nVqDYu6excq{Qx1@41+dQIedieNWe}8oK0LO'
            'RnwY74D!ZMnmC&DxuvuzQLrf04Q7$p%O+lz%6-{awoYk*#<xx!tR}Mn#>GW!pFDp2@CkY<dv2}NrOHIk*)q)4T=tjY2N^}xJs%!C'
            'xlrZgWr7!(DU}t@Xj+spIQbaM32~f1W(tkjA&6h<$J9$^PR6N?@uJD#&nEAHw-F}&XWthpVTOvqFuY<Fp(?6{$CYfUL#k*lGtvtM'
            '*w-nB(c{SYw9gNNB#1Y8l#daF`GOL^>R6MTWKXNh!W(L5l;xOWgmfZ%8;OfoRO=DPtP=OpZ>3f8W!hhM5yI>ax-iE}nD3jkc!~OV'
            'U5qgy3!+RWpmbh2yL$?m9e5VyR~&wUMP&J^#&G303^Hc!(u9<#A!jINhwMOxA)7@SnVSqISu~|;bCsQ6t6b%NVlgR1Wd|9gr$75g'
            'b~$`l`B57ifp=2{hdbFV<bkhmRa4h`tEw203`3x#k`{uRLmgM)3Nnx_ncbt8Y^BD^I_>eD->fxs0Nn9$r<v?brJ;jK7^@1*Kq4vu'
            'ykVk#wKy0W;m*ldYhX~zrlI@wc@sEyl)gJX3HM^?nmY|Idzy(8x6x#*HS`tpeUPZ5I6_rRAiP^zldXr5T2UUvaP$^YJQTMC`ghs3'
            'Z{lztt60K4@UgrS$dCe@tiNw5Ogd_JelXT=hyG-~Z!av`Jo(qQD{7m9&eNS+{r4723vOUAG-`JwF24NGuoXK;<^t&4I#1yH^&RIi'
            ')}GhLx@TnuXu1MOA4WVvdz)@mo5Rc)-%foi3ef<KDxlv;i#Cdj4q?&*vY&s>Al$ph`EUbi(;u(D`<wwZ-E%grG=|eiYRtRO>l(Xd'
            'H$Jx;UVPS;?FLDHAoRd%C?LSNUAsVSme_k!=LeEzZXm%rJIGt4N%Rq;1H=_DY!ThGg{m7o!LWwDu%!fV2eRYA$gjdAN+N+drbMs_'
            'K;-UwhPl7Wuj4Gxi1Eir_R+>-C@&s7l$wfT%4_w|C;_3_5Gwqj!#J*jO%hzmdXnx@0OD!v<yM?aZw!NM^)!vkFU+&nFew`!?>D^~'
            '?WtWZFx^_CmTB8a0z?r86n&H%-f6;)j{%av+Zr1}O^i!@n|m{8cok*qC|*T5teh<)f%s{!q%w;)P}g3KHmjwrq%HeU>{>yp67czJ'
            'HuvghqhlUewMf7_;sLAH01)zuIr_hx{Mkh%&no!$xg54R)`ol$WsbaXr+U&=<-e*7Wu_ND>c+^ixJvmzwkj@)C&QzatGPJ)>FA7O'
            '%XuzG3AzZYx2Y`TYQo1%Qi~+TI7QR>w8(&w$1pDnjRaX)Tb8PXcyOe{HRquU3Y~@sB*E!Y>fbo-tA}n1Qbj0p#7SO$6;9F_#xz{Z'
            'o}6E4cNPjWbw+a@iYm>Q;#I0XvK);a70|UZ4e9uRU654*RutM@=p4pEK;2gqztUA+Hd&d}6H34`b&NWg&El>bjisJv>^C@`MMj<p'
            'ow4~IIk|OP+d-eI5*4Y}0CZPGf!g;mV8f~%)IEfBU%O7rloGH<3pkc}TUH0{nn##;di?tM`PuP{wqy<@!0=&W<Zei{M(8e6Z(U&;'
            '0$5~NeWuNMh^YxKjZ6}K@eAxl3<!9m69*Uv?}js<$*f{Y9?8gmdB~C3k?9|E%tuG**km5as$7+f`q4?`%v_&sSv}<mt+E-+G1>qh'
            'oRaxkRgR}qk9FBuAEIatmjmS7nK~KTD<|3Kv@CK%pTRI`AC7)LW&&)?l}cSJzx2Y$(QbF+s01pICtVj|@+x_TPSUA}VXhLQ87=5*'
            'H&Gl@a!*uRa4$}WF^lH&kQ`O4!zc-&Tr5frfe_(1=T%im&R)mJ78IFE=JT8~xhIsR4mT{TB}>k*`edA9(vXc!`8bgq=O!jhm*IT!'
            'w^3|~v`6DNWgNeiuccPcPQAnDOR*GEL!rYtWwz9j)2y!xeYzy4V}y&?Z~PkB!W)z#dB^garZ^2PPr9SnB{q6LGfDw_5UnEEhAD<P'
            'f5eK=4wR}Uv|{ct*p*^V?D0yiiY)?goUC*ll~b8)6o^|TCs3;k6&_1uME-<Z&s~Od&v@cpZ0-500n&Y!xF6lQCZu!-DW~$)=+*+a'
            't!o(2j{>O8Ce^;p&fN7H>93z#FlC-Q<FZlJF^xxEw@rU_1^@0~QUrjk3bLarahzTm2*HSt;RTD3gh1%QNwGl;)4QKXz&Cu1GW65N'
            '7;cI2pP4V*;`Z+5@#KwVv!%@n>(;BhrnBCqtv1vd?U%X2oYP-^JbQKa`gr{Qr&p(MkKc`dK6!Ec`g8{dIdZ`a<LEIueid#2t8{G('
            'WznTiQW5s{!Tc`UYO~3|l?ZUAc?xM9tnnHY?Ly3fj(F+E=?&TpE6KDDOmP%Lv$`OMF!6_)^~45Ot^?=Y6qDBHW!ZS#WNR+^Yckx)'
            '6gm5IAj_P<F!RoX-p7mE!>#8x%){a~>mR9iXw^g*8@pd$V_qw99c}^<UEwLp<}oT4K=1~3M5Hj>I8q8UV%&77uIZ9Y89J(C)Ae~z'
            'YbWEbU@Gp`r2jF}QK)^9hG96^WS^-HoyBN&tiODF5?!h~irr;Sd+WAW0d9;qvY|Vis$D%NDU%b!USLwm2g1ZUY}H?_A9*I(!zkj8'
            'Cg_qL1G^c;2AM;_feD#at-x@3>*cv8cSMshv?=x?`Pkk%uUDAO6{Rz6mAzH4MhZL_UJnQOqk0ew2Y)oZk9yPLHNF9k@WbJgN7R_o'
            'YEvE;%%Ou^5Ivq|bNuh&Q*IP}7|&PqW>a22IvlJmu)&nta>rqS{|$nvnvhmjO4Yt0Bk<b#GWx{yu^!p;-D|jkByEGw%-u+(T?BB8'
            'nczTft#<7j@h2(6$e{a0isbso<CiDzj#Vr_F(l%-%#<J#CUbft5zL|A4SLzAtY)ML;Vk+A;FsogC9f>A$Sp983zO(Sqd0#Bbrxdj'
            '*-SCY-*ET5NY^SQQYB4HQ_Mh(17S!irgR=<dBGW#`wcv&657y|tONgHaAh3sDL)q`EY910kZ0<|cGNqcD;ugvrltc~`H*wpb2FL1'
            'g-K$o(>C9~ZEU@T3;APQUg2&buStuF_TNPutk6P}p_UTf#j{8nN36KEwm7j`bbqC#uGD<@CW-arNA03{)uth|YzHn^{d9a;?T)Gi'
            'nl){YzgLSE)=RIRco%YJjfWBaIPl<~RRNHgO2?#*D!q}8fZUO^vpFLn@7ze`j@>b7R|p~w++vpfRyj##gcvyzm&vWzxg$C*gzlGb'
            'XhWi%k)zczPFK~hmR<AKPTSFVqIR(Yc9IXPyL@_|?o~z*2ZDOB8;HWeabB)D53kI>B*S;DybAiX_Dl5Yzfx7+Y;i_LuAZC{M`i@9'
            '2DzEBy-!P|bL^g*DUL3xOt<dy61H#Tu~U3gV8sfqKg3)la?LQd(xu)P`@4cu%<hfJe*n;|%z;`i%Igr;V}Ey6tqPivb?Jh28J<d<'
            'MSlI3yO8Q;Xi$|VPt`eBSg&d;f4(-=Y)@3V{U|Ft0f~O7eO15l%xN{|B*yTP85N#dP^hc1XPlv$3z%C`Na_ujFt@^u_N=hF@J4=4'
            'hR(2vtrui2qX2X)(V+3pumg^MKk)CT>-`*cbG^@9YG=jS=j_?98X2Kp9H`UKrAY^^Upir1EeUC?*40wp(YiUJ>W)rLVN`c?ae}|>'
            ')>bPc9PCb*COSAEYxgQ2oHm&E)ALhy<MGEdT~(o62YmxoxxqZ0mK3g6g?l~rt#=#yopDrlS`OZ^nwT{H@*Y>5oT<&m+CV4oUj6CS'
            'o1@p#hJdPRodcl3fB@WjnnWNK2n-$XDf&cFh@?5xQCl<VIZIbej)|h#)eg&cQIQ)sJ?J3t`t{GqE1jL4Y7H}&v2sUFsb8=t5%BNC'
            'vinN9PiQ8T#+W7%<p#S-IZn0u$4HpOsbbKJ{yG$fRYEJoNw%e-IEEJ3=n%MP4xg89`nA_zRTKV?pbF>h_$-C^{{0o=`~OK4;*L+n'
            'XKTk@hl%~4!)rW2*b}5bTw>1X=M1fDc%y%jf?cx{TEDKJbe$GoMZ5l|2q;ht)Ncl-r~DhJ)<4CO;@@4#ZXUvRz2}~(xBNr}=%=VU'
            '@6k)<z_oiw6{*GbU^HqO7oVpuwZK)H*QO7+s?~oid&M)yn{MuLj$o`Us7`vzSi4)-8>-UR*H!Re!~D?zwcQTle)?}=5NVob{rjeg'
            'q>=F-Ypt2%KdF7;nudvMQ~wwXlWMPVZN}V)D(rrmnWi?-CVNfO8tedZpH=WwfQfe>dylnb_c#7b;D%9$#W4ic?!Zo(CE?W&Gpga0'
            'QNaCx39W(XC!^ci7vWgBci|3ui$i5T#qMJ5gO=tDI#oTxI41i;=bnD+T{M7_#f~u|yCwp*gQFDK7tYX3VbB<zD<d{SbsM>3tz<?6'
            'vMoq9!nK>YP_<a;tU&d=38sEqTLCSAHo)WOZSd%A13VT{YvAPLW@!Af2CRxp9mx#fO(<E#?a88|T?dREqo?C@8-Y1tem*5Gv2R^M'
            'CrlWM2PLFJ&-M*YEjmqCvZ<6p`y(lVyvm=zQXEya#xVerM3g)%eD&uvo9o{>rgcF21mDO4Y0txyULb2XP^wP=n3MWO96a?=?l;+7'
            'QjN)cafh62$yAz&Z!K6|@!)|tgolR~ugqGsMkw-CJD9yH!HsF*wH8>v{-<e|O}m4;O}pQSQ#b-)ie)7p_YaiTA`5%G6O{1$=;*mD'
            '+L}inD5*{YI3^|_ki9-RJw7$-OBA_?gAH0M(LtUi0aYQ3(gz?ghOM*XE(89RP`$m}07nWR@0Uay1=QEoNA{oS@$7qjafDXEf;2ts'
            'Q;ErZ40x1XPE_UeP{~5`*P0?W|Ei*eR&6i-q?y;9iqc>uDyduHd129;I0`1Z&}-grGNoHv>zLGSO#dRgx;{9V_t_VGE1?>k(-*Kf'
            'iN!I3PK%)FI|ol@gA8)V06?Kfn~qc@vDY;$$CQhmbS0Y%4L!~*%xe}U)$mloD13oO4YTr_?&S4vx*VCEzEz!g8P{?JGbm6+2S{-h'
            'arUBTLPfvy=((&9g0^sWBM4T($G#%lRu)m@lt0y0+@J~2ic3U+6Q`nz%BCvErTrO>5^?nQ?Yom-k6&26%~3*1OWVnrRB)tLMJ9}#'
            'fR%%<1V=AOB=fbuv5X4ff>+T8S!0d)j+WYWzACCl)s}i%oryOwW&2`pU0ny+C^~JI(8l7^y`BMW4G$1$0>P;}HPg&xq_C8nj$qLF'
            '|BZzi)m7$*R1E<lwRYXAc!x48QVF=RE^mT*Zu?bb=F{(@dDFlu`~}R?%pTPnYG_@Haxg2QApn_vcz1seg>bt!j~y$cSYimVW>qy{'
            '-HX#4YgAT6sAT>UJ-LSOnakMxIcRaOugTKAY9YGy+~zc%#)0jLmJHwbnCpd`{`nk7(lDa7EaG=h@S-lUR^s}6U+s0zz^vO=1Tf?D'
            'y)wB<;0E)y``s2Fos%TkfH-O-zE)Y9k$V!ITTHRRyL*>Od%jPN+fmb~Q=>2xt=k{NxLAeVeJe+H9fRCt*_NkKz4;dCUJ+%oLKP<L'
            '%c4yjC}KOqpo<6|8*WB6Z*9A-)Hdec-bZ!jwwAE^KCVSy;*HYmVpP*A?5xSK+wVvvJBTa1CT9=iXUJig&Uyn`P=y++GhH2<T)%At'
            'yqiOC?^|{M<H6N6!e&ulU%dYsW~pVTW7@I`1G1@MCJq)aRL#@`SWNP<V56L}J*tbhz@QkGW2mAm)+!{JF`C#oi3b+NU{+R;B~0?R'
            '&=gtl>K2DvJDP`B_?VmsgG5H{7?Fe1WjMW}qA8Rx1g)7lT&;QP>uMGiJlsKCG73JVna!vO>b_%HeEa&BQ?gMSR@i;|GS|cn;<<!K'
            'kYr^g)C^yv$&zpFB$d7>W5b7Br@`0TY;$0DZT_C^77=6#D+TmrPMG9=c-Y`kp}RWD(I4+qb&mJz==IrOj{f!4>8pR$)$wX!<fni!'
            '*8%W3047g??1^X^X21Xft13FH@?&)MAh<%`)&w?gnP4=A%2?5@K|5k>mU>1d5fw<bk>W}F<?)AgoMvJ6n>P#e9PqPn9vpxO286l|'
            'a$#nAev{O?Yu2?k0d^-Vn&tP(f)7pj!<;+{{_NN^wt~{FdvbAxX{y&|{A7PE#YgXS*z#_5zpbVSbX7!{rNgH4+wL7z3=u!;IJTot'
            '%@d0vw9UGOh*cEKO7VzN;vFxtV7=6?m-;c+=b@LKJv&30GzFSU9^cm1_yYYwSRXU097GJC5cUQcS-v5@e7vz9Z(K~k1Z2B0LEV{s'
            ')Q++zhJA^ughb|nzWXN7Td0<7|C|OoyGVeuIll%m|6;%1nu~~{rVP_FRokw{M4YqE4XiRKSJn>8g^I=Dy89GYfwS}0w@kV@+xA|q'
            'bc^4x=Jo-o66}sly`GVtdo!|Bb^cy0fi~*!M|DGhU!~U`)ph$e9Hkr{)l|%8n&Y~D+=ZfYAJ;YCc1u^SKWa1<_<QYHwnq&n0K2``'
            'SdZ$eZioG{)X}3Bm3CuAr*gu(Dswa9)ycp3DRQ4o20C?2=<b=@m`kUJx&oXzblt{l=r>N=vTF>f4Kx{fw=E;nP`X=hAJ1_$9Hxj~'
            '^UUp&Jl97VcgMS}EzQ=ub;r9M=F4D>YmP>`dmMjFjdq@Xbp2?`&+$2%WnXt1jHD#)a(kF|`V>#xZOsRD>sy(qM^$=w-%3F}GIA`t'
            ')_Egs^ys1LHahfW3S;A|>ilj$ar5-do3{|qi4$ny2xJTTW9!{x`!suTPvXtHF~%x;G3GPHvKP`P#lC!fXG#}JZAc1BjwNco@wf%0'
            'v9DP#q@IwqvlI;VBDxAO!~84@{wluH%s~-ep%)r(cp;@5GYmh#rAX0<{qT@m$Iu{309vKYqnewT=anjH<G(uhHjYFsx@&l4FCb3A'
            'deC4H;a#j*MDA@6ajag-CW7OuEMVPFiF|8Q*TVqQ>v<e3NPqhs8euq1G=HtO^<S*amxzO<WRvMeW(+hpOyzu{&1?$gk?xez_3U|c'
            'RS@nZ(Upp=YVbmnpRM|$o2K&1<d+o{YPBTPmSZ>fS-E!t5wR8q9g}NCjIh{&ZEQp9p)U>|_f?JHmdTu}Md7Nm7wAHBuY&Sz;ABO0'
            '1XQc3&i+KTE^8`GsY5Z)76yOr{(!2+eqmh#p4?54A&M-G(XViUYPDxFwHGEek~E{SI=fw){T`#;vm``va+w9o0HpPgaiDOEzWo5L'
            'EOK0>Q4%hVXQj7iBtx@NR;e}+VWpqAdeoSXKxG&j>msD6m%ZIlZ`~+P+~SqXJ1Xs1s<l-Dd%7eqb7P&xa+!v^mQW=UW(yPUx;^Z3'
            'a-KnbGBeaEB?4}kVjqiB<(%gm#m@BV!nl8U0$&Fqom%P<tG%o9U)5=p2?eWKj*e--*1luafGXou3jilGs8J!It<<}9I;xuQ^;xLG'
            '?+tIP>hCJUc}tadgVZz>c^?yPY8jI`>OY`Q$75P9^RZktXMB!E;cL#LB?-4Y5km{VeD}Q~QA3AI?NoUOnI-^_F)N>mqFmWZ7r2t5'
            'nvQ;fN$3FYeMdzlyhe%^CN@nIcjyXis<vaiAg)j0!kidtz(1e7`Q`NIBg}NCRDW_lNa$Q`k*9QlzPSI$oAs6rgBHz$t}XyrTqevw'
            'vKQ&9jByKg4WwHu^KtLX!5WWcZkP4F)lDJUk5zys7>uhRg;ym(7&r!G4&-Sd1T!5mYP^jS;4=_IojUh!e4#h2dFL9uTKlD8^lg0S'
            '0%(lv7>bs#Wizp;<G)WAitW+g*v;^U$M4i0R=u{yvg!PMwS}wh8hF=m139Cp-DXHE+-J>WAey<Tk3bnv)G~N2QK)XoH_uRcR@ZUE'
            '%2irUo%p|c)NO1m)t*j!`E*y8O=Fjh+r@MhgdXOfPD@jJhozKFWnN=TX0)rOP3k?*WYMd&Yl3RA?^V-As5|VEh^zOtT6%RKZteL^'
            'BWU#kzXu#v!Ku@B{aH|?xqaB`Y~BW;Awz&FS8+#<06sycx*6m0&A(gPDhJkT{hFgqs#T@uZg77Mo?gc`aB_9;4UGw(W-WGrwOa0m'
            'EWLx#c9KUHT<yQ?HfaM`xwQ`)yyJIulK6j@uH5b<FQQ#qTba#!=cg5>w!u-JBP~E`Zt_EM4~@Fl>@aV3Zd0y%HJ5m^SBrhXtNFdV'
            'of=F9p6lIi_mNhL<}PNjnY)Ytx>KzN&7JlD#a@}WAt}JJ>jvJ4FM&em{b;E6=~-GdXk%;D8r!0tZD@L#H(u`5C0)DU>(sJ|NPZ*U'
            '|McqlPp0o9UWz%V4UKZC=s-UMG=~=C1KAhvISVqmHSW3P%Q*e3%GoT-o&`FCGO(b9^w)^f^eSiV*N~pWJ~-ge=fW7=^Vi2m?@q}*'
            'T5(NQ+UF=y>@#L~BG2RofVwNYOy{BWFDcN0zg$a?THUHh1!-8JtphBW;3`zUAvAUEmdU=}GCIBQV2qJP%}S9|lkZ4os(W3U37>5U'
            '<Dig=M6a=ovgxcT>lW3WHC@tV^ic?cBi%hN-LgjLlv9!{EXBSBb{iwb%TQFVaqe-c`8naSsXc7#WqqXKsnf4b%ta=?<(12nWQ8g1'
            'G+3tGPv&`Z2W@vWgH)@>RSk(s4dHUoq<ba8Mb<KjG(*?c<8*7b^gy@f7w&uvGdtZxD0?WIJqAHG+mYJl&}7m#hGIT;2hyVB!o6&H'
            '<v?gvNg=|vy{k!f$`7k@b{{%Uiw@B_s$ApHr42q+a+zi(sz?lws%eKgbZ}dhITpH|1Vs_ou7CiuNiQ&KA9~i&h1Uy{Fk5VZmMsz@'
            'u?Z0ByreYRH&HT6Z@75#hGIXWqIL;z4t*`f6RAg<Oy!z`WFt}{+PWr}?@-Y?)lQTnKKkz&nZDng7&%K|2Fk@R4i|8t0Q!d9+JV@?'
            'vIB-&U=oHix&@S5X_veLBZBtk<V>X|l>U1Tv?*K;D|Vn~cRaZ+*csJ#oFcuzP=Is}Lz@@68XIT*cITU7v6LwW_B$6xU0ntEQh(Sg'
            'BeFA@B^dIf8e>3N)5$%sxfH^bwCNe9xskCBJ$385voR_jQ1HPVXyejM7zs0=N(vY((RbzibTw}9bXFo<!>bw%U9aPabS0HrG+bUy'
            'U7dTi?nd3rh_NNE&V{i?*Qm=j+-c2h>?d#a17r=%>WFn9<n7@;y~jmh7&03SEX}G3!)}V5Nbg>M<%`3X6>}<B)K<>0BQZ-|JsMAt'
            'NOK0K>S0i*hwgzXh(C7Q*Vrz*ucbo+v@Fyo?q|=mi&~=DUoMVL>UL)#NHZgq@%TE8rXdmr`KH|ivX=eLh3!7opc`D<8Va+1TAH}_'
            '^l<IpY0n$CD_<O35_s2)Ln_-iv){IV%n|uWR=!T-5>u*GdAJ@v_C*q8(pc`kxAJ_|{=tDU7f|(hx;ww!&b)T*U%Fp$lY8YeYyU`F'
            'qO)BI&hCQNnEC9kjAuUqdi3J(8pG}|e`Or@;KV6;4DdhO$Z8;P*fF`+WJ6sCPux@4kJg)HLcGXeFvd95auQPFja4Ak78vNt%}`>1'
            'n`2t!D&tO-woI34G4hxBVu5TGRUp+YOBGcJL?&hzeuZAli{TNY#UcJe2j+f9SF#D|JF$vBAbZ4l&=OtmgWX6d#t)96GOvS#s+=`)'
            'BT2I_!w$JntDP5Ztegcn(-e|WoImY`%ooR%)!CFpi4fN1B*wT#j6K+JiKUKdZKBjpAU@p$o4l*aK4}46eqqR**St)aWqW?~26-~c'
            '5el}Dj;c!odEN-akFR937%+BPo`-`}Qj{sM(!_~S{%M)_jTIiqLDGooMt%a<!H_R$$mdd4dRc|x+Cy>J-6|$TayiQ&7dM!z1=&Jo'
            'OgAA6=PjT@$|YM;;|7HGSfV=TFpw&THjI_EFPBt*r9xsgQ~4vTvITc9eOV4i&WCsY^rxd9T!O%xYE0&J;3C5Cr>xU8ElNTh^k;^_'
            'AH8`2+#KQ+rQ~i{0-Z&$NwKodLc>9$BRMcNZ(r#9sCve(`s#TPYe5T2dn0F}Vofo>AIn1_mMSF<obh^!PM0%LYANGVFIP2~OYy*X'
            'UNf~_5_``Fp3;wnmbVO7TEY+{Dbw)k!2=Z8@-PI#0ZU0$&{$l7qldm4ShCwr9r%qnA#NLhrV~;@W-RtdkDl%W8LiRn0|Qfp6H=*B'
            'G_NXXhJ%1xRDx4q=KSb@+i95teqsqTpj6Lf^nQ}3aY$fD+`=<>g{rO82y0U6F5r&Bgl0zmHC$;5bM+V!4`FV;Ol75_d>ti;5^cUl'
            'cEo@9M04y^ie`4r6Dz@tT!JoYnFXy#ROM6kJJYAiC=J{UxKDIthDCNAs6^fsRlVK$&8;W2AjQRNyZhbRX!Vi(+5@ftw~V>v%gw{K'
            '?_Z_-mvysSAXbe$E3C2+3_72|J3`T>2932}+v>AoX2_h}cQQD5vozlX=ww3WRae0b6UH@t1IsqoGHFUpE~bw#=W6Y^-PS5l-sF){'
            'FH37=r!bOn*30K|cT~K<vENx8rU`Fl0!(+jeflKj#oTYt0%{}x<=mg2mG;w|#+PRQY@DN$3e@=C-zp<${|i!r8pD&BL8+|hD#|%!'
            '-8>5889HtQMMaOr(V#y#Fb+TfEYvP+qW)dKu?jQ4Gd=YJg;|KdlRBC;*2^?gxKHv9dz%F4B+fr4-LCkdn!%odV0xbH-8m<`nj=m$'
            ')|Gcl$&*^gSgWqS3u@qZ7gBG)BX1U5MU;7$O1_Ja0WvDHV6_OyU9gTyXd&B_`KpBbZm6>68bhc^%RgS;)PJTF`#Yr+`*W38NTrJS'
            'XGXDqW)%BpMzOD(QS2Vc#O}@`#&<vKkI_0rrelm8wDNdad}Gy^dB(4u?dvGPME)!Ea<&8f4I*;S<T4s=BX{mMPJCBZnSZpb4WFE}'
            ';U719!k0!5w?q#A4Wov?AY$0Y!@pUiFdr2g2l$zZ0CfsgGurruvBckDnyDLOx?dJ+?3;7=cMd%Mj)F_RDEzn>eEqOv^{P={VcwQs'
            'BR=^&;3xk;0m@dLgTGIpaxwU4fbxIh0A;!K4|qFz%6-fTT#h($S#<d0qXO03&oJKZ)tl3!H{#^Y(O+JP7sr2k_4>uDcV;x8M|wrf'
            'J{AZH76chB#Z-ntg6Z`!g~?YBalT_df5U)h*F%B~Oe5r}mq35|vW(?D`CMdSnF@8hvY7Xtdfje+8Gevnx83vM!IKMVP5DJx$84h*'
            'Cu@Cubawnso0t092}e7#AYJmRI=JYBHX%zhiaHYzeG8N$rNxF!HK~ZxWKo&cP(TdhIb6G$DzA$+;yvYm7Zg<@+^8T|!90{@w4_PR'
            'F^M_mu#ObjB*ZE?uKJwXm#(&wDeD*;@@h7+JA>0zt<=huXt(vGEh4@0i)jMYX4<$L8e-L0>dnU5tIgK`-UHJN6PcZs34KD_Mk)1<'
            '6_)deoAAFeVD$f=QTnF+WycsQ9h@7|ZxUnoy`;g|%nh)u2+}hmj=<ZkF*Eiogd@fl8kugcK@WIWT7f$T(`oY069tTF72|QiDCOFC'
            'WgZJA{;%FVKYo4u^YQ5`@#FEIU%&dx$vZb=VTXg8OS%GPz)4`4qBO|@o^G(oI5{F5XOP2v(dvRg8f2<D9cA67Y)0tA=%R(Kn{Qf%'
            '??9IE`T}$8!ic6AY@K77Yq+B}81SCvm}?g`)-~WDA$_I5{Dy)3y+W*;JpEJ%LBp%o?zT&y3&DQ-sZrW?#l62%ymqztb#t>fY%@gv'
            '<HdWohi!jKT=X45-Tz*J+`GcFzg|f8r-flV3JmlVc2pWWBMc3l26Ww_@Nm4F9qk7O12~U9DH!of0ub9{4%;HW?h_C7AIHu93q0pP'
            '$xZ)1rJMaUMxQs!IZXQpze#)PTKmsw-@Qco%W*Yg+(Zi2*$6<>v_$XBKDyy)e+3pPQA+Z~Rk-oB(j4SG^hqW!0@3gHbq`j_a}QR6'
            'VQ3EPeSGn)(#w1I0L3|>36g9}IPSFm{{Z&^<8%'
        ),
    },
    'google_ai': {
        'sha256': 'ce418f526ba99dfd8b64de03143bf068a1597b355592a29ff17e735c41db80ce',
        'encoding': 'zlib+b85',
        'data': (
            'c-ri}dw<(DmN5LkKLu8Op0(IYvYn)vjx(CRs;wm2_>!^Zb~>(?rX<SdiXt^6Wk>ONKl^(w00{6RD`~RN?EbdD-HA*BAaHPSZa9}l'
            'qjBIxaTJWbuq84towdDtFY(2jIKBye(Hn^78-E@|f!Go7W9~U3iI<t5h-sXN>^4ZnB8mUQA7`DdtzX{$Du!<d=i>PE@ZzX1`hOjq'
            '56`#SmH%!1<iij?k1>xYOZc3B=O-%>#hE{f<Cz#OgD?|8mWud364y6=<OQ9BFz}<S+39qgYvDzcty?ddjN^$v5!Wz`A9lBIXvXd5'
            'otN!t=%u&Y*CL3*Ao4q6c<W_i?oE6cRs{1!oTkADzMRFP32U20@%`5QZ7{wSG*$}J^TjmreYH$6_9Eepv!xe?E0Or#__igQBk1b;'
            '3*EY%1%9%1_WpQh2>*Zf{t(tOx$!f28PCPY5271*l0-1+(Mr&W<6r{o@<K87qBelIg^Lr-u>%i+5EdZ3twrKbf^p`-jN$ci63|a^'
            '@2zlUIUEC?_vuMrEJAOLK=!iztt}z;I${iy&iqt(VoabE&D4iySv!a({=$!N8ZDuIj#uy;7w%6S=>M#Pd%?pKga8CLL7Ks~eJ93o'
            'G!1T+Fv%PU;_u5i^I*8Cw+yoshIrl)lOP?(u-7R-bK#AGFo341xC=aS&6Dad_&Xtt-Q<l15CGo6Tr>LoT-*oQEsO{M0Ra6Ei!d0&'
            'roz6&_t*_yz37NVFrMLBCq8sWv`DjH%s4C-Fg?V1X!(7IHV|9F(-c|}6#D1G!EtZcKa|S|eb~hW+Q00;!m|Wty(2~*ej}bv;Ws>^'
            'QN}?sUcz)n0PPud`9nwCd0{Xi-~+^#fJ6u?n95?AiF5_KKhE$Z0%G+}4n^wEy+}TQ{y%g?7~iDwkN`T~0mdZ4%QAny$PhX)ZHt#='
            '!4yV<E`RKZ)QbSPAAQx&PZBt|BR+e02JiqQWMLwSUi)NA?Co|CsBucDlKKlTftj~7pkb1cH-~QH7{~%_4?<It4ZW+Yt*sMzngKbN'
            'X@-XzAlHVY7y5I=J}+7U=y>KMKnlcUe?@0A$xP}11m@@d+2Hj6cBA*EH#j*TiqqGx2M2@Rk$7`@`UWTt(5u7KgNx(-$*?yZoSq<V'
            '`{JzMJ3Q`-#yf8nc{4GIZ<1x`&Ao<GLz_PTzpb`7reiTUOc63ZpZMD>TcrCtJ3zl~2t=KUf43v)l(z_WCh<7kVVs8{3Wop)pO3(I'
            ';ea7tHkr-=32X`ZDAj+#3G5PaYykok2J+`4f1*dxB((O>5qb7u9GKn`cMG<g=ieq`EYf8HsOICbZf>~7;0BlCbl{^j*=r!jD?ms*'
            'd7%en4k;Xr2iOSIAX_c`wDawOY#NV{6uq*T#==VxM-K-F!*h|wh}3E3tx}0XNFlcbEpr}^XG}1Ga=@`8q&q)+2WJO*gmAb!2z(ek'
            'ngNA&LpU<wHPU8(0Ys;PE17#?L;OQDfXL9lZ~}tmoPOR0H@6MZgmcg)KsvYsn?Xf=5<Y)$dUn=7AmqJEJ2XYbdPYERru#5KIPi!N'
            'Kt8Tp@MpYm7wX51nIdjI{7cjZHhdXnyqzbfLpXkOIA;^aOdP-Wey{^z=!nzl6pl9VpqnN1jDT?fnSimR2=JU3OAJ)dy93O~X8}Zr'
            '*tDebz-4xAh6G<ph~&UvDW3h6IO)S_7U%tw!*eF*C{U;mbv`KmO9%McMNwelng;>#fgsH68u)Mk+r^xT0JWVG#{d$ewi^)zh$HY`'
            '514A0`pD%Hf9z#J9CbE>z1+EHo$pz;Xakk=uMsc^+K`ARd<dAeP}K0;_Y#=l4KN+p5%@YY&J7QYJZ~utLJMXbmtTYfgVW@}{Yg9V'
            'xh1o>Pg@j`2b^L1*)<$_pTCeU7kG{krI1;=cqFf15#ywTgNzK|mjY!_vw`uEC1w7HY;($>zn(h81NaNDKhxj?tQH1>t%?&A^yFt0'
            'r=w~39-ZFTTJSKXuV@K_gSerNz+yRqt-8&J!Wq2Mzr-R2)B#k7j{-|M&p_0QiG%>10Kp*)=7{b%TEklaKlGXXNw~I%%i#k!Ss+)e'
            '8q^nsFc`3b6C!aU6}Vd&o}PIyiGX}$;=MNx;1G%|nEQ>*frqD}&C>$t^^w&hO}T;f;l_bL-am5B`forWKl}As3W$6p;V>MWp1dBs'
            'xj5_n1U&Sx|GIZ^G=xQ<#6ZI763$No+#Fai9ov1eT!gVVNn09{_!H#_78X#RY{9<?5EBwU@`rHBlNh0xZjJox-uEMQPJqrLp-Oio'
            '7>U+QVvx9?I{~b-)o3)fh;F&=bcv+Lb(N^#MG=mRsJOKypN+iKfB8cFbqnV;3`XkD|48FV{f<-hD}k+w=lU;S{bEE{f8qA<M3BPb'
            'Uuqf?l9JTZ7PW-qo_YN9V;sqj3opajsDAI^&#kR*#FrFCz7cOhIRb13iESlPpo3%BDLP%LPv{FX3r0j!fh04rZWG%>Q38-1wu7)4'
            'hMF!TI_S{vhJ*ollg5PAVj2dE1)>|Q4w1_PS)LWEG+rjaN5;2abi;ZGi4@<U-T>I2;(#d#%04MpK<qnT7zSJ4?29u(4Ai-7jO!g7'
            'oL-y^&v)LRo&H-N<p;nxc_vFL3@nR&;D99E&+h;T$T`G-$T27Yth8M-qj96yl{oEEPdAYWioz8fz9q^nKxAORWEqKIio!5L5WxxD'
            'E9-~@gd5JBPVhNWI*|Ga%poXVfKf<~6g_VOw}!z&4;6IYZtM$-0tG0TPT|A@S%by?JUHy1IXLMlsKu;ZrNBvfca(~cETFU1J32c3'
            '#g(9P|8j9U?73&BQoinr*T_f-$~oyxm!o+=dz&ih4%SM#u7o^H3Gg0t7?KSFBAt^Zo_P|~ycG@z-9USyZnsO&G2pMez`sXA{xn%c'
            'bx+em98GfzQ+T`wvgV^2nvgC>uK?WxKw0QFg58>TwqBq0`|fbiKXVUzuy|k~gM&USy-C_<LkcZO>z7F5(%_?*c^OQ2A9x~g<s=9J'
            '=U~hvn8C0D$W}&=R;ZkXp4OWhJYWW@7VHa1V9<0Kh0ssj_$XTh$=nC7I|i;bTZSQY4@uEVa9F?%=y4N&1hriab>jo7`Z$CzMvZ67'
            'NcL+!3BVzId8)>{^=4rJgmf-|s;#_LP+YPnmpu1ZLDXq%IeY-Ud(j-ni&cV5&Yw7nE}#w@?|8&tQKBKCkeLhPixeqe0m_H8jbY3h'
            'GJ+MQ*bE4EItHy9<vbwW_o$JlASjS(KKC;ZXt77cE=3(j;`3SW7x%b-IOw^<Q`gvFW_~$dp97*_oV!2wjxPG=@c4n@09r77e@+&g'
            'MoTm%NHgOFK5)t8avO&*Lwv<Z>n8r+m#`I+MvF%PxdG9Cf=yuOmLz&(cbqOhM6(aSmS+QwlZ)qJ1w%Z-%nsi6k9#fv?3_>H-qzOE'
            '7-Z5^qEm11o(O**RsQA~kRg~b{o(BM92$)?FTjI&e+ww%iQ5vlz)iUeVx%aO4l>`ZE!Yw9G=UTCKZ_F}Q~FD<VL2)6-f{w)hop`>'
            '?tpkCR5x+&v7JJIb~z$_jJh9?awa5o0yPIM2G{-ve++cWW%PB4XaQv<t`x<&Eg}qww*WOk!ygTZ)-*CN($Wz%KvhSgnz2z%PYCq_'
            'xd84s73=6BuNMDH5U`aSp_|aS>qhGilcf)2Fi7x3PrZ+!r&a|F91bu`N@*ZZrydkedWt6<h^coA=N>3|Gy$rU0Pz6nb4QBCEXN{w'
            '0rKUgb8jYQ$mt5B1-BvEt(eG+mdtH{39Y=;n*mAc0DmQZvGT$w_&D}N@7I$7HzQg#@ry$eH-jU5Lp04NQq;-Y0MYp3lF&KHpCJ7F'
            '?1f98h4=deP9bbJi7|kOW0D~eQ>0C-g<9zxO>7gCTwxd#8rxd1W%I4H-F^+prT+mr)&U&Akz4^;-B`^9o(l4LNHhWDQzUsHg@?<O'
            'ERy$dnA>Ae!~m1g27)Xw3_u`;Mgj8DG!B=H<bYpEXY0IwaMmBXy`w?z+@y8#d;w2r?2D#kcke)}0n)^FHy3!H`>y|M!;!>o1qiVm'
            '`9#Zf$6xs&wu7F(o?Z+ue(EzzE%pa>_V4_$v(vxmw=MO0jAR7VZxH_#{Rt-2{^02J;_&rR@2s!S)ZnnXBPca)7SH^s)bV2Y7P!K@'
            '{z=}^`-r+my72w+t&18R_UZlY0-B${@AnVh+7q;zqlG+<$~{mWJLPWxIhUNrg*`azp9}}RBNqg<*MlRJG#Zi(Hn!A5_oR2+hauo)'
            ';&*VHfm<|_#=l?o+JE!fA9ve-yvl#No%YqkZtLaq$N$rS!*y_a)ceVm`~9|m^uBJh^@(gS3IfW2-V*u1+h&2LE=_s19JKA+06juN'
            '8q9!N0TbB%PW<WveYpufh9;qE$llx7s5jXc+ZwU9Mf1*IdBRWeBNKRIVcE9R5{LecpX{5%<pN<!>bjC@05m(W8e}uxB*DocG4;Yp'
            'aD!ZcC($uS+EylO0mmFx6l|+zB<39)_kIOQFIs}Q1Lqy&o&}t%RqD}sM2zp;_!)3OV@L)NjQbkRG{gsX)K_}rG3Y_`3}sGD2-lsZ'
            ';=afY(4fqZS_zg_=MEui%vJ&yHNY4lh=;}iCw>J!FoX|?LzJyh^1XYmf%GIW#~=pB;T)irFZEFiF9#qbp<?BQXZ}=+4?QMWDDkl+'
            '0T_1-9QF>Wx#I7vWGpjfy3m(uAabO)@qHxaiR;8)Bp@p6T!U6m|1z0py$odl5;uv+stXip`)Ns2jxTM|EESN_ApY(A^hEOpzNAZ0'
            '^&0CHDGn3*-i&W-(7YlQlNK2b1>W)qysac#30-ZW{TT^05dPF7H5!DSn_DE@LF6SX#ic;m%f%J?;ks!UXQ}I=y3&{d@+0jfKHws>'
            '^k07J;&qmiH>W^N@TW`gMVf)5q3Gwja0)WlZKi$*`YsN+kH(Rf$Va6I;vXacwuCi)i+8ML$piGC{3`eDih6qJ<Wd&Sp@UO^4)~A#'
            'BA*bv&i|-rq-F>0GSEK!ruEdZd&r4GTXmo3<~!`lw@vwPp)bReyTZS)!3ZOU?*9i}K&!}B8rJ!RDnXlsYB)=wA!ahR;=`u#0jF+6'
            'dYccHZ&N-b;0{_}6HwXsq;<q89)@dagJ%Z8H%541;9xC5jwVeV_2-U~m5uosQ_C@l&XYBl(z+Ts1R5xN(>TU8vyVpn2H6^-n?^o#'
            '7Q1Aa9J_d%lz#*NvQ~PoPIpTm!*=RT(J;nJCE0flVjv=*q=}PLMSaljf(N~7wGXziOQ&u>$5k8m8w3XRo?+q?ny3T0Zy{izk#VT^'
            'i}+vF6tG*G_NfP5`jt<FF_lM?2p|QOL_qJJ5Z}Nl_HO3hJ}O{z#V^`;C4L8xDnh9_ZT!ii^>l)}4;oI7j3M%ggv?_>mB9a|3P7@{'
            '{Ogg=ynL7g<MGdmo&j~t5~7W2LFN*fsA@_F(%kqnmrCr0Kqd8OSIToPW0MO41E5nAMSG%ulEf)qly>~+E=T|sQL?~RYM>1$5D#db'
            'Fu>WLwJU<hHvBQ#c~P|?^j0(ugbP1pzDgVsFX`r9v;<ZL>+#7bp^t!~ZOByB0MNeCaA5b~Ff|>eLn99?O0u2lG6bO@8{amWh82}*'
            'QrxO^XoJ71&+fHfz5vNA0R<H{hJ_7OU$u&Q>m{(K@ZOuZ#CB&J{>R-GBr0vUx5-#H_rt){NsS$ml#I4O$|VZya!Z>AV>CBH|CR0&'
            'HL75Q`F|F6bdgNcK2}ReXFQJ-<OQH&N_yeRfM^b)Rd9p04G=oe(!slkf)VW_-eB>>q>Mz@x7y~<0u^5^BF?}D-tvVPfg5EaZX)k?'
            'gf~UN%X=FdkUUR1;##dAFa8&)L_QngQZ>mmS9zr{M7pZv;l!%Z6jY(^Dyaj+!Z2nNdL}O`&H>-@#hTU~4T?`-?Gl?owx#=$v3?C|'
            'Hg2$k6W0S%(<sLQz5Cvshqhc2D^jA#O*}?3hmITyi$V{KBR)JjPW}0qHqfLJ&HWYVI_2ZBFCNs4{v~;Q1#;!xc$vXdJ&VT%k8beV'
            'uAcR!>U>3Q6r{Uo;#JLC4noF&it&mG67g=9w(@rtOx3`&^Q&e7%$^(v-cn;lV+dd#nfSQ>IzlpSn2?Yw4A{bsYT`$<zKI(j$N@Z-'
            '=;%$*9-sI#pbtD@KB2gE=846rRa1KEMDxTc8gtF3nkB1!i#90gDtQHU$Wefm1kqQzLUmsfHr2yGii37jN#QISL&Qu9@~zy8#FD&1'
            'i>?BsMJcPsvD5<fXL%Hxyq3x8=TS$xI*V`6txoNyQm5!npEhKC3*w$q7Rx-^p&NAN7b%*ogkPKbKl<a?eEXU#h;(2CP{>S7Vjz9K'
            '!`8?AN{Z<y-wLPVTI=2LiEEr93NxZ73`8QTMyQ=KQbl4JrGB;(#^YJ8Fyo~^`F@gvN&^qg6wMJ0>C5Y7`Ek~_|K!V<c<HT8Df2UL'
            'TMbD-!9A)v$om@-EVV7mWC=70N5iw(9KnI}BR@jw#T3&1w)5!x?{WN`!>5j#mSD!9HBMaojo98)KPtqTl3dxFOvu|qTTK*WryH&h'
            '=&gl5dw9v7+!0s4rK@ajtnd2BGotf5d2-%uohhNDaH^%!bDRf!7CkdG3@rEp_n^6^nLmH*XioAdpVEpS)twn@lp|~QtkmQW_U$bz'
            'L3i8W$4zZ7b+*Nh*p^zb@w_E$!zoMwj?W};wk2shp))>K+AVGcAyMhhE1#jEeV2jPFsJumbTX7Lr(r`-Px}F7@Xmk4LDVev)XM0i'
            'UV?(E0nA%JT)60SmNN+(fa+8ft>2nxwTkvsvURh?I0$c0GEhcNfyAiO_!>7=9O1xz`Y3xhD}3=^q1JgDOmJpwiXpN!TB7kiO_{?v'
            'U;IYOyk5>R^_aJ?>WUeibLCD|nU88Qza>}YqD3Z7R?Ob@Irs_3LONr4Bg=RO@~*<z2Lp|yPUFoC{Rkz2YwN4mXwR37j|?K34T?n^'
            'hcbMGo#&W1NjH8E;~=~gO|(G3g0{SslAc5QHaWjv^gkd8ls+-Y{b@KU0kF^OQ+MpwjuKF8P&K_pfLB!P1#Sr{=<EoRM=Ec~Vag%;'
            'Kkr0-b{{7*A*T^UcVuZVk%58)p2N3Qpj$;wKU^&;HTMdffEmt6=(cS`#QYP{#Eb6*XzzE+kINkdMY;mO0&UGmI+*((ola3{motBg'
            'h#5EbW!&Df4Nl5|?EN682&;J35f7CF$@nx0M~@X#w0BY6DcH7INx^D1-1v$i3(#?35G!9pJ}%-L`8Qxj=zR>(77$AWiF{I45j#xM'
            'X@y|be^wf~l<2MwJN?sT0i@!1w~QS!&%1`r1~4fzEOINdJ8ZtujT2ve8;FH5C|=Zsli-!S)>Jp4=vX!P(8UTJ!RO;Z%$6K$^08B6'
            'N?K1AOMtTf{2J`%*b3_c76Q+%lg0dc=Q^YRO%@XlCa7{7J|{Vvq5!2=;&#dbbm$qVd`Xv4fZ<;h?KVcIjB}6%(R$>ki#Vc?qX>`^'
            '8X5lA((C#jG&9hD*omHEz`hYMrS2}d7BA-T1M!62h{E88tU*#O@|IaV_jCl<7w3;X^0Rk`{lB6MS)&-Z(@+i0hi4ZD!wZl{u~{Ar'
            '=5jEYZujQw^y1`@TS*jQ8=R}733S`x5PdcpS(fTY{o|jwuNC;CU-#aCG>osk8^pG;tAYoVaD^TW<L0+540uFm&KAPAB{KXU0C`K;'
            'p)zT=>9k-+=Riuj4R$4Lm<CSMYpzQUc7zaq+HEx8Ba5J02AORNe=z(9cutLdb!EqY8y6OaIO@X0p(Reg@^wb9v%1%?Sn6$Dq%;a^'
            '*YhYmH|N8kftLKkcw_k!?(=<~`OF-JAlBGt$QjSsHvs<Ec*Y_FJ6pobhCGw5u{Pye!oBbw-i{W~z9Dp0L~mbUpi}$rOAiD5q?mf='
            'O92Q)`KOMF^h~vbbYXUkZn;U6H?-R{PE|{kPLO`24vPURBnRIU!gim1zw7Sq?h@j5pY7&tN;s_JyjmRQ)zVR3Eg9q`zo5O9)Kb3x'
            '@4gh?;-BOegA}-&`(~^vNr{_K!b-SMtkhQn&R!LCzTf>iJH7WMR`g%L0N&q&eR-49R$*g8hmEOl|7V@=jfgcOBaW!v-)H}Xc)u1H'
            '_@87ubs>6m0U!f~qd6OAF=nqKB(Kw`#F1xT2J4GBvir3_Q4o9&FaFj)8Xoue5jEyPDSDA29m$za{CfJXg?_=Sr4f0c2T~~J8j*<^'
            'x+i;RghXjMFh%1EbUoY`DBCm_>`9L%9lU!S2eT{?1@C%CY(kIJ=>=J-El1s8gNElVha{lxbKZnKV-NO>J+Q<a^x>|NZCr|bXq1pr'
            'EtDYrN-h+a{L#y!VDyro#6LM4uJ(V-Dj72XJM*Z~&y(aWm>B`EPeY1?YXz&lU3oB{o6w@4`2YwVTM~cb5bAha(5}vh{dY(GljHv1'
            'Bo#8^WkC&%0bT-w#}ahFRS~#_vpDy&2%U4cao~N?>2$<D@PD__7B&l3Av#EWEIQ*?0G?98h-SQi^DYCmV-BRvgP|O;8Q|w$`$hZt'
            '-l!d98&XMypAzJIFAb2ta9`kWLkXFCW!zJGrkjp${kdn^k6w`<r-EQW=~Q5|IQ+p|B^0v2Fe)89TFbMSqqg_rzx{f6r(c6c0lMK$'
            ';@zRNZz&}$5ic-p47mo?Z?|-b&qvV|Y4`i~v)yMep9T|YstQgo6`Xc0%4D201uf&7jaCK0DuNL&7gi~7H}5|@iP*s_I1-BhO%+U`'
            'ZoCACg)>Hc@6Jn7Zxw))Al~)f_l^dYQ2d*p#BDYP(>V!JE2+@FG`FB<iw;IX3rn7CMGl7l_5XTsbb&^JrgBC0Ka9gAh0<c|t;!4J'
            'w6sso62rGdyIy{SiF`HbO>URmsNE9OL3GeTgetaZmm-l~iJ0v*GZGqBp6Hz)^~4Q^L4K^^d&u2kluzL6b%NLW$yczy&sfisOJNoc'
            'GX9PKY1HH?r~}7XM%j{sQ9;1_a<eAz|96^kE-L-A&-#&q!iKkgMZT*}T^b*mU<ydX;AGf8JMSN0LMyhM?h`S3B|ItCv(F&eVpZ~|'
            '5hcp;wDq0hCAEbW?{quTVnnfwM}zaBcB8i~N$O(Y)=S+ejy?iW<U1`+XygXQrETu!j#~i(+C$Nmt?Fgn@?i7Q<p2lsnhxpE>au7b'
            'yb^zgt)Mg?<<}E_Ete3auC^s}<f*G{&1)A!kt_^q42;K=N>zqwcD6+aIs&`iV_*-M8SGXjm_Z~XFs@0xrX4Bd%GK=fGO%Fi!<E<X'
            '&DXeV7-EVST5&3x@8R3Nc<yw>DLQ$oEOt{^Jvw-@>nEaWx}=b)=`s}4AW1Wd@x>6&;pzMHJb?jck(qnQSH&V&;7z`AubO&MiU;G1'
            '-v9s$9QNHF`kM5HgP$o)K-t#WsxTV1>I`PBl9=qs&-~>6baC{~{zOIB3X@f<Qe1Ta8YqypcYkQ_{eaiiA*d2T7-!3UQ6!c!-C|B>'
            'WT@|()3b|{i(gOAU@*|+V3eCA5HEILU=;Fl6(@Mf-iTs>ez}PK+VeGwVR!B+j4thn_o0vYuw0<a4~(^nm*}f9p?1(lDjqu};9|;x'
            '8NY?;<5e<}QxI1984Tsm(RXvj;x9!4BJO*_OJTyRKxFU_aCQ-glVyTaCIxF5Et7CLgZ{K6nRy`uay`KTDV#a%h@Z42B)6_n3Pew='
            'fY`%_J97ID%=?`Uu|#eCp}g;DuD5Emp0Ed~;rjZns9B{9w^X~=`qHSmR-%UvTf^S@JD0i{!Nj+5HjgPyf|k#UM@BMdj=xbkzc_-n'
            'rYS}$l2S8XJvpqrw;-!TR^M#23}-I6yI?vd`}=(CWW&24@qj!cAqOC2Yg(#Y6je*I8s;f2U<SSyEf>}8N~Sg4EXOQ0B-wrp)RE6~'
            'BZ7V{QK3~*+sdDj*r4Q+_kPlDqG_SUC(Y5;NI}-fj_8dr2Z!kX8GQ?EQzepnPS8PkGkiNZc{ezD;~w>Y#>6-45a^-mGRZJ_R@~3%'
            'xAox{wbvJ9QGHy4rC(=08&0V5EPSOIZ8)Xkk^2g>*=!=!hx(zuvvIvrclD#&Jgz4QU-&E4#N-Qqq>>PQ;crwCv1$a77a3L9Avam7'
            'M<`T+)Qy5*o{5(K#F+6&3E-HIR;O~7v5=Ddob~^5F*xfVx~%c7B|oN8SWgtnttwAyxmoe_lv`Azb-7&)W|kXPBUiayC4Q8fRc&#>'
            'WMr`^<0_(pOIhakwM@z~9u|;8`I@tG+@?ezV(eoO!i~&bN_p%+mhJu0`&Ime^W?;i9wywDdE3%kN`;EOIRi9eS%iV3A-Rw5qrFes'
            '#3^@?Vu&&yI5NG3a$03|98(U=eUPrb57Hfo)0CvwtOug&rA#c4QOZcban9CB>Mzb`9!LI4poX=N8e|5d>zJbQaCEW;&x1*u6Bx88'
            'zj&&g>tU-o#lbjEDVgJW|KRlGkaE=de_#4&XP}g7GKfK01M+l|R!Tuapm>R?0x24Rb5wJZI=rdK%rth&iY?<&R0dE?XcJJ<Z(Eq4'
            'tWZg+vc;uY5QgM%Y~=$l!?7#ViBrO`R37-_-d|miY2IH9IrYoA;r2r5jYO~L?$l+Tv@b)7Kv2erVT>JGpvKw!&%N(=cg64wy(T89'
            'a<2VGP_Ple=J`w_j>BtAm!Oa(s~t*Ghf0;o=2L+<omy(N_tJg-!;3Y9$ZAkYgRB-s@?*YKp!ul`8m97iK=S07WS&@fD@q+oCXUWJ'
            '0``Nu_s3`JNtfBAnrN9VD#?}6KrQEBKUdwL2PGWDl+wHyP7@2L#koe=OEHLas<}=*(UaeLlt!cyW@5;Blket~=?L@C{=?3shE@&3'
            '-Xn>Y40}X(W5F`GOfJF6-Slxn+ffv&Fn6MXO`x>Kbs`WGA|*Gmj*P2Kz+AL0Ri7oY*5TLDhNPl#u$7EYk><nUuz&oXwAvRELN->}'
            ';Ax4np{<zDf&%|=Vtj}UsM#a8$jYgVZe&4HWerkHC2o4?<Mogd8dPd<pe0Sd6J_w`$CW>csq3Ia4lY)Cx}r+g(a>sPlyAMIlg22S'
            'I(1Ch26j^pBsTX?s$x_1gDs`htK{(55=}>j@p2NAFCPh$OK+-V1lF0Ddr>e&-XTqf&%S)1OOB)`gV+7@p^MRCn5e+;#LZ<(SIww{'
            '>ERmX-oxRM`}XwW3~~F}i;l7)yVJlACrx0)EsAk(sUukkcR38kS({Cr)C0IpJX<d4$YcI`=H#hLO%x<%fN3hjL~s!CdT@3=RF-^T'
            '@j$eK@xGONUuTp<RK8}H{`7k1x&s3IoPNywReH@~pjnm#BQ)QVL~?zDq&Z8P`aYUc9EfcmWZSt0UWmzJQjDg{sDKKg9UHSpNHLL)'
            '-o_J^(_7Vs(Bw(n`RMvyicJ9+RJi){@Di%3sS(0Lpr<hXQYUz&({1RDaqMUaUTt~*bO;2Cf*|_1eCQ~iGV$(9k$pvDY9h%NXZs6-'
            'nJ>0$44&JXs!*PUu0kvL45&-^tieeLaAV-p);&ljBbqxY$V+g@cN)%Rd+$mo?BI3atXyrVRnZDOA3Lt&+YLiI>}))+fP>!2>B-=r'
            'cO(u@kKy1P?sGn3l5A^kw)l#Qh044V(T|Mpits;d$1@Bq<i6Yc;xd?|SMY!mQf#XqE(YdoKX%}pC`#jn$Ubob94(27<0)max-Da3'
            '&o?l1pU+qP?Y6!!yT6r+C&{X$a>@`ZS<8m*;W(SUy5tefLaL=Qozbyh2XM-kbCUf<F8A;}_RAJ}6zFwIT%41AjI1DMoo}sM6UkHD'
            'X>*eq$z&9CO{fh9mqUxv!Fy=LGZ2<!d_aTO01qx<(;Bo_4dVDs=<KLRuFSGzB!h8PUZ}_N;uJff<;iIBhw9Th=Zq3_oir08<1D~F'
            'IyFmRw}5;F`B#mV@cFP;fzKYdsshPtfo6Mp)k4s2rlBdrwH3<IbA+$8r&XQRjHUFxDve+#kgA9TEo1j*QIxfIO5?wL4<!AP!-0;B'
            'Qf<UJja+1)9EYKDL2vqj!XPe6a_EO!*2P-Ynvb*Tm!kN7b7DH;kXbvWp<y1=B8-w26cuD8#H3c&h3o!n`hF5G#K+vvNxB8L|19&!'
            '%onIyQFRbMX=ld8FVRZ$cG6<?GDS9{lQouih8%9LaU@i^k*XaJP6o%Y-gFL{6eoT5ZZN7n6Q?Ifzv_tHMG~NrEeSXX7g)hf6*Hll'
            'MCHvh!h&)B_qTD#gTaOYkGN539&=!hZ~gI1hx_9Bq|zeZYCFXQ%(A`-%4jsKni)oMV-OqFIpMtmS<9x;u<3x2_`lFnm6_<k-%V9s'
            'qO;7#T2UyWds#@zMeFV+jQ)v`#7&XcIx_iDF0l|edDT-jX>9d3bHcYdg)hPDvF<o-K)|<3tcDndUd5%F;gut^s^oqiU2!?Dnig*A'
            'yzIPjrMN}7M56=XoxAyHGAq^E5p$1_!eHiSq-I*-MB?ype6rV%P~BTvXa-XQ2{=W*xJMyPRztd@nsSW5bQam9s#QkgY{?OE5UODX'
            'x(E82aQ19zsnm9WZ7xEo14^+*2%LH(VWj4JksOZAkRzWH&hCI-6=Bx3csk{rt{}4C+;9q>WrShGAgs$sd%ZAMJqJKhD4>Bcfb>_E'
            '8KB%NKQEQwE;?(i2RUP?JHlK-c<LT?vK-TsNTn<TeaYBONBjc;)#Pb7X5!W=<HV&J@k+Ndzn;I=w%I(1rpjdL=3$|n(ULs5)WfTy'
            '3{#2_t#%Bn@0C^e%GUSdc+zSwvqG;z1Sj{g9u|9)15*u16knDgqGMFKz5KHzINI^5+}3(n>`~_~2fa6OrYlY}Ct)yviHU}cZTC=N'
            'vpiS+>#CI9RLUe4qO&cI9rFTd<7B@Uwuv+?;{nI3DBE7)BJmQ(x-xSDQ)Uj3AZ7uTBIuHubD1|Oo-~?9F-H`o9zi{gfv4QTU4Em1'
            '*wWK$Nz%}&ARmr3craDtmjs96DmqwAA(|AdmD*w2;^ew7uUmC}V2)_)DOFIx?_&Tq{V>%n+ARiZnJ6$c<h50v<dz;6@rm0qP&j`s'
            'Z%l#A@NBUG|0<3m!7HgyV?PIYIqB~bu@ULHm9L6Z8b`653riu55}s;?X0E!y;UBn@DSzHB5@Q60tNO7x1(x!RZy2Y`^)d=IzwZQT'
            'tg@t=QSe#l-U1!MwbYwED~8VT%}|W<hm}2)#AUEUesopeTuEfbKK$J5n2Vm6V&BMWQphG<%4}Ji2B%IGragX8iJ{G!3#~jKbYmG|'
            'IqIlkN}vtRT@^MLxVclki&Z8FGy6cb*a+?GURCwZ#Rs!v{?zDWBDW!+OF_YCs9TtlfQf$?BFAI^PaFNPb2ym#oRZefED}|@l@R;@'
            'DXKwao(nYcx#(NX)fl)^7+PAYDufyR^5bq7jw-frHK{8^0<QebkI<`C9{WJ7U=KsXX!sq)DTP?{H3>otS+1((Bg2*EAr`YhpHwVC'
            'mb{X#zJ**XRCsLU-GtC%j0J$oQGt>gA(N=EpX6ekoc3jDLy&A^S7i;dU)xaFO#iIbCT;qw=F?R_T7u0qmHL0HI&+WetJ58A%6uhv'
            'Sj&T}ZZlC=Q<6ZX%2v*wukiy&S1PkvZz6jW*SBV?zE}{bE0O>zTwTR8(R?Tjqw@dNYl=#EC@-cG!FNTo7&gOk<sb7e6`{s(FMq(Y'
            '>T<jv*U?%@E?uFaF~6SqDZZ8o13Zjsx9-K|1u5V9GupTy_5wXgZ=33g%m}XSdVrLQyY7SD7<<YUK&B2Y#Fj{qYRNz*M%IQsy2Q0P'
            'yR#-z2EL7PJZc7XF`oaCXX0WA*k8$|g4P<Tz<I(hudQ3?9F>b7IWe&pbLDKUPrX`NqGccJWFLcQ8rR#$cusly#HUJsP43q%o8Lag'
            'RHup}VC%N=6-WH3o7a$S5>UH@3rzsBqc7-UZ{mVb!>08rn>t|HGR#t-1Bs%b>kaH%pKiS4@nD<AD0UNaf1V9}18aPKN;VyH{bBly'
            '^}1Hcn)1=KP4eCt>rLlpJ@Kjj>$TjNohQ5su4U@nK$jdsTRf_x!y{>cPJApQ8&Bb9q^d5B{V>eE0~0I~oo4b%l8o2-@_4hdHe2;z'
            'KrB~R;-*Y&)Dv1$)MqQ_uV&s|18p6#Q=6i=)?Y)XE4FNA7K)PPk23ZmMaD%sYw|vic^|t}39v)1Y)$HnP7|E%(T~P4x+-)V%WT^I'
            'q2V|cBe*%dn@xViw9`;Sq=OFo0BJBfPhcwac?!;ka^EaYr)ySqsQgr!gYg)ACgJ}L#?{roO!J=&3Xlr%Uy=nIw?S5}@S<`&Qu7Uy'
            'Y?CLb_(qK5Bv~#p?u^GW@rg<n)#xX0CAYq;99ng)Rq-qbs&PULc{+z2UUxI-sH(Ax2nu`=Sm&59w57YP6Jys1<6q}mR?Fg2<f*?d'
            'b;?xrnEgz3CJMg(qnowj_1~JSKXvCR{UL$2p1yu9n%8F+C(1?f;Pm9AuRPn&b%Ee(%Bf2}-%N0w4rkRiuk_?S-YY6?yW&D9<oT`c'
            'VeK0}CV94DC-I`(pCBW1mTNmkm<|V$Xclh#$>xbVs4jw&`s08@B!dWk$(TuBx{vu2PH~zo=UChsnFXicM!)Bd&KLO4<T%1$QhpEh'
            'aH<hTY`}f4vx#LlCj3B@-nWgGMD|TlU6Xm!qLF)mlJ~6D&azlKW*Gr#sjf2Ds&AY3L8d)BcRfYl&ftkacjwCdED0D@)@g+4rH1WQ'
            'ZtKM_Vctyoptk(V`_y$CRTqw?Hmp&oj?PCOwYW#8a&ma9ri*?Tyg9NUTKe{l6zY;)UhL)W)mCvd8}Y8{?cycrLWe+fOTI?Myjsl?'
            'RH$J$w>QX!m}?6$vkXSQdg?wjN}lNGto~?sa5mi4@~NucX2Zb0<|02&Z4?1o%oo{8rAKx38xp$IUyf7st+K{^tgq2gA}bpq$d?C1'
            '9dY_S&RU=6uw?*DSF*!Zeg#)=u77KLLW%FxT4f;wcK80nmv{H>pPikaxo5p$-#r=}4~Fj9`(qTT=)93Fl<tA6+8vhKO;o#|21iHe'
            ')=yR4u^A8}4AnqIo$bx)?w17|*$wMYQ&#3>cB`J9oPtV&A!sD%vFl0<43z5k$TsKwv!DBClu(*7xX9}g`F1!s?gNdd6w$%lk1@-%'
            '?ALqUAO1@Hys1wibW>m}p?W_!`MGyAIFunB*e-}jMRY3?S92Fb?>HWup92uETLrh2?JrK=ot*xHVV3&VgUg@e3Z#b^S5(MiEc7Nr'
            'aFX#d$ixUVubH20r&O`nA1^T-aa*}{P&X`ZK0NE4oDX1DE`Rgg;N*}pD45V|2@6_|ISVdn2U_^}Hcv8wyamfjNq3P5!*6JRI$N)M'
            'M@K*P4&D{NN9U;u<62cu57kqR0xe^J%2}&F?Tu?~!D>|5lcT}AzWW{{Zq82fNm-S|dxMYO<!<|rTtwXct$p>dx7&JD?ul00=8u8%'
            '+XyQHTw=o?LE8NXqHmnYv-^*c?<Icnj}L7mH*ILxhBnaWxAfb~{q7a~^V|7%wPQCgfBe_0=8yX{?jmsc1C#}Z8J5#^F8}>se@lOB'
            'Y+rr1W{T;oZB#`!*W~B|)aM$jPYJ2Ha7A9MDv!%J!%6}Z4zTia-Yt!zbVexLvXxt4sK6zAVbWbKg=xv)qI4ByXeA$$6C`)XxTN#`'
            'aSsmD0pQcGpxeb*Er(cP?rlIG|1FGWZO79#h`~292ne)a5T>vcNC!A<aC3vfY^nkZ6-fd07?nKA;Ee%KGLa(%p)w{@HE9JLA`xdy'
            'As8|hWnW~J+FX_rn92g8f(eX_u1i5x>LbO<)V$QCw62&UKTU;xYWY<$l;Rl6N?LWZo3fBL@feJ3h;AG_v6v_kjxdH(q?mh&8z6&}'
            '8sRT-GSjIoW%cbHzN*`yR5a-_0VRgxae)*m{)K6?;_OyeSjh`u${-yhJcsOfF(&AuVjQX_7W68`xEPnt#O}+RQ8wtCG`@D6DvYli'
            '%BCuHL)NHnb%T}}#+LLOp2#9=WLBi18~CsxOP85OM}8}BQq4xJqatYr1|u3=WGZ*U!E&`;lCc*k{-SD|mK0=vLSlj%zEwGc{H<0W'
            '^Ls=D8GgyI>4fpUyko$bcIeNhqwiM<5;(6HyL&D1V)r>My|F;_P3gW7ARSPh>`0G=EtU~dVk+=}wg0U5zikx^hgd`aZ525ID~%`1'
            'zPM5dFyGf~E($c(GP+&L2Zf{MQcUqX_g*TD8ii|TEh%nZ&-&>}X?9wg*|R^=n5ciDf1T@pxb>E4RsmXQ`ozT88=}|`#YMhiubwE('
            '#>Ei7)WyIrSBZ^$K`L82m30Cu1kburCm<lLHXqX#O17SEb6mn)%w;dW-+hMv^Bn)@1>-P<7eqiQSzuicQ9x(qo?e10UYVgqzGAE3'
            'iyI&Xi&JXiT2L+yM)z@|>dnOWCg|sjZUDYN*$5M*)TY16OtLieh^u*)o&(do4Mss`q?KL|L16bXczqLy<c!05E+uy5`>-Cbsxl#}'
            '$j@lioO{U(^Zn(M<}%~zY1439^YJDXv-#pr`Ad5ErS5_pb5@)shOH&)ooa1(N8vo>DZ~!TC|u5HxP|ih(bua_p2pf-kJM_%((?rS'
            'QmaVu1Zc}WP_5gQnFZ2E2XneFR^_<iNiHBs4wS+VFJk@Z-U~>0@C3*+Xyrqx2Su|GvwcLED#THeCZ{n;d_dzB%7s|Mh2@SpT0v0z'
            '9eX6SQF%@j4k-;F3Qa5}DiP6)is$HKauLA<BIHXC%k`zNzD(mNwIkc&{aLtxNtf}Xaz`khihrV9hgnU`M@m*UalntrKi(K%;epUi'
            '*)#L@l8npptYWuOEU?2^WV{#K(nXsdHTE03jqSXXddUWZQWK4JARwf8O9}9HN%_R8wJ`|JEN4{5iocB`k}_o)*S18N3C6=Ft9Fl+'
            '*d5-*LqGIZtiDi1)<{Kv9U;17(t>6nORhU8K))Rh--~Nwplfjp1P|+ad3ceH6P@6gSS3t<pb#of&JY-94xdBYKgC@H5$99D!>szD'
            '9ryWO+7Q#kyO~q{w@Alr$OF*PVZL^%3z@?<_1Nj>7PI<TD97_o4cXF`;gm9$OK}a^rC#zZG^h}bWs6eCIn{UvsS6RjVZfKE8~*93'
            'KS6sx`?cMB4VrtSz^~z3auE5Q@`Y>3;9Ma3bMHfQw}T1i$;CvDa|y#<Ipuq2X+dr3@{TQ0Qj?4SQAJQ)ZyA8ci#Cu)FPgEyDkgbf'
            'R0i;u32$`;0xjhQ2q*i=_rTt>oKVPZF6>PHS!mypt?8Ns2dVmIxk4fP$kWNiJF#)>qgbiVvJS~NhElC-hciR?xvjhxM(YCJ9dtP9'
            'Of!*?{gFWXTjNK=;L`Z9`P<~X=8yZob>Ls;M<@NUVQ4Ej%0Oc=4PEy3BQ&9R9n|U<%{|957^~C^l8;ubeZS(IQ0fApJ()yAr>r*?'
            'Fcn8~>PAA|Y&xTAAkj~N(Z2&Y3b24DDFW;N07$GYD#KzUc;pD6lVnhS2Kl^5G}dB3#kEltOyn3&>B?iH16#}|Q}$%s=psU1>*Ebm'
            'h_%}>p-~rP|J{<^W=M1#gn-;~vqMTdbU|AUWnVIfZd1+$w&!$I(uWBaloU<Rk(CiJnU<0FN7k0-<YkyP1i6NnlIj>@+=2FbX^xQR'
            't%!H9gf&N(>y<b<J$-l1DLFhOxI31mQ_|j@e45f@<>KrJxE0Wd+%Bc_wj?Sqb(+NqWu02xV?wo8oRf&MTj(_Ig`5p+)__5)%rGRC'
            'l7r^7@qm?N=xBFb3Ve0l$9?et4<4&5O!Z+6oF8s|HC;I^GG!=7$zq{c!2WN){r0SX*u#wU@Ux*}e(PsK3$nkz=|}56$5x*Vbx`C*'
            'l_v7iH)MxSx*Rn_Z$6rU*u5{BbA9T?cjeC!?cf$^Q|?Q#(b~&y{CKrBi`L9JTx<f2FE01x@w@s8H+S#7G2Ym<QCUIVjMMm_qYc(6'
            'TB6{F(&zg_R%#G1QfT%knEzYe>b?B(w|c~J7Cb#So=)NG?26WuN%NN3cwal_s&s3b9$lQPwTX-2f#3o5Mf(q8vP6<24$p@z(f&ci'
            'nEdcQNR_U3fqtFn-b?<(ER%i|lye0$uqg9IixGgtRkRpV+2@1`fWyd~5t2)~`QjO1{5gV^#JgjvqqeuZzq?CQlb26>X=XG!iSICJ'
            'QX+Q$DBC$qIR^7Z=*w-%3(Uy6GlJCOw$wk-Obt`Et>k!9E`AB3NqnEmyiFgp5&Js(hzq@@{7snjX0j&543(zVfRe?j{|0{#;BVcE'
            '0T0NY?^?%8l?!!KwQn|hK%j%Mw{sMyZZEp=LqBae4*FR05KW7T%VBFpEO~waBbwZ_lL6L-T6S^7&W?Eg5*XPNZs%I!U4ckuF9V-Q'
            '!7U+~1d*|cPg~-7OYE7VgPZc!c4g!547YH2o_3orMVsGs?tOm-&(MeG|B63A(?2MWbcqwzm-hDbrzpu-{w3YM`eW*K;d?yP1P98{'
            'nx1>|H?5^~`1AI&c^sjuqWuv5e+^y4qKH_ArOpo@XMz?9V;<%@<ut5Eau(LJb(8qx`WZ-E0@8sY2O2PcGLb?2Nl%pMLi-1824D&X'
            '-$%oHCGo28KDR_GIM~}VJr<=QL2VvMvb-G^BOeo(;x%>)v#Ky9(A1kzR2q@miYO9|(C*{LLb}43<NDZ#T)oH-MO_?*^#!B-lW`bz'
            '9StH<?i9vjFSOcQhk%`%pW-eyE2bekbnqNAoL25L`VQy;3AmeXwR)AaE?^2QK4mI_9(yJf_fWIt{|@`88Nria-+Tvo1W6rn1E`m)'
            '0A)rYjpO+!09-M*u}m2EoKMPtqeln!+9A^9JWWTcCfcrD8e><TPdIXGLLw_`YLA+5r+ttE<_kd~0fFHhFbNV$#fT|6<5X>*N`{np'
            'sq)oF$F4XUFOvj~co?HH_9Ch9guX`!^=`dz+Q!diek3Ks9UcFoLsX2w5d58oi*y!VLJ0zAFsmm2k;<}c{s)YIjA!(xl%8QdUG9j2'
            '3sCs!H2A=A9Q+&BG;MUC-9t>~o<%TeI*$$Ay~FS%*F%Z~nJniZ)FU+Ei%|;Nkn5%6AW(Jgu{KVWb)S)9@Ea!oDuszKu+sb>%oi=e'
            '>s0O+?rYeEqj)?kdcXs~;}q+@#OM?w=!W4g@Wd$>Hu=Y?i)8~_oWRRNfiv{m)Jk4*N!j&H90UJ}DC@6E!_C{V4N5Z17P`^WlQ!n@'
            '(s4#hw8oNRhBd-j-U%jr>x`CN5E`Y(NR5tyXqgL*0K^1+G4S31pc43oj;k^i3)h8}X08h|L^x%*wAAFgdt~-l#9`QlU*ETQfowJJ'
            'zJy0+?iuLcF;R&f007LmH5~+s;ZIGi<!4&5FMLM+G(ID`LN}8w@|X5-?sOu_x4Aqj1}4{MOjlhg%JC(UK;KAXX640w;)$TJ5ehKF'
            'XSO6<X*4z#0w{}>U<r=;`#{FvxkNwUS@Z|up!)sG8AfI}^=8d*x>aFp#skpD`KM&G`UDd<2Cy0cY#Xpb{R?<90A)@r_Rpo=-LEC;'
            '`6$Z>_G#9XzKv!0R=8i=zU76bo}Sk%(b6i`Eb-4>jXl+ae8a&i@<x(^WK^9r%;k@-NZZ5LuLKbne^N6Nte!5YMU#@`A&tPa{Qw<`'
            'b)mFa%z309y6MK#rsAs7(DxTjeFPor@O&_6!?yrH7g?xd`BZ3>t=0JeF~?z?`c2i;*{Z>LZ5I7B6tF^&^EA^({lho?v#KQ1d{ao&'
            'RVJ54kJJO(^;KEfQ`1HIU~qPDF&Mf(!2x^+gAd=H_0Qj)9-#~I^U~XcQ+RXw%L!V)puyhv;#=`@SM!~};53ei4`|Nw9<$6~?EoNn'
            '{qy1AxQBHmkt^_hFk5+Kf{olno7dJjI>ii?s;jD2>!BB2Fc;nkS%b8#qQAnOzSdJ|c3JVJ>t)XChU13b2!og=;u`a0U}1>~2Nzs-'
            '#2{nZm$J13FE~N?p}5(I?#|WicvbK$oCqx0anG0Ssm_y41w8m36K%$ZYp}`+#~zSqrkQ(WHKzQO^nK#5B&e#{>TxO8Fo6m!BY;qO'
            'PRv8{U69IK3ZH}FVgg<+<16D;i?1t`U!h@>W5ItTPAjeWAosh|s4lRtE;72iE*^CaUuK?6o^tRoPc&~IM;&DkGec2Rnq&8$)K>Hm'
            '6`{=QgP_Zg3L}!OuPY}=uDZ7EDPJm88`b2aA^=&BiH=Rm9}o?tc+zC;4*zrQrumlhBuVtJgDmTF3~!ulid`+#VXO@Gp;Qid>?tEv'
            'Ikc(qYyYf&B!<1W$Agm&S~-_v`2R?({A`9XjFbxmt2NG-T!?X!r?n}^6Qk#zz|=PxCc?o&=%0}K#z=o-I}@?e?kZG6|Ah*ZBq-H|'
            '6Y4aIx$-_GDN!MJmBL6)VpfIRg|1v+QzrT<pX@vX)5Bx&jTbFpSXy;4*z_?cCxo~fvl<2~X{@=@WIsMT&OSdV8wcC-FlMklKPkrA'
            'yPGE|sQ=Htw&D*-kAj?hl_T(L`yg>p@Mo7Ok89FY%V5Q|cIz;}sS)QT+2V2Pn{;5mrNq%JAOWT>bT$moucee`)V{nVB6}v9OsQ~>'
            '5*5hEuvH~LYUIOuSUK|95}&FVt+1-f{$Hl+m*$DN(sM^9V_WsKT7@dNR@@B@$^Ks1K%-l>@~;;dKwE`aa8}#TX8=3`RoG#!UoNg='
            '^Mt3%%TVyLu{$K7bxlC(0NHYl8=ZtoiFH*MG;+j>NXzxyM(1iIuTg23a@lPXI_y@o=9lh0*)Z6AUr8?QuY(|ERCn6wL#1g=$P=!r'
            'sk{)VpGRIO+a+(znsGRd^4cb#1Z32pj!opPfVHk|G>+pCU>LdNn_AzFS*muvf2z3=`{z=jFPUM0q5$ibB|`{ZopX`2ze%HRZ8ETu'
            ')G=2T(pa+u$*uFWeiZ@9)ig0b99G(6#`(gbswiaNHR?7MUOMY~XX|=rPxYq!#r5BwRP?qp8C!j=SDZ!&@2^~AOv#0c<~kP@B6(KR'
            'CV#5@>HL%?Ajbx%0O}Q>CzZ7^yDibIT`_1K=+2}^t~|q6TjfZ49aT-cnoHx>{FxrhRg}3H>pun)$}NX<hg-Zpgf<l<R%VBj6vYw$'
            'f)^jFDuR2dTV2uZ$f~kUH3*=?Zq1NYhyAK#Xp#5H=A5f2v#o65qfIQD*lu0oIcnfk8Qo;p+AFW>&!}EcUVIe=MotnZyE`Sngg<x1'
            '%iTIRUxs%T(5_gGu9RUm+G5Gq%DY^D^49Uk@;K||$uF_K&s*E?SywqN_NT7F5C{{$Jq`If@b_8-$Rtc8k<*w~NQEs?y<p|!xm%gV'
            'RGOQ9!FU%ogdJa;52<{^$VY!})B?~tMeCq!VzQE@G=AZKr;#ommr^7PWhmF~{{@R0sBb^_&vhX#S(qiKDkvtQ;Tj31ra_oLg(a08'
            'm1zKPIa(^bMpp18>z`rKR(F+Unb#1p%d&1g6~|jR8Qx8%P4bs-)=*3x$ixPbYzg}@37#w|)GSJ6<QadLVBV@U7zYA)$gEM!;Ksp4'
            '7B!}cmNZ;y_z5>Pd}Civh-o~h?vjUlZ$yi&kI8<YM2)o?c`yDX2dMv&&XD^+8~G@jqaX|5xIEB;RBGH$qj;sepdw2z$e`6;-kJ{='
            'S{u2I7Lx+wEy{Kz23gZskgeh=nE==ow5)cgZ*ZFH%)5nCw=X1abGX$c4wsZ*m`2~O8h!ifapwUBe?&(Y1H2#yuUpK#axwGui_z4b'
            ')-&H|Jri#h%w+gj@Z^H{3vx|!T+^|!rfl7svdT4Oa!u7_oL>F*yP4g0b;JJ|zyB#<;v(%9u>SKQLr4rQNLY~y+7J|Zi}W_mHZ*U1'
            '0_m={j+e4KeUdr6-Z=3j{?&cx30uS`ZtGL?Wvj*dlV3YiTOZqhT4Q^vs!yGs{8U|^It}@$%06|PVzFPr{Ghv8&&yvc7*O)BSud_i'
            'cNEH8R8r(hc6$~l<23u7@?_DzNS2doS@K(u^JbjettIkQWfjSlYlN96gwd9;%{J@OlNMrb<?W}uZFjR<OnM8j?$+M1*Nm%1&^v{L'
            'ue*x>eUzzQsC=4>eb*IxTc1ty*@Cj$BKP?k`cGDq4W`6WLgjocv*bGc?w`o!gYvM2h}GKP8haxzR24F`UpDPOoh>WmVOIUElAX8!'
            '?t-~*ot?F;AZvNFMAz_!s)3H5E-F@ydTqYcPaA4$rfAc_q7|ve@@5WUnZygwfCc5`iL|lJlesx-Hd`9b*P*ffX#ii<H|$S*BM!MX'
            'B6&}dHYt<fV^RfSq*IalT)mUX=0>{;S9UZzZm=fIM_ZMsZ~ya<v4qAZHa^uE1o;xl|0E<Ow65<^MpLrvnJ(qhsMoDx98NVnDSEOB'
            'OLWGg8mn$m_0y>)#b_BWxu^-IpL*fEf+uK6!FVwE`ot3|-|pP|*m%mijqX*hf|fr1wQ{+wy?>HiUb`_l(b?LRc<AYy7v5Eqma<Aq'
            '8l|Lk!>%EtmGX#cG%8p-C9lKLW&RhC!6<t<J8efgT2D@vPA7iDFpO#My2$Xn(@Fl~NIP}2^PS>{<Qk_u-CCB10UOKY3RG5+G?>V0'
            '_w4uvbHdv8@?L%fv?S#kr(Q+Dh<1AKEyz#YqeOt!#xs>snoWDMn2lU7#(XcDr8Ese>KpOw6<$QRcYx1i8^b((NeEai0Hl$)0gZDu'
            '_Ql6IK_kOtnFyVqd)pT?Z#BV$=t1HXU4inGhY3x}WeAI!ASGcc2Kp9@9I7RjS}__?JN~Jxqm?b`BDTgsp`t#fl^vd4u2=^T6u6RC'
            'A|>Z50%D49CR8F92!a2xV-+gYmI)6XCsRs0uJ1C<vP|R~!$nHV_n)>Ul*i(%>NM*9Q}?HxMZ9Qo<6@$ajXhdaPG{QTlIBlORrv&_'
            'NYZ<k)gAU>**BeNH`RWgsG;x$RUR}WvO}RRv_r1T8`YMlbU~DBh0XW6_yXl%P?NOczB_sHCrQ!eb4yb!rCmobb>o=@qx@&Swra?9'
            's~?x(WdCVa-av0u2`2q5IdO6|DrC!M&d1?>YbACKXX~y6V06v|H}N^Mp)`U~pB52Vu?Rh?dkcM=d6oL}!*?>jgwrunh;p1x$tYe-'
            '_F;Rn$*&bs<M7QqY#hDbGE!wCd#0CYhpZ^$&P`<6gqgpxb6M@{3@P%vW(#r-i{ig~`=bAc=X)|}*$*kR3f0H~^y6)HiT&YK#u$sn'
            ';ax&ne8X3#*2Ar0()L>nhRV|Npj0<U=G?R1s$grm;pC%4lcc4|OOpIz2dNpa;nBpN2cQRcoEoYHKZ3{@Q57D>jk+`A86FiokXWY>'
            '-okgv7E#J~aF8s4ssHDT_W0IIQVQ>4WK(0bn5&aWqGQA8rfvBCSlleVWMZNk=WeONv|Y0tlYwg3shF@w=DIM#?6_4q%&t(_=0kI@'
            '1*bET>)qKAFPyKC0ppB{YH$T#PFwGPAjLxGtWU*r@vmPaaXneK>BEZUQ0M{G@W?p}5>G97Vv<3?Ukn#WK#EIsf*!LBPc4sNS||H('
            '`aG_4;i?3G7W=3l2s4s!N69lZ+sbQs{Z&=pkwtEY!~XI6;rVCt!;P33>Az;e3m~)0Nsy6>ENcg0Mm#zK$n%T}LioD7jS$}E3BmU$'
            '9)v_B3{%0~!Mp-tM#&99eA7i>zEh0E7)nX}+cXrNylV*)D%-+<IsF+lyuo4roGW2pzfw8xRKCYbKNPH;Mu|U`BUom(<T~WBrD%+A'
            '<6unrNmi-2h1I4iQ1lnnNv<m~pesuw=|~u)0vU+alz<q+U@-AG_2Cb4mJ13l71EAIA;KIjg9-JqMh<|Qvr5w?eknONRojsTK8=z#'
            'X7;#(elW${kU=!}S3u<C&^H+N7VL`$9PVEdyrcDJAyz$tXR=DMF}m~UG#s<csG31jowJ65rJcyGG?g2FjB>?CA-9*tr~QQw<H&`{'
            'NG$t@$Z-Ux!ZF0^Y-4by{!PV$toP>Yyvk680H9hGDYG%qrTTrfWx)e&d>RBM8<17mQ4m1MUk`;LJdRUV17-Wl3&V|Stw<HDRaF?C'
            '%01EJLL5WxORG!2Q`@|va}>L3eXd$Wl>_Pe3eQ@{X_WTgqyng}t~>wRgtw2|R~11o5-d?L4_RpoUkP10mkApd1e*t3NqX*m?34#_'
            's=QZi>2B@T8fS-zl|<QMwc)O8+5gMM;H-b>_New6cd<&~$x>OLP*2NzYn0WZQMsHJT?Sag*OuJU?W~ICWS`1vCS=z>r>|dXFff0R'
            '$cD%23KdYDr%bENK~!}lKA}Q1kzD>vUXZJzB$39~VlC-HC9P60bA1Wvy2{g)Wu-rPo=+E`-eBe%*QWk-FjiNw-egiWxQ6z(;;yf0'
            'TK4E@N~~82Q%yXxDrWz#b*m|XI8dn0Mzm-X0LWmVH7UEuR;W!0MIXj|Ue$u4Q@$!76K>g6_)}KSKOUW}C*%L=^p~r}nW|E?$X%)}'
            '$ffD__pY`oo+`}<?`F$lxZ_-HZB_DCi>Gc`_D#iGw=9nk<Jb`#tS4sM5<k|zi>>e-(a5a|&MbeK?=+j172FvVoOuyAw|P7orG7cH'
            'ZWKG)+`Dlbj7Pf8U=@0cCZ6*hNt$$O8!-b?N<pa3s{+ebV~)L>M6+Zpyt!52#g9_eXS=Q}(ug@6np!1QmE}umS-+*^wk}rm^y~uL'
            'l{v(8#Zq!L&b8c;7e&(UH1|>reb+JcTmw}q4F`bnXc&}$8XPi7TErmUtSAej&lykKZJiB(Gqx5I#!G~5P(&$bz|@O1c(vxPWOHxU'
            'RfUwkV@9APPX%NOkc`JkFtWnH>1DH6v-IhJ&@~xV86j2eGrJ^wZvSo!Nv(g-OTFC2v$%cr_?vN)HEobV0@@J_3h(j{LxbX?3T=CQ'
            'gQUwbAsX=|Vc~n;y&x14%LN9H!pPf9s3zMoGd80buX9_CvMm$O>M{#8B^`^X*vEW9m^ha#LzG1l#9~?T;eZWtXQ~<(6@M}-F6H)6'
            'X;^Z}_@+Ncm193-xn75Vl?+(;kg?_#wv8?~axqOC94Vn}Y53e8sSwmo1+Qul$)55b-HS>faitH3N){wp#Xw}y4}7xF$`P=oy*dL?'
            'scl7xZd#sVO4P_g&xSmH+^)QnY{%DIX%+t{L#E*AqK89+%s6xH0)-_6Dca7@i~s@ZQ3@20_uDda4V!9+^JpAV@4AKEFeWXH&Qsr?'
            'AUCE>p>EpeT49nLtei%@I7Y|p-+hq%<Z=0(7{-IT68SM{Qw4%e(fD<%7eMs0;dGXO21&CJKb8-N#N4jpVZOC0c|4UQui4<zGiXA2'
            'e|GwI@Y7(}Kg7R&IyyagcLcfvmqR!h411^>mckP>Y^}9)AB-tIHfmCvyRsU*QFC`1+?Zz7^8KAgc}|S6LHYA^1-x?ZB7?vJY(Qa+'
            'Z0S}#-VFxhqIw|h+I+ml+10vvvPjeZ`4cXaE0kaJ8n5{#3HR!{rt<K$u9qGHXT&$avAL8VUYSz5KzxklH+Mu9FWiOvcLx7zEaS_P'
            'j(9KAqs7x{Fa}lxl$>sypNpF%G{p2K=7{NSybLE~l0@5JkfrKwNbF$}IOV&5Yog4|0-bk!;Z1ZY5P`YXQO1bpL6%p(2~wHdkR-Ev'
            '7(a{A=ou$+K=_=Vl1B;mBmtRvbC4)7w+GA}Jqn_6xSU{Yo+Y+C9W!s*B6vSobi*5@$Gj9n0XuTM=RTJe1)b56MG5@Yo4n*e2^%>('
            'jSk4ARm$Md#tkmiFz9_Fc<HiOqTIvY`++KSl={GakYMq5FlmwhirSiAGz`|?01}4i<U;xXDLEMtNi>?PP%a`RFq~&Nkti6H*TVtE'
            'rN^8t6vRzDG>9OiGtvWcEn*v@?O;UOC^RU8UJt+z=$j7>dsxsY=wH311=?Y%xU>mu{&E3dE)IqlXZ=H@t@+L<y9t$TFHcF2J`d&~'
            'Tdfw-4a&os2V84;hm(ud_T+grI0oH%rk0S?ACo)CiN85R$1|9=v?n|{^%^VrgW82hWw1Y|Iw$$pff&I4&O*SyRh)DTIY)00P)9xw'
            'mk%n<8dpo9BsY&&YDGjii_q+F_KzmLv?L*ZuNVlqTe_586d5`Yx2DvvT}%b~p=COmx7a(=oklItrBc63lx*xRx<BBLB)Tb7c`}PR'
            '-J?QPtbC-)K0&uvd4_tt*?}h{^0Roub7L2?|G@OCFd53=&$m<8Fo|pPR@azywIPNI&7cJpghE-HCGP^F5oP17G8@C*b?JYB1SWv='
            'GW~sVOqdV@Nqj4s&&0Ri3Jb1|gU@$6-;3{Xunv!e+1kZ3eRO%eKMNyDT5Uj+-W6)tz!P<!PCBn&d)rFR=(vp_5R60gD3vUo|8S=s'
            'rOZ_i3;K9HgV!pbk`TV!mF}0j)BQS^x=(vK>0E8~x#`Rq=FYx|nZUNyP|xynD50TF-kK51F2ZuOPYn*K3lOOr&x{jUGo*A^a+Md_'
            'r#g2f%P7_N+$c*?ZeSF<*a}0G11ZmR8Ojo#(x4;7#A|sbntTLWhLIH608Iid1d6X*<CM{u#o`zon59ug83%hMu8l2knw<?kjpI@Y'
            'y~#Cf`-<5K&#1EC@imtuE0p0rMWGf>lKNYgy+tkvA}327KVhwBEaDSH=x~Mi=v*c~NI6|3USy{-Q{@P-4@-apR+H6TB|(<CCw(ji'
            'IWz2KXz5U7sx5l*TnTB>2Z>;CFg*HIocI57!P&~-4-^^B`XDHroGXEhN*M?l>5&UdJrR0b#UABYm@mrs9<G56aZKo0`10?9c*(c`'
            'D#0>{Q10hQiI(9N#_8tCRf#h<vL$coMO4H{NzN?7WnLKHboBDD@+9Ww!@VQPyHK`JaVF+e$fB5kwYc&mC%MlHO!AP2dPopMvCaXX'
            '@>g=vq>@f5A|X_szRI(uvuoCVSXPzXVXsHK1Sn33Y!>lC1aq>Dp%z7M*iv35Q(#RR4OJYpCBdD@qVuviCvHFdp>z|b;rYZ~?C$Nn'
            '*nO_OQ*tgu#a^UwPP=&B15xvl47{TGrshB|w3|yEEL-}TLzSr^Sn10|7E$sszyI(-kEwQ>Rs{nUvfeUoo(Ln5LulG=Oj8}%RG)fP'
            'Obfn26~geGpY*kx{GBfWDbLP%@3?P!qRMSA!&X+fSN10O!1?g>{dq?wS|&@XFHi4|F6Ro68LcpiU-r+(BbvcVs}8i0+3+$ri`#ts'
            'wp(&V#H`fErrs!_qThv*VelP4$=oP*s}oI1`;KCTs57abh!Q##^HwfA=sUOSYdW_38vf7f6FL`;k+ym(uYspL>PlXO`KxZF|C^od'
            'N9%80b^yr*CH>2gmeh}y5{&pCo5fZ;K2<(#Rft91@}t7|pX9%6gc(Y~6`7M2icoWTr%({@xEO+s!qr|m=UT;mopRnU-M|>F)Ll$z'
            'AS~n?tskbobAJ0*)mgo*=&n|-H=yA8)8L^f&m%P`!$Iw^i+-{2nhkCh?`!hEHTck00N9Xd%5R@YKW7BF^0^-Qn9Tj@c3fk&)C>iL'
            'XuLvE6w~N+b}R2!nGmhJLoIHN^;%k(&Dm+MerrmPTh<NW(->pkp3@uC5{}2@Qmuk)BA`q!$q>mgya3;t=4PDS(@QJd7Z+|Kr7eSw'
            'TYxu`uf)^0zomJZ8+FCk{n4aZsCmE?w<VreWT8u`QfnzS54?!OJM?uY0UobH<Rke+OZK*sYwEXFku+XzeiM1j>GCn00b2a-{V^~H'
            'yKwZQk<IsbaDF~Gd9%sBZc^d&t8N5c2gYeY9ut=P7f9f9GQKwntDPrOv*vDAx}9>5`4SE@L7p+JcniZYDU;hP?LJd>maJk+qRPuA'
            'o|WltH3?}@s6x1M<z*WA=3)bQr>0IfXJMGSaMmgkwAQ&Om7`PbSf=Xu)Iy>rJ51Ff$JvUwcsX)S5N)pfif>3!13-@SrRJ9XZFyR%'
            'xy4mo(%mO|>9y;GI|WB3CNBjGjFmMY7sIPx>m9NlD=Q#dAPOP>)=L=|ZN&9{a`hWy4y6!tHeF-p4Q2Q!5YUnZq#{I|)rW5F&3CbA'
            'xZYHHUz7Zi?CIw;(it~oySnUE9+Ax_W;Fyhx0qHIj6llYvZP;h<S5RznAWUB`PF}Nr>F=?FkO}CaCypeKcx`2PbS{>8MY%hx9ZKN'
            'Eju((-MQw`iFRD#=gauYkk_>hdgY;wb7o+MoeO~P*6VVFeg?(<vXr};f)gZjqAhp2NVD@ls>gkF)~=doO8H>qT-P=?MH4iT`S-5!'
            'b8M{bSP)Y1#|9rVg_CAI*xp?H+&?)Qyb~bw91ixibuaU80vK@uL?sJ4;?45TkHTO^@ksC?ys`Dtxi{mG_(tsqrVOT?R8`0jha_eU'
            '&IU)v{lfuTASM(tBRA=>daU(ZQLUe>XEE!>`&_ZCGH*;SqE#KVJS$A^V)(XID}y@!dj!{?DXCWeKRimTn$0k3W+o=_O|qn<5w{+u'
            'Kfrt_WK4DTOF2<w&-zo;(8njmfE-M>ijfrp^geHf6>K_dkD%$jdFzJ@=C#|Bae>CT?L3O!3jJ?`-+2->9_sotS9y)H#{bkqthwb+'
            'gQFujtnObfPKUk!87H>B_a~iXnUJt3uVc^B!!;w6^Z_(f7N7lBv-5uz(vz2_LPP0}yjBu-{>tOvp**5!jwERf!I2VxkCmuG@{a*z'
            'g0tYM?pO+o>MZ2xUDgaYS$_1xELdP-?8M7J?OOQ}R(%C(7Ws22P=F|51RAaxF7s!Jm*5mwUb*p!GUmpwwdC-BDczS4SI+xqKljg+'
            'z%m>h_fIc|T#2fzwEwBbouYCo<?fZ5GyYfQJXIzKtev`rT%A+{0X4(T%W&tuSLp5m|6dKuvbm?0Vc4kv{}+0n2ED<_&%L9;p$nMz'
            '{$fbbt1t=uN$l;{e5us77FPOhT70w7O)wQcr{h~*hh<}GRhyi&z+A~G=wf@TkiJp6Y}lFirHP|9r%);W!n-dN#@oouQEB3kHKaf9'
            'g3?++A>q<yCJyJn+`L><K&*U0m5W*O&y8iN&zGS-M|OIwhT#*kl4>R|Ew8Ay5ZVVTTUl3_#J;@y%7zumZS90omi$z_lV$cwCe}JU'
            'Kt8!wIhuWcOAC@$E$ACs3Xn41l_`&QkLrp`PkNFC(K`IR+;8t)shdgf5^?S7YO78fDBdNLSn^5*Ki1@uc<?T_X)m|09zihj=%!&i'
            'M_kYl8~I$-63P}ux}1%pu|EqKH_-83-(jL`*4TPVoQCCXZ6*>o8e3mOzWNe!RgFCL$yDg~YkW5ODKi^`7}<<zYa?y%Gbz%iQlrQ6'
            '<>~L)<{GuE3?0)=re5pYL~ENivgO>mz?ZdyQ!fcHpHK6_pikQ+4B5VN9;Lcug@8B4!7K~J#jl4KXBS7uy~9DLVdNqFJ;+=@;xC>L'
            'E%&Z%OFuh#K|~?M<X-!M{ZXYG-Fg4GcQPCtoPU-4Xg)^^b7@shiAXNP6xM4bG{TbhK+;mfUH0G3PfuhP#T_q6ycIdH##4$YRq@5r'
            '6E}_LRI4Zz_ld{8aY4ppHBKq1h9|DCuW=S)ibcDfjyS^`xl~lTBda2}vdR^E?unZvy6+nC806L>SEIA!%D>Bba^)3Z)%Ti>QJiv7'
            'Y#B5oBSed%v0nm}9mimx4#&<klLoA=`Qtt=(E0JV^xJ0VyB{5R2850C^+DTn?T$kOlydyh(+&pnESh_c^+i?t4`V=@0dcK>6jz$t'
            '6b^N>@qn?n#^09>hb|}s3|VXMIeDT<_(QuH^1#?&LqHLbGLO+?6vj78^x2V6^I6dMKJdg6p2?%0nS5}CQ~OQ|CKzl4-Tu@q7Assk'
            '4de4OZ{bCAZyuy-?`#JiI`Bp%ZJfy^e+xc*WqIe+-^#9=&6KC(trD_;OR)fi4j2Iz&r$kDgsr|aR^Ih#lcmX4o+yNqVs@JVy78?)'
            'p5af&$c)Mz4eBC#uZI|gN`YxRlG5a_rC0sA<>1f0!8CdFc)(2groTVE-WS&o4NL?<UVx3p<28%Esmwnrqx7IZtUU}dPiBbcR9y4C'
            'uPG%r4E*;cott<{9h;GdKHMO8O~_AILcn3)=QwoWD?o=zsqP9nGZowl`B~TRmdYf@E06pX{x4_=EK!g5b{Hh6rqgj<oT%$QO60OK'
            'mlk%*ivHt9GUks*s|xX}o1@IdEzmqE{8<`c@(UXCUy?^7vhNN<1hqPHNiD9-d2j<GzmZvUonCTgJLx%}aR)_yoX?&!P}K5_5qgD<'
            'NANJF7!<MScM{KFiHH`>62jD%T(9)3Sib@<g0qX>Co_ybi9@mdfYW{4raLbIJgNFz7t&{yij92QN3{S2pQ6v(Sy80sDT6JEwaar('
            '{wz-u5R(SgMG%!JXa&)#k}k^bFZzkInPsi80yTXL1Zpy0L4xZgS%6NQWFN!j$vn1VZ}%(Zu)X!3egBfdrIQoulrK*u6NahXhQX-A'
            'qclgDsp8iEFbQrznP@s|Qw=vMY;&~A{M7lZoWmG{lVn&&D<#udRsPI1S&d010Hy#q*K91aY5NBo(F<!M;8=^Z05}EZrHMgTHUPOX'
            'e627Ve~BXa$bArdV(PLH9Bd#Qw?Ga<-_A`#d0voS1)!ESIfwVYY}o+~Y0`Z8{rAsb!imFeut@+<Lq$;2NOqttzb~&&X5)}I+Y&jO'
            'D!>5M7?t7b+m^7RaTnxaopIfbbw`iQl`_LcRnajSVWXNnnUO;_XGm8L$ZF;01<r&NgL7T9a#)>h$xW9dOmG2A*G>tt77UcR!o6Jr'
            'fpfhD>Z_P4%8dV&f$?U#din8n0i|AvrXV_TOeCw%ZIx&wkFMk>NKn1dim!uAl@h%}7{CE`JBZplXhLkubYjvO7g;(dC*cT7$|jHH'
            'QI^?loCF_PaJ&V~MN7A5_y)o}F3yfxth%8`r?0sx6M7cnC{8lZjK=<_C^|$pDJKo_LOd8NagW~fGJ=)^Q7e5EuN}E?D&{g>MjYwH'
            'g;<1-wNJNsXE47Od<?A$6V7Gf(wDYhQLc=bmqr#S7FB7kDu(x(Jfhu2u<$WmTu1G;{9&9l=em%8O*yp6Fto0UM?=x7<Oioy=gG5b'
            '21zJK=UL-vwn(m}YX(9ir8M6-Mq&+Xrkt8t0<Xczn=H_+P19`Kx+<xXu{As8eG$MsGJo!Wd<9#932Z=JD&_qHL~du^2t=YlfDSdE'
            'c%oUmHx3G97(`ME4bd~Q!oUo}V;PkCB?GfQAgtV#-<Sbbpja;AAj-(FsOYfqK7Ux(qB>-)Ci<*ArOypOs~br*+cXxfZk$gHN~;;G'
            '+_Yl6iWs(v15|9}tch_W1*ANN-OO>;M!~J=u41ODV$2F>d0nWP`8VYgsi7M6;b*`GJcdbQKd(cMMU<?H;ME0{3Ndr#ubb~%7*pr0'
            'i|;G<FXqYHB>JyBN=;K|gBZc52jiCJSeA$-1tZ)nL-x0tu<6cxQjRIE45%!B!xF}*6sG;4IIK<S%^BKXaE!6m&e17EcNPRReXz=}'
            'tfQ3ppi>6uo&?+O!e@dpZtd%{e%~Dq`e*K8Z`i|1e}jX*(XTqr8%<f-&1ub^tIA;HY&P+4{DiJJxIWmh_x5;jayU3^sls3@Kbrw1'
            'Pi2z%39=>?yQx&8YAG`h*O9F2vSGL2WLh@6)XZF6ljV=NP@24p*AKmNz%31g%S<cUmypTs`&g4MO0A(jVU%j|6%_dQe$viaDvFMA'
            'ga%CzzUKhYNvak47AkUcsn|o$z9EnsTv<>BH_ZBS`pXHKuO_nI9r5@M$g2JdvpV2mNqm&mVK8!+AySdpsVbDO4TY$M-Wan?;4AH&'
            '#7Q$yc2}$chNBcK^F`=$ZPKq|iJ(*NqR1ihzH!ny(kyH9HExvpajjLVHs@oW*{oZwynXvE1K6G2d&x~I#aCHlxm!~JiWh|!5Y_L?'
            'D|KqK50=d`<tE+oGo4z`$!lw($-OR3kd=rH4<wGt>o84^n`bfXR%9_O3_oo=l*fDgE&45b;4eQGQ|4jFUUlx!nv@j0!Xg3{{<b&0'
            '0u9TLst~I5#2z=tH2tep8E($$?r57jH9120R<xV1P(ih;ezYXSy8MsC0%g?1JN75cy0%fY;H=YJmr)S~LVGMhmaXd1SRjzq>a5_$'
            '{i;AmX^giHrBz?D+C>g;NsX9lRa&wx-5h6{vs0g`->ni&P{k>iks9eX&gSu)eKJs6mlQ!|v;9_f+$2HRG>Cw)z<_NYQSgYO$#v0s'
            'twK)CE($oyxiWH``C~u0^HU+?vw;vK@VE4sr8_rFaX5+ZQ?Vn`F(4X{Esq@H(=o`ask$jgo}+?qQ8kK1GxcEtM@Of>xM#gzTntU?'
            'xx-Ue%S(j>trfdfKorbV+2D1;Z@JeM-58m$^;VwISoXuJBB*Dr;I%}n#lCf+r&MbG3@``O0r_?cok<p#`rVa-?r2g*bS&I4I9a2('
            '-3^1UeOrqp41Gn(Ff-8a#S*!n`G^`z@~rF|Zi2jy{nR~>YnlfUU}JQX-CCP~(8Y|{atG8mF!#0gMsTn&_=~4uUV`3cKp!o`nJWXH'
            'WpdWv0ruPA=C*wohs(Kd=D)%ma|-l7l@DkT0fh>cD3PFVLK5E#dsfgb$U=XIOC*q8l#H;McVldD%Q(lJz%TT@Xt@9?Dz#Vb#Vp$!'
            ')1<p;;174`4;)9Ih$8h{Qvu`6Hs>x#JfQr*!IDMdXTNuM;K9M!U`P+`LD9*G9M192l$;R+eca6Nq&Co@ym881h~(+C)EiT#l9@L*'
            ';+Ei2wY<#$!x95kkC*;HHdI&~$<xxSq?X@*BE+h){_=eRKan6gP~7Drm7p*a&dKD|d2gz_BL*m#Ol0OePn;hE=Md3yJ_4}fcEk&r'
            'fl@6(*DAsC=iZ71nq-c;3?&}-uum|iw2I3l;c|;XB(vb%OYv*8%*a!WoliWY9wSZU*LXQxj(j{WNiZf!MFL@n2Q#JAX_)pmP<h_Y'
            'xiIg=*-^SP4tN(+Ql8{xwa>RAGxQXY)6v@MQ~XeihJY9Ob*D}^W@1wbCn{0N^k&q9yIen*eKOS*d+BY{vaOm9=T6<Ye_s-yYQ~js'
            'Hy+o<X?C5r9$Bpw7~uV#?_~u?(#9Kwj$VQR#^=4m{y|@CkFeBwv;?%kTdP}dMCO+XAVjc@SI<i{yXdUfiRD9T;m(Y#HWW5!BhW|q'
            'Ttpaz0eg{H!7nVf**`e$|E=J4YI}~F?v)k0WBNVd;SP1#Z0&2A8n{aABvme!*Etj~uR8hYifyPW1h3i0$V1P|X^f|w5c)=6DSCmp'
            'exte{tKMRVZ{YNs1H6v@#UEV@^nFT2^E&?~7dL@v-20=>G6{Ah=ZoiF5M4_lQdJwdyvnn2YcJ6;-C<H%&OBAQT1v^m%Y<C5*MqD#'
            'BcTm^)Au7;F%XLnI_qnNSROBH>xF1OtxKSog+j_3(<l}1RB!gzS9r7B3*@JC4R-TOydBjgZ(xUviDE<O3J(hfZ(tMiX7Czo&m=*)'
            'Dh^H`+rpRdA52{jVBUYamK$fO^rMb9nKYrRJ+j=iH6cnkn|=IdRmcxwAH21u&krlyiyDlk#rE>4QyCg0QA|#=_M~fix%Q-E<*;k`'
            'sLZmjdR-e8csR*3a>q1XqAz0=Z<6SaUeJnEf(#eTsaEiaa=(scN?^>WN|~4MJ)kl4SwAmME+@cOYnLFPTy0RkY4ZbvL*OS?>uWew'
            'ogBv10-&l99gvehBHb(1qDuQOUnpFoPS!5V2FeUAhQo6uM;$u*C<*o7_b{YYYTeodD+P9(0PYicG?8OoBBVw2P5*dsGH_u;_vh2W'
            'LH}H{yvE=B#qBDg@;=M)4F5g2^%9ad8}INM3jTfVM@c|!j{J#-|DFQB!+(3RugmN<4ght6B)N^LQ*W8Y5&!IiHXV8s?lSNee#Sp9'
            '<|9~fNUg)0z#n;Ps6*Id&{U~2a6!<SWg|ZdMpSUfOFj4tG$xvs2Q4OkKplF3WJ%;DYOG#J(NXy79P!VimAvuB$?^lQJH#Fd=vuOS'
            'kdRNp-+*UFK7MmK3e;#nV|jyl-~lJ6-sgZa<J%E`8vh7E@$%^t=gZKWf`HN5a<)|BeZ2)3<U;VZULEyL-dyzF(Az9&56>D~-|UOS'
            'c)Vm=yONm^H`s)@FP^;+v$r2xV((>p6p+CdGf;c?;{Cy~Xu|I?m;3V*;E?!{^WO1$ENzUa)P?Tu?(VMn;+MhU@GZWfhX-$aCnx<Q'
            'M47$9WN;Q_NBp2{L;Qh*)01KUudt%icl{F%z=qz>pV5b;eP2;A!ey5n|Lj6y*%HwEl3uh7|0;R>{+Y_V1raKArTT9C*e&p4(+QFq'
            'Y{<lH*3Vs$eUwMX>M|8A^P)4Q$&`hHvhO5wf7^GmVt%OJzI@Rcy?in8$(a(LS7mo2lE>RlO<FYXttX=`5a%fH1MtmLpl5=3N20Tg'
            '0bw*-jOT6^yZ7FmL@{`<FBDRFNI_k+mv;;20*%_}WfG=c^9<Cu?8$-dgGqMVEgmEeBfu|ePW(&aEmD0%0R=sAHhBGd=a=5k{rt2_'
            '!P+?SG6qR{$zf;%4$xv|eBFUnUSke1S-f)u8-UXIBJ@^xxy;^3vXB79NcaoTw)2ZT_CRI34-$?H#05~1t{waUsK#imLS<We@X?w;'
            'ZJti4`VCeXh^a&v&A}F?u`L0b5&~+li*GfY;in14yy&0ng~1I<)azO@L=gWP2+4-{POhV~05PQ5_|pIg^7EJQm<+e$IYs@98VK>}'
            'JQLU<ugF~J`u6Pr{=b0~wuLvfY9bZ}wemGqV~MetmbCy_&kv6E{ShDQ%0=k;Ti(gM{O=Ioi-A2%V6{gw8-ap|&qH@__t^{dBweuU'
            '(V8PsVHZ-Ex`QWuypskWK^c>^6?URHYNL4_8UUV>5m;Ju*c}h>4WKWP{5^&7kobc8rxp}ssUJ>T68|FKpHO*)>6G)SqV<H4Qi`Z_'
            'cOir3sLV!?6%dgsVRYep*)2P0%i^JEqNWOA{s$07(f<J7c7l{4ha5m(7vvre`m7>_sY?mfAn3SAiqgH8q%o-e2Dn`#!WdqlGirU7'
            'sfL2G%hB+6deBFCi3`&Tc|o`i^2JJc!D@8ce@d=v49AXAmG|c6WR27c@+{;}Kn6+O1?DlK_VUG}>HW{(zErTjD(TcX>om2TN#rfU'
            'zU+*X=0&1k82_r!x$?@<h+dO8U14gbI9#R_Eia+Ptcx}r{X-?Ia7<8Mnm|+84szb6R-|#nY9jtDUc9i{sr@Z9>*gmjFK=hOeer^D'
            '9{;}Fe{qF(=ZfIz=RfT4J(F*cx*50X>f<)nUP*uQ++&6b26@xf5<8dv5BHjjmm-x~Mv_ixf=Ls4w!8oQ<y9^Nkk1qXprYqMb$YS8'
            '|Ki1!bGdtER3D?AP>wtg1%$MwXX2$^L4C@rGS1=?BPu=zoqd$ac4z_&0l!C^!LWh9UY3U`oO>F@WQ*NyB}uj{I%a#y&7;_(`gkGe'
            'L)E<L8XK@9xHMcXOK#FCc6NY6*+QH?78dEqL*LcoQtAc;f-(ie&U%(A@dwiZOdfzPDb%+^QGsdUUdbRC{Nr<IMx#(^UVfd+f-?n?'
            'i6_&*42`Svn*UCq9>}>tnETCI;w}#}?4Z*ZtV9vUJ4aQKPZuF3A?LeG=;&ry+LV9lh<SON1NyqY$aw)OR@wA(j-#dG^yKJQHj1ii'
            'x{m0n%U}wTm|`=m=}K|3K}6nXY`tT@F<BCIKBq{C)K@jN3`a833Sqm7^}WsMt3t+RWVs}E9C!AouCca~$ramN>3B5S9%(mXn`BE}'
            '^hiT&Bd^$;{{QV=TX);WwSMQXK#&KK2}#B6O4>?gyO@fuwPH(NlAYF7wMdYFDIQ+H0}!%lSO2~HHiH?=fTCoj?ddvsXiaD^nEUL@'
            'x4(^|E(ZTn%xss&xtWE{KmNXV)&1&^*R8+j-%6-_%L<~!wn?omkt0=I6hRJ4R=1cpBp0q&Tdh3aQbfD7?V`F1>N^b+3v*Oc7#T_V'
            '#~r4q=Qr=d>QGlcZQc_1wopNjsAR+SLc&^Q<1kx(%>|nlo1qp%Z`0XCr%!Y&B)d{2MyqLo(Cdkjf6-@0*-^&|nIz~0if0+84bhn#'
            'Ll?WVk5@EbC()dV%ar{be^5m+?+F!AgdDS|IMGt1;N6iufFqR$ne%=Typz4rMlce~w$7W2GzI+<lPXDI7dPkh?DF(vKznrLLo|(a'
            'W<OCWF^UIsJqzpF<^u@hAU<<#S<<2YnnWE50v8zRr_KsSkV-Bpjs@e~buuddla5vHVCo)xgV^3EOFV^)<lW#o>J6NQEx}Z9=wbm$'
            '2~DGV8u<W&!5L#3K)gOzITFWVV~Q2X8Ho)3i-du(Dsdty<NCCdj~*!utR|kLh%Iy>^KoeAx0c=!xSKmCnA|c0Y$oGsCnQw%cGh8O'
            '%ORfDKdYD&&P?^aNZxNSJd~Z*oet+nyGb)>g9;{NE+Jt-X%ldGoZLjnFO_kc!CTJXw+%=brzG4s=50<Q?_Lg)GM;9The{9+H&M(3'
            'gu2-xS=xG++V@KLQomc)UqR8O(`Gq`J=x?RxnjD*_6>#jj$a$6F{Yz6PMMl}1@BfY>`^o%xGCIha*0Fp5JHvtmE62H9<8Gq2Sb!D'
            '(3?qgEVtt{vnW-4mDD80stvq))4%*_cya#r&EWEidUNe_LRBNJs75>g)y`B?CElNeUQsR;R-c}gL?2rBQ}aLvzVehfPKg~6fA2%Y'
            'zps|b7Iwa25d)bFNf4;sud+LkSLdI)Zq2PwaINbsc|ZGzYsPc@t5t(y8-j~NlFNK0E6Yp-#8A%9HCZjitwqY5IJO=Nu$G#-h}BZL'
            '`UJ_qKIz_l@L$_7T!#ggQTdkkNW1DCMsV<FpK7hLnWr(#_=ZWwuEPG?lhgC?+P5v%NrJEoPRM=~_?fsv>vG4{ZT|LO=piitqh{DW'
            '3F5yuL+3C20D%hdlYZRYxwTdv-T4sMhU)^fVP$sT>DZ{c<c&i6({zvx9FG*#NW|zJrz_B%;=)X{u9lC8a>loOxvr>zjYp7jD5H*i'
            'PO)F9-=M-juxm!QpqHTwc`s{jJQn2bkGH_}->N{Vyx4EKE*=K{u4H`Dba!8I<h7e{%{XJPtErUiGRB$fDd~_UejW5T>2fwfuQFz>'
            '>U_vB$ZdrdS-$J@BD*CxI7Z?}`3<GRMIq@^(x#W-5lIMAAItP0HL}3T*^Jrf`ynkB2qRA-0@zVKBHy7j2qFiI3ff77wz#U_rpkJ%'
            'sL_^dC01w|f*6<Smj0;4j4uDilO1J@l3GpP@8ST6d9SIzD#A(L6*Om4moF6=J79thEI!-2E%4OLFdAuh6|78i`O_*(hZ+~ae^rPI'
            '&BK+<kLDArtmopRH)<|Kz3B7;C)f%@5hZGmI%$M)(uQlew6-#G?pvO*{MtxEUJm#O$+R#*Hwa;HSP)D<K(tC$T7iyn9TEggX)9kN'
            'R5T#?HLUdoLq)Z}f}&PRx(wEXl*y`Vb;B`>H4XkV0yL^dkW2aVT}z&t@7W`>7D9)TA6T1h^(v77%^`DdIMBvYa4IP>UtbJ<J{|n!'
            'm<7~;_%7k(Mc^zW5-U6s77|g}t#se6lo{@NcmBv~R)>0yPqS|IxKZ+Q_u1pg3JO(f?!{$+`Uy(^3F}K7-$=ZXh{dO~$*64!B3m?1'
            '@SI{_)q5DQ{06mmoa{1TyFOS7^gz#>OU|8X>nHo1<(1RfwWN5geqU-t`7-m#J%$r)1S5kw4V!74^j7&8V~eqvHA;%z(5R-SA68zN'
            'lrq2+*&OB-^<I$`g67B?mfiv3Zw^MSc9}FDlhU&eqtyS>{7|Y9!G-edGYRQBL598!T2z?j5N}ypwPGk)SuuApD74B$gWrymiE_RK'
            '?A_p7xHPd_u2u!P@4tS1asG4v<&h%ysUnB4d85ka9wBNVDnLFXD+&~DXw8AK>vCK`C+SYa!xJ_N6qvo=%1NpQex-T@Z_2^MR6Qe2'
            'qe2pbjw>IbQHkTulvN4(jIu^c*&}oqGL0Bc#UVo%5q!voh}jDE_>8e|nhu<bqOs$kY>e1UY?c;fqJmGso`fw{tW*ekFtWqoN9@xN'
            '){7!n!2}Jh7*M<a1x64#f_GTg|D9G3Zq@%$vxgWi*v^eI$UROO@8Q=H!vh9f3?E^C*j3Q9JKPg$ZZxy?pJR`iqzfj!c>-&Rs>(5!'
            'Sb%n(OedS}W9%vS*i_63|5nV#KCGYLzVbL5%LD8zkF~Y@GWHga&BgP`c94CE{be5;%oEvR?8W&2V+;zUizwez!&h=iJfraHUt+XT'
            '?NU~oaVYL;{g>7gTW>sHXgp1*{(rH-<`K4$(jHQ8+W3E~g=@8ts3wKX;-4*_VMDIe8q<b#;eMHg<6mU~;9M8=ntyN!3($ybL}Dps'
            'nV)5Oa;UNnBDo?z3M5$#ku2)mgVSV!f#jcH9x%(GQQ<a;lR2IsBGenNV{YT0FZwTE_5b!CV}bC+VKVCsA|NzvC;qVMab1=E8~xCv'
            'qxEF7uDz5!oMlW^<Z9EJX;5oLoe4-Q&_z^mwJO9G5EFl1DE+y=_<$H8SAA^C8s+6hwyhD5^yqNps1c*mj~;Q2q`*hWq;~n@e$`9k'
            '6zY{i7H(N1j7oh_YoNA*hTQJubnReBHCR3UtjHz7w62ahk$d_oQlwtJ(2+Aef?x0IfW*Fo>z1(yEMC5zsf#qc49LA|?|i3`p+Rxy'
            'dtNeG7%e8{sU0zs3btCO?66*i&DB_LaxU>wnjPc004FjgNKNF7i7}8GaV`Bb_}@PbUJhOjE>D9W2S2?$eRqCQ%~TqmYqHnJ5xSba'
            '_IKLDoax(`o@ecb%NeV&t_{E6z1jxjk0xIsi{P`^^9J9#HfJW^tWemQSo<EP`SqC%TLP;KJ0T)?J(Vkc<DL%~Z~Crv_kCX}i^SA-'
            'bGy}eiy8HNjlp)P=LaAMuO?))W3F^G0c<8WWC`2kT|OJ!^Hwi2vrah6kO=_(TEN=Q4}ycTJQ|aRRCSU|yAFaW;&nLy0$OncW0#o5'
            'Z0r2iRdwq|13%%LbpK$8jp^#>*$>xBj;bGMWl6GrTU@I!I&pG4)C{yb%}XT!q8h8vZ2C8WA#_$eEPMD}Oaj_8-wca99x5fIiijyo'
            'Fq+GWp9ufAz$1J+B$}+@v*Gt{5f5_a{%)CVV4>B0OU3A0<i@5HS<S2INRXbER7%+7vUJtvLv_=Zq@mVS-tDB|3<k?_630Z{F2;R~'
            'KYA3Kq&lDmuYMeyoM8U=lhap&v&+--Gq}8e^TTgFC0?DM3|<CIObDB&lc?r&eib$_2-ba04)RqRjPQ=9ILpy7qbA{ZFijI3q{NmX'
            '&N$I1ze$EnxbmK#{*~gh`OFVhij!eJTh10y#ys7jWLHqRm&!i|{|H#DQ~}GAMUiiiN@QBTS3Yx!Gwcy(H<mzO^K(S*BbxR<0#(BI'
            'HpREsze<0m_wHQ$upU-U_mS&-5@JrH+<{s`_Knc4+zN2>BQzAfR_vDAu|jsIfWYmHH#rGmmE;bO@@bVF4{m1>trlujha#%a-UN!w'
            'ox?Mg*w|4t$ReG?qAq)xTTZS>Pa4rRA(EfTme8lv>INfsWsk(Cu}M2R(Q$5FeCkUij>8^+`6vkAVNr@v@~t%p21$IZp|g(Pg-)e*'
            'St(Dv0`8#`XbtKLjd*bRqThkbNO)hO4=fGqBsDZaF++DL>S0uwGV-ESaw<?n1{$Dg<z0I+$mEiwiljjDLC{k>zYW&1!|zf~^e^Xr'
            'nWoDE<PK1%F(cxN;-cHN4wgEf$!>>>(U0p^+v{g>E8fHJVqrCF&Pck9s;2^R1gh83cZRV;SK11XrYfA97AG!5%4N=>v&<jtaPSha'
            '(ttc1en~C$BQKSPrguanxf<XYXzGV83!(T3gRpV*^lHdfjLCL@jOE`RU@o=>E6{l_eMa<K5TMg6F9~9lV~3IUFo|_7;ApY3Q}hD#'
            'hM)VH8TfAx^akh#Kx607*rH<pHsXl5{uUaN?OA--Cu`D8>zsn)bPLDfXI=fBBo+)l8;!VggV*df%g`&&#TADP-trcrU^cGlx>XIA'
            'k~gWJ6t^x5{Q)k8h9{f0Rn=WEon`SXfYVX+OTKz1^T5j~tZ^i%EN=YLV?7bOv*zT*vg!BAZg><WL9KLYFMo!0gFMKsVZqeY(jY^p'
            'y%w#4)pM50S0C2B!8zUOpH>kb3e-C#sQSuELQ!{4yH!x+jW*d+F(a?6Ev7jJUwX{nr#EX&yP)VHp-U-dHlubhCc%#O#E~fhMB1Z<'
            '|G;KNdLN8;B{vG4&jisH6!_v3CbXPyB04D3#WI4!QkFR8fu~)c$$1l}6gLR#?ql#xu%%J_*beBR{4`s0!?`HRXv4@T{gF!(FG&lQ'
            '58TM#mnFwia;xj5O2<}78UwDj7ktMf!SBLP=LUrv-%39RZ9hZ-mn!&1UR8`PtMoQmC}p~2>lyL}ZfgtgP#-?_^njfAEa)ImL*~=e'
            'ouX-kqMoh<!Jox=K?uOZGA)|iSR!pd>wbssrF279AE}LL06CKjWtPDZ<`u`u25|vWFbdDz;3R>qi4r%`!%}tA*{!e^QrZS}`udX3'
            'R{QFV>@ykJF{d5GY{>2+cE>P9+Em_p^iMgR>~XcW3B`P4^=KNQ+|bYPA2>7}`g#kb!ynWfV(Mj`yK|^VBi}KdNREtntaJg2A5ACp'
            '=ezFje&q%`pQ!pU##<bH(IiWAjKM-y5V9;Oo=X*Nr#|H72u9NRG3q4TCd?NuJUm34$2PejBGgqhhULi>>h_LM-oo$7%Ain5dp=bq'
            '7X6*|RdWi;X{JqpsFgh?Ra$RR2+-?)=ow>cORU^1O^+B-lL#h4O$#lX1TO-r6X&{0VVJ?rX-bVqEV%6n-}mw;=q#_yx~|ni5j(|j'
            'yfynqUtgd(m6u}8+W77F?%McsmL&1EO)EohJ0Z|!l3>F$(&UiP4g!3QZ*-93DlUoA<qgGSh&ulDJu_MDK{@OcXAmnZf(>Ef?+}Pd'
            'eX=TCx(h)~zr1o4PEY_&XEAZs*G|3Bv8!;2!bS#Hq*@8oewWotNIl4H?!4QoufJ>(j+<AOYEF=if<Dj*o*_gI1kgkNA`Oo4D&Xbj'
            'sn=cp>c4B}iJk(B6-9G1fr34+YCrr+B_dzusN6kjC_pWy0YnU<ZlY+{(oB2vT$%4qj4{3AzMD(pTDFcpRfq8P2yV=#+?=vS7^^6M'
            'Gq=sJzxQ03ueqGE-vkV1Mw;>tXThdZmd_JPter$zM>j!Vm}<*NQ{+mr?+jKiq(rObxo4Vut*9cfFpRFqHN$4)K3p)<EV{+JUF^uu'
            'TSpWep3Mjo4%_nl`E%1mR-cmyEtFANqvq2dn)Nw^>(DIB?6rUHSLZ+M-6n;)MQdQY<=&}kp9F1hcF`JEP5BPm3!L$Cot{WOGaF!k'
            '@%C)kKYMff=ZpT&r*Ga3F9w%`{>7hOs8Ja)gFXdqAPGARxQaJt47(0%XPt=i>l8W!U1T-u8N0=70J^xJ%wm+zH9|4KP8B-R7D`5-'
            '3qAaX(-H<z-Y5y<P<QPJww}EE3lQ-7DD>ezfci((nH&Y<BAzB_t)zNztov`yFWw2`ncx?6Bogc+3R^+JBaXHQxgg+V)K53Lf$bMW'
            '#mJnx5geQjXu~U8s%7igFl~>zGHCktMZIZLxt=`f-%6{e+3wI=sx)>s!@9M_7#Iw{%JOInx8kaW<Ugxqn<L?I79oIX83p$xz70Xg'
            '7O)dS*W1C(Y%-o@3~DbS?cFMq^OphIy~A2XxIalye^_0cP^kLIlWmW9#Vf@E5^^@p>T_cC`EKvLzIz?FZ1;&Kvw48C;WTAjY$i$W'
            'B=`N4W~fcaMTW_Vau6dx)E^hyV$M4%*c5Y2HHtHvNAs4G)mQdy&)-p<i*RSww?Fixnt$Ua6Lijf+v|O(yq3`o!3lVja7V%5R{P)k'
            '%>BB%e^<%;K#fs8+osv^fAxmqVKKMSQ}`Sn&H6w&K>SVzCr2G02hC?~we&l?SGUI@@nm88>Jxj|Nex_sfAf&(v|%_4v<Y?ZRCA7v'
            ')k@V-rYNx%!*!r`zYQ(cl7t$5ktFDaQYrMo(tCBVbQAV!-Ku+lUn__5$>418?sD)lh)fwU0@1HHcJab*5v8|85ejA+vBfGCZM%9P'
            '6G9k{<6_BaR2W#iOg`z0Hcz&2Dr{#fq~H7G9LV)pTyx?zJk!MERs`Vvp`A9Yvvz^)z<F9%Su{l-e&`@K6}r=fZ6Y(dLTJ{AOE%c@'
            'UcjP_KfH2qpBCu6ELIhpE336E=-mfa0{d!#YK!*WA-{Zf-{)D9bTADxGsrR;Dwq~g7OT9zziT~tc3jGrhG>UZh#!5s26Z|95y8+&'
            '+`K4On1+g^DY-Qe3@_UOVe#2!SYl5p85g`p|6e|5RTYh7I-{Hh+svcwDw=a%$B;HonG^>!h>PTXK}60ek`~6qaXv}c+l*M-;(+SM'
            'zfyEBj9)<G2eKHFMB_tvJ!6eev?gSgTyH3ZTGX7E$nqRKf4*ENQ=~AB(D2dua5Kx1aP2U7y}iWrlqY2Uh2M+WI7@Sk`Nvc%C0!ja'
            'tW~tw=7bi3{ddf0k^0=1ZOZcnWtVHjf#7!z37OZ>_E=$m)X52S;+|=WJ!d?oSLdqELq$^d&w0c4j_yjc$-2-{@Y$HzyP%lBRsjh('
            'm`9r>3`g~7e)smLmxD9RK}9s_-X!N3vKqWA<``)|5Af^;SuV$XFwr)PrjGoL0{bKaxgGJ_(<Dpgi&=0J%@?4MfmC@jTL<YPE~a=H'
            'Y?3Xg!pX)u^2H*fmonVDtJ!MWq$LS!^~gK}3@PC4yZ*)L(avO~gbL~gFQO$c^Fv(Ff}+)9WdyD8u+3ID!HC|S5$NW=!TPZP)ddQ}'
            '*zjvOa7L}9*X$gb82w4mD)_A<V*UwmN9GI'
        ),
    },
}

def _load_local_pipeline_modules():
    # 1) Zaten import edilebiliyorsa (yerel .venv, repo Colab'da klonlanmis) dogrudan kullan.
    try:
        import local_pipeline
        modules = {}
        for _name in _LOCAL_PIPELINE_MODULE_NAMES:
            modules[_name] = __import__(f'local_pipeline.{_name}', fromlist=[_name])
        print('local_pipeline modulleri dogrudan import edildi:', list(modules))
        return modules
    except Exception:
        pass

    # 2) Drive'da veya proje kokunun bir ustunde daha once birakilmis bir kopya var mi?
    search_roots = []
    try:
        search_roots.append(Path(DRIVE_PROJECT_ROOT))
    except NameError:
        pass
    search_roots += [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    for root in search_roots:
        candidate_dir = Path(root) / 'local_pipeline'
        if all((candidate_dir / f'{_n}.py').is_file() for _n in _LOCAL_PIPELINE_MODULE_NAMES):
            sys.path.insert(0, str(candidate_dir.parent))
            try:
                modules = {}
                for _name in _LOCAL_PIPELINE_MODULE_NAMES:
                    modules[_name] = __import__(f'local_pipeline.{_name}', fromlist=[_name])
                print('local_pipeline modulleri onceden birakilmis kopyadan yuklendi:', candidate_dir)
                return modules
            except Exception:
                pass

    # 3) PROJECT_REPO_URL verilmisse git clone dene.
    if PROJECT_REPO_URL:
        clone_dir = Path('/content/youtube_automation_repo') if os.path.isdir('/content') else Path.cwd() / '_youtube_automation_repo'
        if not (clone_dir / 'local_pipeline' / (_LOCAL_PIPELINE_MODULE_NAMES[0] + '.py')).is_file():
            subprocess.run(['git', 'clone', '--depth', '1', PROJECT_REPO_URL, str(clone_dir)], check=False)
        if all((clone_dir / 'local_pipeline' / f'{_n}.py').is_file() for _n in _LOCAL_PIPELINE_MODULE_NAMES):
            sys.path.insert(0, str(clone_dir))
            try:
                modules = {}
                for _name in _LOCAL_PIPELINE_MODULE_NAMES:
                    modules[_name] = __import__(f'local_pipeline.{_name}', fromlist=[_name])
                print('local_pipeline modulleri PROJECT_REPO_URL klonundan yuklendi:', clone_dir)
                return modules
            except Exception:
                pass

    # 4) Self-contained fallback: bu hucreye gomulu, build sirasinda
    # local_pipeline/<modul>.py'den birebir uretilmis base64 kopyalari, SHA-256
    # hash'lerini dogrulayarak diske yazip import et.
    fallback_dir = Path(tempfile.mkdtemp(prefix='youtube_automation_bootstrap_'))
    pkg_dir = fallback_dir / 'local_pipeline'
    pkg_dir.mkdir(parents=True, exist_ok=True)
    (pkg_dir / '__init__.py').write_text('', encoding='utf-8')
    for _name, _entry in _LOCAL_PIPELINE_PAYLOADS.items():
        if _entry.get('encoding') != 'zlib+b85':
            raise RuntimeError(f"Desteklenmeyen gomulu payload formati: {_entry.get('encoding')!r}")
        _source_bytes = zlib.decompress(base64.b85decode(_entry['data']))
        _actual_digest = hashlib.sha256(_source_bytes).hexdigest()
        if _actual_digest != _entry['sha256']:
            raise RuntimeError(
                f"Gomulu local_pipeline.{_name} payloadi bozuk/tutarsiz: "
                f"beklenen sha256={_entry['sha256']!r}, hesaplanan={_actual_digest!r}. "
                "Bu notebook yeniden uretilmeli (nb_builder ile)."
            )
        (pkg_dir / f'{_name}.py').write_text(_source_bytes.decode('utf-8'), encoding='utf-8')
    sys.path.insert(0, str(fallback_dir))
    modules = {}
    for _name in _LOCAL_PIPELINE_MODULE_NAMES:
        modules[_name] = __import__(f'local_pipeline.{_name}', fromlist=[_name])
    print('local_pipeline modulleri: repo/Drive/PROJECT_REPO_URL bulunamadi; gomulu (self-contained, hash dogrulanmis) kopyalar kullanildi.')
    return modules

_local_pipeline_modules = _load_local_pipeline_modules()
video_analysis = _local_pipeline_modules['video_analysis']
render_utils = _local_pipeline_modules['render_utils']
google_ai = _local_pipeline_modules['google_ai']



## 1. Test Videosu

İki seçenek: (A) kendi dosyanı yükle, (B) bir YouTube URL'sinden kısa bir klip indir. Gerçekçi bir karşılaştırma için **3–6 dakikalık** bir örnek yeterli (tüm videoyu değil).

**YouTube "Sign in to confirm you're not a bot" hatası alırsan** (Colab IP'leri sık engellenir), iki yoldan biri:
- **En garantili — SECENEK A (upload):** Videoyu kendi bilgisayarına indir (ör. tarayıcı uzantısıyla ya da yerelde `yt-dlp`), Colab sol paneldeki 📁 **Files**'tan yükle, `SOURCE_MODE = 'upload'` yap ve `UPLOADED_PATH`'i dosya adına eşitle.
- **YouTube'dan indirmeye devam etmek istersen — cookies:** Chrome/Firefox'a "Get cookies.txt LOCALLY" gibi bir uzantı kur, YouTube'da oturum açıkken `cookies.txt` dışa aktar, Colab Files'a yükle ve aşağıda `COOKIES_FILE = 'cookies.txt'` yaz. Bu, yt-dlp'ye "gerçek bir tarayıcı oturumu" gösterip bot kontrolünü çoğu zaman aşar.

In [ ]:
# SECENEK A: Kendi dosyani kullan. Colab ve yerel Jupyter otomatik algilanir.
# SECENEK B: YouTube URL'sinden indir.

SOURCE_MODE = 'upload'  # 'upload' veya 'youtube'
YOUTUBE_URL = ''
VIDEO_FILENAME = 'THE ABSOLUTE BEST OF DAEMON TARGARYEN - ABC CINEMA (1080p).mp4'
COOKIES_FILE = ''
CLIP_START_SEC = 0
CLIP_DURATION_SEC = 240

if IS_COLAB:
    SEARCH_ROOTS = [Path('/content/sample_data'), Path('/content')]
else:
    SEARCH_ROOTS = [
        PROJECT_ROOT / 'sample_data',
        PROJECT_ROOT,
        Path.home() / 'Downloads',
        Path.home() / 'Videos',
    ]

UPLOADED_PATH = str(SEARCH_ROOTS[0] / VIDEO_FILENAME)
RAW_VIDEO = str(RUNTIME_DIR / 'source_raw.mp4')
VIDEO_PATH = str(RUNTIME_DIR / 'source_clip.mp4')
AUDIO_PATH = str(RUNTIME_DIR / 'source_clip.wav')

def resolve_uploaded_path(path):
    """Tam yol yoksa Colab veya yerel arama klasorlerinde kaynak videoyu bulur."""
    requested = Path(path)
    if requested.is_file():
        return str(requested.resolve())

    print(f'UYARI: tam yol bulunamadi -> {requested}')
    generated_names = {'source_clip.mp4', 'input_video.mp4'}
    exact_matches = []
    fallback_matches = []
    for root in SEARCH_ROOTS:
        if not root.exists():
            continue
        exact = root / VIDEO_FILENAME
        if exact.is_file():
            exact_matches.append(exact)
        for extension in ('*.mp4', '*.mkv', '*.webm', '*.mov'):
            fallback_matches.extend(
                item for item in root.rglob(extension)
                if item.name not in generated_names
            )

    candidates = exact_matches or sorted(set(fallback_matches))
    print('Bulunan kaynak videolar:', [str(item) for item in candidates])
    if len(candidates) == 1:
        selected = str(candidates[0].resolve())
        print('Otomatik kullanilan video:', selected)
        return selected
    if len(candidates) > 1:
        raise FileNotFoundError(
            f'Birden fazla video bulundu. VIDEO_FILENAME veya UPLOADED_PATH degerini netlestir: {candidates}'
        )
    raise FileNotFoundError(
        f'Video bulunamadi. Aranan klasorler: {[str(root) for root in SEARCH_ROOTS]}'
    )

def download_youtube(url, out_path, cookies_file=''):
    base_cmd = [
        'yt-dlp',
        '-f', 'bv*[ext=mp4]+ba[ext=m4a]/b[ext=mp4]/b',
        '--merge-output-format', 'mp4',
        '--no-playlist',
        '-o', out_path,
    ]
    attempts = [['--cookies', cookies_file]] if cookies_file else []
    attempts += [[], ['--extractor-args', 'youtube:player_client=android']]
    last_stderr = ''
    for extra in attempts:
        proc = subprocess.run(base_cmd + extra + [url], capture_output=True, text=True)
        if proc.returncode == 0:
            print('Indirme basarili.')
            return
        last_stderr = proc.stderr
    print(last_stderr[-3000:])
    raise RuntimeError('yt-dlp indirme basarisiz oldu; upload modunu veya cookies dosyasini kullan.')

if SOURCE_MODE == 'youtube':
    assert YOUTUBE_URL, 'YOUTUBE_URL bos olamaz'
    download_youtube(YOUTUBE_URL, RAW_VIDEO, cookies_file=COOKIES_FILE)
else:
    RAW_VIDEO = resolve_uploaded_path(UPLOADED_PATH)

file_size_mb = os.path.getsize(RAW_VIDEO) / (1024 ** 2)
print(f'Kaynak video: {RAW_VIDEO}')
print(f'Dosya boyutu: {file_size_mb:.2f} MB')
if file_size_mb < 1:
    raise RuntimeError('Video dosyasi cok kucuk; yukleme veya kopyalama tamamlanmamis olabilir.')

if FFPROBE_BIN:
    probe = subprocess.run(
        [
            FFPROBE_BIN, '-v', 'error',
            '-show_entries', 'stream=index,codec_type,codec_name,width,height,duration',
            '-of', 'json', RAW_VIDEO,
        ],
        capture_output=True,
        text=True,
    )
    if probe.returncode != 0:
        raise RuntimeError(f'FFprobe kaynak videoyu okuyamadi: {probe.stderr[-3000:]}')
    streams = json.loads(probe.stdout).get('streams', [])
    print('Bulunan akislar:', streams)
    if not any(stream.get('codec_type') == 'video' for stream in streams):
        raise RuntimeError('Dosyada video akisi bulunamadi.')
    if not any(stream.get('codec_type') == 'audio' for stream in streams):
        raise RuntimeError('Dosyada ses akisi bulunamadi; transcript uretilemez.')
else:
    print('FFprobe yok; akis kontrolu FFmpeg donusumu sirasinda yapilacak.')

# Kanonik klip+ses cikarimi (master notebook ve verify_local.py ile ayni fonksiyon).
video_analysis.extract_clip_and_audio(
    FFMPEG_BIN, RAW_VIDEO, VIDEO_PATH, AUDIO_PATH, CLIP_START_SEC, CLIP_DURATION_SEC,
)

print('Klip hazir:', VIDEO_PATH)
print('Ses hazir:', AUDIO_PATH)

## 2. Transcript Çıkarımı — Aday Modeller

Adaylar: `faster-whisper` (tiny / small / large-v3-turbo, lokal & ücretsiz) ve `OpenAI gpt-4o-mini-transcribe` (API, ücretli — anahtar varsa).

In [ ]:
import gc

transcripts = {}
transcript_segments = {}

try:
    cuda_check = subprocess.run(['nvidia-smi'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    cuda_available = cuda_check.returncode == 0
except FileNotFoundError:
    cuda_available = False
local_gpu_opt_in = os.environ.get('YOUTUBE_AUTOMATION_USE_GPU') == '1'
cuda_available = cuda_available and (IS_COLAB or local_gpu_opt_in)
TRANSCRIBE_DEVICE = 'cuda' if cuda_available else 'cpu'
TRANSCRIBE_COMPUTE_TYPE = 'float16' if TRANSCRIBE_DEVICE == 'cuda' else 'int8'
# Karar (gercek test sonuclarina gore): large-v3-turbo GPU'da varsayilan
# (~%91 ortalama kelime guveni), small CPU fallback (~%85). tiny (~%72) sadece
# karsilastirma icin tutuluyor, production icin onerilmez.
TRANSCRIPT_MODELS = ['tiny', 'small']
if TRANSCRIBE_DEVICE == 'cuda':
    TRANSCRIPT_MODELS.append('large-v3-turbo')
else:
    print('GPU bulunamadi; large-v3-turbo cok yavas olacagi icin atlandi. CPU fallback: small.')

print('Transkripsiyon cihazi:', TRANSCRIBE_DEVICE, '| modeller:', TRANSCRIPT_MODELS)

# Bu notebook'un katkisi: AYNI kanonik transcribe_audio() fonksiyonunu birden
# fazla model boyutuyla calistirip karsilastirmak.
for size in TRANSCRIPT_MODELS:
    candidate_name = f'faster-whisper-{size}'
    t0 = time.time()
    text, segment_rows = video_analysis.transcribe_audio(
        AUDIO_PATH, model_size=size, device=TRANSCRIBE_DEVICE, compute_type=TRANSCRIBE_COMPUTE_TYPE,
    )
    latency = time.time() - t0
    transcripts[candidate_name] = text
    transcript_segments[candidate_name] = segment_rows
    log_result(
        'transcript', candidate_name, quality_score=None, cost_usd=0.0, latency_sec=latency,
        notes=f'{len(text)} karakter, {len(segment_rows)} segment, VAD+timestamps',
    )
    gc.collect()

if OPENAI_API_KEY:
    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_API_KEY)
    t0 = time.time()
    # gpt-4o-mini-transcribe: whisper-1'in yerini alan guncel, daha ucuz model.
    with open(AUDIO_PATH, 'rb') as f:
        resp = client.audio.transcriptions.create(model='gpt-4o-mini-transcribe', file=f)
    latency = time.time() - t0
    transcripts['openai-gpt-4o-mini-transcribe'] = resp.text
    cost = (CLIP_DURATION_SEC / 60) * 0.003  # yaklasik; platform.openai.com/docs/pricing
    log_result('transcript', 'openai-gpt-4o-mini-transcribe', quality_score=None, cost_usd=cost,
               latency_sec=latency, notes=f'{len(resp.text)} karakter')
else:
    print('OpenAI key yok, openai-gpt-4o-mini-transcribe atlandi.')


In [ ]:
# Ciktilari yan yana oku ve kalite puanini (1-5) kendin gir.
for name, text in transcripts.items():
    print('=' * 20, name, '=' * 20)
    print(text[:800], '...\n')

transcript_quality = {
    # ornek: 'faster-whisper-tiny': 2, 'faster-whisper-small': 3, 'faster-whisper-large-v3-turbo': 4
}

for r in results:
    if r['task'] == 'transcript' and r['candidate'] in transcript_quality:
        r['quality_score_1_5'] = transcript_quality[r['candidate']]
print('Kalite puanlarini transcript_quality sozlugune girip bu hucreyi tekrar calistir.')

## 3. Sahne Tespiti — Aday Yöntemler

Adaylar: PySceneDetect `ContentDetector` ve `AdaptiveDetector`. Kalite ölçütü: bulunan sahne sayısının videoyu mantıklı şekilde bölüp bölmediği (elle kontrol).

In [ ]:
scene_results = {}

# Kanonik detect_scenes() iki dedektorle karsilastirmali calistirilir.
for name, detector_key in [('content-detector', 'content'), ('adaptive-detector', 'adaptive')]:
    t0 = time.time()
    scene_list = video_analysis.detect_scenes(VIDEO_PATH, detector=detector_key)
    latency = time.time() - t0
    scene_results[name] = scene_list
    log_result('scene-detection', name, quality_score=None, cost_usd=0.0, latency_sec=latency,
               notes=f'{len(scene_list)} sahne bulundu')


In [ ]:
scene_quality = {
    # ornek: 'content-detector': 4, 'adaptive-detector': 3
}
for r in results:
    if r['task'] == 'scene-detection' and r['candidate'] in scene_quality:
        r['quality_score_1_5'] = scene_quality[r['candidate']]
print('Sahne sayilarini yukarida gordukten sonra scene_quality sozlugunu doldurup bu hucreyi tekrar calistir.')

## 4. Ses Enerjisi Analizi (Destek Sinyali)

Bu adım tek bir sabit yöntemle (librosa RMS enerji tepe noktaları) çalışır — aday model karşılaştırması gerektirmez, sadece hot-moment skorlamasına ek sinyal sağlar. Bu yüzden burada kalite/maliyet karşılaştırması yapılmaz.

In [ ]:
import numpy as np

t0 = time.time()
# Kanonik compute_audio_peaks(): 16kHz yukleme + RMS + min-mesafeli tepe gruplama.
audio_info = video_analysis.compute_audio_peaks(AUDIO_PATH, top_n=10, min_distance_sec=1.5)
times, rms = audio_info['times'], audio_info['rms']
audio_peak_details = audio_info['peaks']
audio_peaks = [item['time_sec'] for item in audio_peak_details]
latency = time.time() - t0
log_result('audio-energy', 'librosa-rms', quality_score='n/a', cost_usd=0.0, latency_sec=latency,
           notes=f'gruplanmis en yuksek enerjili anlar: {[round(t, 1) for t in audio_peaks]}')


## 5. Yerel + Opsiyonel LLM Hot-Moment Skorlama

Önce transcript, sahne ve ses sinyallerinden API gerektirmeyen yerel hot-moment adayları üretilir. Aynı adaylar, API anahtarı varsa LLM sağlayıcılarıyla karşılaştırılır.

In [ ]:
t0 = time.time()
TRANSCRIPT_PREFERENCE = [
    'faster-whisper-large-v3-turbo',
    'faster-whisper-small',
    'faster-whisper-tiny',
]
BEST_TRANSCRIPT_CANDIDATE = next(
    name for name in TRANSCRIPT_PREFERENCE
    if name in transcript_segments and transcripts.get(name, '').strip()
)
print('Kullanilan transcript:', BEST_TRANSCRIPT_CANDIDATE)

LOCAL_WINDOW_SEC = 30.0
LOCAL_STRIDE_SEC = 10.0
TOP_HOT_MOMENTS = 5

analysis_duration = min(float(CLIP_DURATION_SEC), float(times[-1]))
content_scenes = scene_results.get('content-detector', [])
scene_boundaries = [end for _, end in content_scenes[:-1]] if content_scenes else []

# Kanonik skorlama: master notebook da AYNI score_hot_moments() fonksiyonunu cagirir.
hot_moments = video_analysis.score_hot_moments(
    transcript_segments[BEST_TRANSCRIPT_CANDIDATE], times, rms, scene_boundaries,
    analysis_duration, CLIP_START_SEC, top_n=TOP_HOT_MOMENTS,
    window_sec=LOCAL_WINDOW_SEC, stride_sec=LOCAL_STRIDE_SEC,
)

# Kanonik onizleme klibi uretimi (her ana bir 720p klip).
video_analysis.generate_preview_clips(FFMPEG_BIN, VIDEO_PATH, hot_moments, RESULTS_DIR / 'hot_moment_previews')

# Kanonik manifest: status HER ZAMAN 'review_required' (asla otomatik approved).
hot_moments_manifest = video_analysis.build_hot_moments_manifest(
    hot_moments, transcript_candidate=BEST_TRANSCRIPT_CANDIDATE,
    window_sec=LOCAL_WINDOW_SEC, stride_sec=LOCAL_STRIDE_SEC,
    weights=video_analysis.DEFAULT_WEIGHTS,
    raw_video_path=RAW_VIDEO, analysis_clip_path=VIDEO_PATH,
    clip_start_sec=CLIP_START_SEC, clip_duration_sec=CLIP_DURATION_SEC,
)

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
hot_moments_path = RESULTS_DIR / 'hot_moments.json'
with open(hot_moments_path, 'w', encoding='utf-8') as hot_file:
    json.dump(hot_moments_manifest, hot_file, ensure_ascii=False, indent=2)

local_latency = time.time() - t0
log_result(
    'hotmoment-scoring', 'local-multisignal-v2', quality_score=None, cost_usd=0.0,
    latency_sec=local_latency,
    notes=str([(item['start_sec'], item['end_sec'], item['score']) for item in hot_moments]),
    status='review_required',
)
print('Yerel hot-moment dosyasi:', hot_moments_path, '(status: review_required)')
display(pd.DataFrame(hot_moments)[['rank', 'aligned_start_sec', 'aligned_end_sec', 'score', 'reason', 'preview_path', 'transcript']])

# Opsiyonel LLM karsilastirmasinda yerel yontemin sectigi ayni pencereleri kullan.
windows = [item['transcript'] for item in hot_moments if item['transcript']]
print(f'{len(windows)} zaman kodlu pencere olusturuldu.')

SCORING_PROMPT = (
    "Asagidaki video transkript parcasini oku. 0-100 arasi bir 'hot moment' (vurucu, "
    "paylasilabilir, dikkat cekici an) skoru ver ve 1 cumlelik gerekce yaz. "
    "SADECE JSON don: {{\"score\": <int>, \"reason\": \"...\"}}\n\nMetin:\n{chunk}"
)


In [ ]:
def parse_json_score(raw):
    try:
        start = raw.find('{'); end = raw.rfind('}') + 1
        return json.loads(raw[start:end])
    except Exception:
        return {'score': None, 'reason': raw[:200]}

llm_outputs = {'local-multisignal-v2': hot_moments}

# --- OpenAI gpt-5.6-luna (gpt-4o-mini'nin yerini alan dusuk maliyetli model) ---
if OPENAI_API_KEY:
    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_API_KEY)
    t0 = time.time()
    scored = []
    for chunk in windows:
        resp = client.chat.completions.create(
            model='gpt-5.6-luna',
            messages=[{'role': 'user', 'content': SCORING_PROMPT.format(chunk=chunk)}],
            temperature=0,
        )
        scored.append(parse_json_score(resp.choices[0].message.content))
    latency = time.time() - t0
    llm_outputs['openai-gpt-5.6-luna'] = scored
    approx_tokens = sum(len(c) for c in windows) / 4
    cost = (approx_tokens / 1_000_000) * 1.0  # yaklasik input fiyati, guncelini platform.openai.com/docs/pricing adresinden kontrol et
    log_result('hotmoment-scoring', 'openai-gpt-5.6-luna', quality_score=None, cost_usd=cost, latency_sec=latency,
               notes=str([s.get('score') for s in scored]))
else:
    print('OpenAI key yok, gpt-5.6-luna atlandi.')

# --- Anthropic claude-haiku-4-5 ---
if ANTHROPIC_API_KEY:
    import anthropic
    client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
    t0 = time.time()
    scored = []
    for chunk in windows:
        resp = client.messages.create(
            model='claude-haiku-4-5',
            max_tokens=200,
            messages=[{'role': 'user', 'content': SCORING_PROMPT.format(chunk=chunk)}],
        )
        scored.append(parse_json_score(resp.content[0].text))
    latency = time.time() - t0
    llm_outputs['anthropic-claude-haiku-4-5'] = scored
    approx_tokens = sum(len(c) for c in windows) / 4
    cost = (approx_tokens / 1_000_000) * 1.00  # yaklasik input fiyati, guncelini platform.claude.com/docs/en/pricing adresinden kontrol et
    log_result('hotmoment-scoring', 'anthropic-claude-haiku-4-5', quality_score=None, cost_usd=cost, latency_sec=latency,
               notes=str([s.get('score') for s in scored]))
else:
    print('Anthropic key yok, claude-haiku-4-5 atlandi.')

# --- Google: SABIT MODEL YOK. Her cagri kanonik router uzerinden gider -------
# (local_pipeline/google_ai.py: Rota A = gemini-3.6-flash -> gemini-3.5-flash ->
#  gemini-3.5-flash-lite -> gemma-4-31b-it, ilk basarida durur, kota defteri
#  tutulur, hatalar siniflandirilir ve sanitize edilir.)
#
# Ucretsiz katman veri politikasi: buraya SADECE yerel olarak uretilmis transcript
# pencereleri gonderilir; ham video ASLA yuklenmez.
GOOGLE_CREDENTIAL = google_ai.resolve_secret('google_api', get_secret)
print('Gemini kimlik bilgisi:', google_ai.describe_credential_state(GOOGLE_CREDENTIAL))

if GOOGLE_CREDENTIAL.configured:
    _g_client = google_ai.build_google_client(GOOGLE_CREDENTIAL)
    _g_manifest = google_ai.build_google_capability_manifest(
        _g_client, credential=GOOGLE_CREDENTIAL)
    _g_ledger = google_ai.QuotaLedger()
    _g_selection = google_ai.select_google_route(
        'hotmoment_review', manifest=_g_manifest, ledger=_g_ledger)
    print(google_ai.format_route_line(_g_selection, GOOGLE_CREDENTIAL))

    t0 = time.time()
    scored, _g_models, _g_failed = [], [], None
    for chunk in windows:
        _g_result = google_ai.run_google_hotmoment_review(
            SCORING_PROMPT.format(chunk=chunk), client=_g_client,
            manifest=_g_manifest, ledger=_g_ledger,
            required_fields=(), checks=(),
        )
        if _g_result['status'] != 'ok':
            _g_failed = _g_result['status_reason']
            break
        _g_models.append(_g_result['model'])
        scored.append(parse_json_score(_g_result['output']))
    latency = time.time() - t0

    if _g_failed:
        # Cikti UYDURULMAZ: basarisizlik acikca 'failed' olarak loglanir.
        log_result('hotmoment-scoring', 'google-router', quality_score=None, cost_usd=0.0,
                   latency_sec=latency, notes=f'Google rotasi basarisiz: {_g_failed}',
                   status='failed')
        print('Google hot-moment incelemesi basarisiz:', _g_failed)
    else:
        _g_label = 'google-' + (sorted(set(_g_models))[0] if _g_models else 'router')
        llm_outputs[_g_label] = scored
        # Ucretsiz katman: dolar maliyeti 0; gercek butce RPM/TPM/RPD kotasidir.
        log_result('hotmoment-scoring', _g_label, quality_score=None, cost_usd=0.0,
                   latency_sec=latency,
                   notes=str([s.get('score') for s in scored])
                         + f' | kullanilan modeller: {sorted(set(_g_models))}')
else:
    print('Google key yok (GeminiKey / GOOGLE_API_KEY), Google adayi atlandi.')
    print(google_ai.credential_help_text('google_api'))

In [ ]:
for name, scored in llm_outputs.items():
    print('=' * 20, name, '=' * 20)
    for s in scored:
        print(s)
    print()

hotmoment_quality = {
    # ornek: 'local-multisignal-v2': 4, 'openai-gpt-5.6-luna': 4
}
for r in results:
    if r['task'] == 'hotmoment-scoring' and r['candidate'] in hotmoment_quality:
        r['quality_score_1_5'] = hotmoment_quality[r['candidate']]
print('Skorlarin tutarliligini/isabetini degerlendirip hotmoment_quality sozlugunu doldur, sonra tekrar calistir.')

## 6. Karşılaştırma Tablosu ve Dışa Aktarım

In [ ]:
df = pd.DataFrame(results)
# Not: satir bazli status artik log_result tarafindan ayarlaniyor; eksikleri guvenli varsayilana tamamla.
if 'status' not in df.columns:
    df['status'] = 'experimental'
else:
    df['status'] = df['status'].fillna('experimental')
display(df)

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
out_path = RESULTS_DIR / '01_video_analysis_hotmoment_sonuclar.csv'
df.to_csv(out_path, index=False)
print('Kaydedildi:', out_path)
print('\nBu dosyayi indirip experiments/results/ altina koy; roadmap Faz 1deki provider registry karari bu sonuclara gore verilecek.')

## 7. Tanılama Paketini İndir

Bu hücre transcriptleri, sahne zamanlarını, ses tepe noktalarını ve karşılaştırma CSV'sini tek bir ZIP dosyasında toplar.

In [ ]:
import zipfile

scene_report = {}
for detector_name, scenes in globals().get('scene_results', {}).items():
    scene_report[detector_name] = {
        'count': len(scenes),
        # detect_scenes() zaten (start_sec, end_sec) float ikilileri dondurur.
        'scenes': [{'start_sec': round(start, 3), 'end_sec': round(end, 3)} for start, end in scenes],
    }

diagnostic_report = {
    'status': 'experimental',
    'status_reason': 'provider comparison notebook output; not a final production artifact',
    'source': {
        'raw_video': globals().get('RAW_VIDEO'),
        'video_path': globals().get('VIDEO_PATH'),
        'audio_path': globals().get('AUDIO_PATH'),
        'clip_start_sec': globals().get('CLIP_START_SEC'),
        'clip_duration_sec': globals().get('CLIP_DURATION_SEC'),
    },
    'transcripts': {
        name: {
            'character_count': len(text),
            'full_text': text,
            'segments': globals().get('transcript_segments', {}).get(name, []),
        }
        for name, text in globals().get('transcripts', {}).items()
    },
    'scene_detection': scene_report,
    'audio_peaks_sec': globals().get('audio_peaks', []),
    'audio_peak_details': globals().get('audio_peak_details', []),
    'local_hot_moments': globals().get('hot_moments', []),
    'llm_hotmoment_outputs': globals().get('llm_outputs', {}),
    'benchmark_results': globals().get('results', []),
}

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
report_path = RESULTS_DIR / '01_video_analysis_tanilama.json'
zip_path = RESULTS_DIR / '01_video_analysis_paylasim_paketi.zip'

with open(report_path, 'w', encoding='utf-8') as report_file:
    json.dump(diagnostic_report, report_file, ensure_ascii=False, indent=2)

with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    archive.write(report_path, arcname=report_path.name)
    if Path(out_path).exists():
        archive.write(out_path, arcname=Path(out_path).name)
    if Path(globals().get('hot_moments_path', '')).exists():
        archive.write(hot_moments_path, arcname=Path(hot_moments_path).name)
    for preview_file in (RESULTS_DIR / 'hot_moment_previews').glob('*.mp4'):
        archive.write(preview_file, arcname=f'hot_moment_previews/{preview_file.name}')

print('Paylasim paketi hazir:', zip_path)

try:
    from google.colab import files
    files.download(str(zip_path))
except Exception as exc:
    print('Otomatik indirme baslatilamadi. Files panelinden elle indir:', zip_path)
    print('Detay:', exc)


## 8. Karar (elle doldurulacak)

Yukarıdaki tabloyu inceledikten sonra bu hücreyi düzenleyip kararını ve kısa gerekçeni yaz. Bu karar Faz 1'de `packages/providers/vision-analysis` ve `packages/providers/llm` adaptörlerinin varsayılan sağlayıcısını belirleyecek.

- **Transcript:** _(seçilen aday)_ — gerekçe: _...
- **Sahne tespiti:** _(seçilen aday)_ — gerekçe: _...
- **Hot-moment skorlama:** _(seçilen aday)_ — gerekçe: _...
